In [0]:
"""
id: source_954edab1
template: source
templateVersion: 2.0.0
name: src_account
position:
  x: 0
  y: 260
description:
  text: Read all data from the table renewal_risk_lakehouse.raw_sfdc_dev.Account.
  hash: e65aef60
previewCodeHash: 3f95c95e7b75e1a1
config:
  table_source:
    tableName: renewal_risk_lakehouse.raw_sfdc_dev.Account
input: []
"""

# generated from the system
from typing import Dict, Any, List

def _strip_sql_quotes(s):
    if isinstance(s, str) and len(s) >= 2:
        if (s[0] == '"' and s[-1] == '"') or (s[0] == "'" and s[-1] == "'"):
            return s[1:-1]
    return s

def _build_metric_view_sql(
    table_name: str, dims: List[str], measures: List[str]
) -> str:
    def q(n: str) -> str:
        return "`" + n.replace("`", "``") + "`"

    select_parts = [q(d) for d in dims] + [f"MEASURE({q(m)}) AS {q(m)}" for m in measures]
    if not select_parts:
        return f"SELECT * FROM {table_name}"
    sql = f"SELECT {', '.join(select_parts)} FROM {table_name}"
    if dims and measures:
        sql += " GROUP BY " + ", ".join(q(d) for d in dims)
    elif dims and not measures:
        sql = f"SELECT DISTINCT {', '.join(q(d) for d in dims)} FROM {table_name}"
    return sql

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    file_source = config.get("file_source")
    table_source = config.get("table_source")

    if file_source:
        path = file_source.get("path")
        if not path:
            raise ValueError("Source: 'path' is required for file source")
        options = []
        for key, value in file_source.items():
            if key == "path":
                continue
            if key == "headerRows" and isinstance(value, bool):
                options.append(f"{key}=>{1 if value else 0}")
            elif isinstance(value, bool):
                options.append(f'{key}=>{"true" if value else "false"}')
            elif isinstance(value, (int, float)):
                options.append(f"{key}=>{value}")
            else:
                clean = _strip_sql_quotes(str(value))
                if key == "dataAddress" and clean.startswith("!"):
                    clean = clean[1:]
                options.append(f'{key}=>"{clean}"')
        opts = ", ".join(options)
        sql = f'SELECT * FROM read_files("{path}", {opts})' if opts else f'SELECT * FROM read_files("{path}")'
        out = spark.sql(sql)
    elif table_source:
        table_name = table_source.get("tableName")
        if not table_name:
            raise ValueError("Source: 'tableName' is required for table source")

        mv_selection = table_source.get("metricView")
        if mv_selection is not None:
            dims = mv_selection.get("dimensions")
            measures = mv_selection.get("measures")
            if dims is None or measures is None:
                raise ValueError(
                    "Source: metricView selection is incomplete (both "
                    "'dimensions' and 'measures' must be explicit lists). "
                    "Re-open the source node and select the metric view "
                    "again to re-seed the picker."
                )
            sql = _build_metric_view_sql(table_name, list(dims), list(measures))
            out = spark.sql(sql)
        elif table_source.get("isExpression"):
            out = spark.sql(table_name)
        else:
            out = spark.table(table_name)
    else:
        raise ValueError("Source: either 'file_source' or 'table_source' must be configured")

    return {"data": out}

# generated from the system
if "ld_display_outputs" not in globals():
    try:
        _ld_param = dbutils.widgets.getAll().get("ld_display_outputs")
        if _ld_param is not None and str(_ld_param).strip() != "":
            globals()["ld_display_outputs"] = str(_ld_param).strip().lower() not in ("false", "0", "no", "off")
        else:
            try:
                from dbruntime.databricks_repl_context import get_context
                globals()["ld_display_outputs"] = not get_context().isInJob
            except Exception:
                globals()["ld_display_outputs"] = True
    except Exception:
        globals()["ld_display_outputs"] = False
ctx = globals().setdefault("ctx", {})
config = {
    "table_source": {
        "tableName": "renewal_risk_lakehouse.raw_sfdc_dev.Account"
    }
}
# Project parameter override for bundle-driven dev/prod raw ingestion reads.
# The Designer UI keeps the saved source schema for interactive development.
# The deployed job passes raw_schema so the same visual artifact reads from
# raw_sfdc_dev in dev and raw_sfdc_prod in prod.
try:
    _bundle_catalog = dbutils.widgets.get("catalog").strip()
except Exception:
    _bundle_catalog = ""
try:
    _bundle_raw_schema = dbutils.widgets.get("raw_schema").strip()
except Exception:
    _bundle_raw_schema = ""
if _bundle_catalog or _bundle_raw_schema:
    _source_catalog = _bundle_catalog or "renewal_risk_lakehouse"
    _source_schema = _bundle_raw_schema or "raw_sfdc"
    config["table_source"]["tableName"] = f"{_source_catalog}.{_source_schema}.Account"

config["meta_state"] = {"is_preview": False, "is_focused_preview": False, "is_disabled": False}
inputs = {}
if config["meta_state"]["is_disabled"]:
    def _lb_limit_zero(value):
        if hasattr(value, "limit"):
            return value.limit(0)
        if isinstance(value, list):
            return [_lb_limit_zero(item) for item in value]
        return value
    inputs = {k: _lb_limit_zero(v) for k, v in inputs.items()}
out = run(config, inputs, spark)
if config["meta_state"]["is_disabled"]:
    out = {k: _lb_limit_zero(v) for k, v in out.items()}
ctx["source_account_b58d7d0c.data"] = out["data"]
if globals().get("ld_display_outputs", False):
    display(ctx["source_account_b58d7d0c.data"])

In [0]:
"""
id: source_ae97be2b
template: source
templateVersion: 2.0.0
name: src_product_usage
position:
  x: 0
  y: 680
description:
  text: Load all data from the Product Usage table.
  hash: 674472af
previewCodeHash: 94aed9714d320226
config:
  table_source:
    tableName: renewal_risk_lakehouse.raw_sfdc_dev.Product_Usage__c
input: []
"""

# generated from the system
from typing import Dict, Any, List

def _strip_sql_quotes(s):
    if isinstance(s, str) and len(s) >= 2:
        if (s[0] == '"' and s[-1] == '"') or (s[0] == "'" and s[-1] == "'"):
            return s[1:-1]
    return s

def _build_metric_view_sql(
    table_name: str, dims: List[str], measures: List[str]
) -> str:
    def q(n: str) -> str:
        return "`" + n.replace("`", "``") + "`"

    select_parts = [q(d) for d in dims] + [f"MEASURE({q(m)}) AS {q(m)}" for m in measures]
    if not select_parts:
        return f"SELECT * FROM {table_name}"
    sql = f"SELECT {', '.join(select_parts)} FROM {table_name}"
    if dims and measures:
        sql += " GROUP BY " + ", ".join(q(d) for d in dims)
    elif dims and not measures:
        sql = f"SELECT DISTINCT {', '.join(q(d) for d in dims)} FROM {table_name}"
    return sql

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    file_source = config.get("file_source")
    table_source = config.get("table_source")

    if file_source:
        path = file_source.get("path")
        if not path:
            raise ValueError("Source: 'path' is required for file source")
        options = []
        for key, value in file_source.items():
            if key == "path":
                continue
            if key == "headerRows" and isinstance(value, bool):
                options.append(f"{key}=>{1 if value else 0}")
            elif isinstance(value, bool):
                options.append(f'{key}=>{"true" if value else "false"}')
            elif isinstance(value, (int, float)):
                options.append(f"{key}=>{value}")
            else:
                clean = _strip_sql_quotes(str(value))
                if key == "dataAddress" and clean.startswith("!"):
                    clean = clean[1:]
                options.append(f'{key}=>"{clean}"')
        opts = ", ".join(options)
        sql = f'SELECT * FROM read_files("{path}", {opts})' if opts else f'SELECT * FROM read_files("{path}")'
        out = spark.sql(sql)
    elif table_source:
        table_name = table_source.get("tableName")
        if not table_name:
            raise ValueError("Source: 'tableName' is required for table source")

        mv_selection = table_source.get("metricView")
        if mv_selection is not None:
            dims = mv_selection.get("dimensions")
            measures = mv_selection.get("measures")
            if dims is None or measures is None:
                raise ValueError(
                    "Source: metricView selection is incomplete (both "
                    "'dimensions' and 'measures' must be explicit lists). "
                    "Re-open the source node and select the metric view "
                    "again to re-seed the picker."
                )
            sql = _build_metric_view_sql(table_name, list(dims), list(measures))
            out = spark.sql(sql)
        elif table_source.get("isExpression"):
            out = spark.sql(table_name)
        else:
            out = spark.table(table_name)
    else:
        raise ValueError("Source: either 'file_source' or 'table_source' must be configured")

    return {"data": out}

# generated from the system
if "ld_display_outputs" not in globals():
    try:
        _ld_param = dbutils.widgets.getAll().get("ld_display_outputs")
        if _ld_param is not None and str(_ld_param).strip() != "":
            globals()["ld_display_outputs"] = str(_ld_param).strip().lower() not in ("false", "0", "no", "off")
        else:
            try:
                from dbruntime.databricks_repl_context import get_context
                globals()["ld_display_outputs"] = not get_context().isInJob
            except Exception:
                globals()["ld_display_outputs"] = True
    except Exception:
        globals()["ld_display_outputs"] = False
if "ld_display_outputs_for" not in globals():
    try:
        _ld_for = dbutils.widgets.getAll().get("ld_display_outputs_for")
        globals()["ld_display_outputs_for"] = frozenset(_p.strip() for _p in str(_ld_for).split(",") if _p.strip()) if _ld_for is not None else frozenset()
    except Exception:
        globals()["ld_display_outputs_for"] = frozenset()
ctx = globals().setdefault("ctx", {})
config = {
    "table_source": {
        "tableName": "renewal_risk_lakehouse.raw_sfdc_dev.Product_Usage__c"
    }
}
config["meta_state"] = {"is_preview": False, "is_focused_preview": False, "is_disabled": False}
inputs = {}
if config["meta_state"]["is_disabled"]:
    def _lb_limit_zero(value):
        if hasattr(value, "limit"):
            return value.limit(0)
        if isinstance(value, list):
            return [_lb_limit_zero(item) for item in value]
        return value
    inputs = {k: _lb_limit_zero(v) for k, v in inputs.items()}
out = run(config, inputs, spark)
if config["meta_state"]["is_disabled"]:
    out = {k: _lb_limit_zero(v) for k, v in out.items()}
ctx["source_ae97be2b.data"] = out["data"]
if globals().get("ld_display_outputs", False) or "source_ae97be2b" in globals().get("ld_display_outputs_for", frozenset()):
    display(ctx["source_ae97be2b.data"])

In [0]:
"""
id: source_b40f8c24
template: source
templateVersion: 2.0.0
name: src_contact
position:
  x: 0
  y: 0
description:
  text: Load all data from the Contact table.
  hash: 194ade2a
previewCodeHash: d02f9f093160f43e
config:
  table_source:
    tableName: renewal_risk_lakehouse.raw_sfdc_dev.Contact
input: []
"""

# generated from the system
from typing import Dict, Any, List

def _strip_sql_quotes(s):
    if isinstance(s, str) and len(s) >= 2:
        if (s[0] == '"' and s[-1] == '"') or (s[0] == "'" and s[-1] == "'"):
            return s[1:-1]
    return s

def _build_metric_view_sql(
    table_name: str, dims: List[str], measures: List[str]
) -> str:
    def q(n: str) -> str:
        return "`" + n.replace("`", "``") + "`"

    select_parts = [q(d) for d in dims] + [f"MEASURE({q(m)}) AS {q(m)}" for m in measures]
    if not select_parts:
        return f"SELECT * FROM {table_name}"
    sql = f"SELECT {', '.join(select_parts)} FROM {table_name}"
    if dims and measures:
        sql += " GROUP BY " + ", ".join(q(d) for d in dims)
    elif dims and not measures:
        sql = f"SELECT DISTINCT {', '.join(q(d) for d in dims)} FROM {table_name}"
    return sql

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    file_source = config.get("file_source")
    table_source = config.get("table_source")

    if file_source:
        path = file_source.get("path")
        if not path:
            raise ValueError("Source: 'path' is required for file source")
        options = []
        for key, value in file_source.items():
            if key == "path":
                continue
            if key == "headerRows" and isinstance(value, bool):
                options.append(f"{key}=>{1 if value else 0}")
            elif isinstance(value, bool):
                options.append(f'{key}=>{"true" if value else "false"}')
            elif isinstance(value, (int, float)):
                options.append(f"{key}=>{value}")
            else:
                clean = _strip_sql_quotes(str(value))
                if key == "dataAddress" and clean.startswith("!"):
                    clean = clean[1:]
                options.append(f'{key}=>"{clean}"')
        opts = ", ".join(options)
        sql = f'SELECT * FROM read_files("{path}", {opts})' if opts else f'SELECT * FROM read_files("{path}")'
        out = spark.sql(sql)
    elif table_source:
        table_name = table_source.get("tableName")
        if not table_name:
            raise ValueError("Source: 'tableName' is required for table source")

        mv_selection = table_source.get("metricView")
        if mv_selection is not None:
            dims = mv_selection.get("dimensions")
            measures = mv_selection.get("measures")
            if dims is None or measures is None:
                raise ValueError(
                    "Source: metricView selection is incomplete (both "
                    "'dimensions' and 'measures' must be explicit lists). "
                    "Re-open the source node and select the metric view "
                    "again to re-seed the picker."
                )
            sql = _build_metric_view_sql(table_name, list(dims), list(measures))
            out = spark.sql(sql)
        elif table_source.get("isExpression"):
            out = spark.sql(table_name)
        else:
            out = spark.table(table_name)
    else:
        raise ValueError("Source: either 'file_source' or 'table_source' must be configured")

    return {"data": out}

# generated from the system
if "ld_display_outputs" not in globals():
    try:
        _ld_param = dbutils.widgets.getAll().get("ld_display_outputs")
        if _ld_param is not None and str(_ld_param).strip() != "":
            globals()["ld_display_outputs"] = str(_ld_param).strip().lower() not in ("false", "0", "no", "off")
        else:
            try:
                from dbruntime.databricks_repl_context import get_context
                globals()["ld_display_outputs"] = not get_context().isInJob
            except Exception:
                globals()["ld_display_outputs"] = True
    except Exception:
        globals()["ld_display_outputs"] = False
if "ld_display_outputs_for" not in globals():
    try:
        _ld_for = dbutils.widgets.getAll().get("ld_display_outputs_for")
        globals()["ld_display_outputs_for"] = frozenset(_p.strip() for _p in str(_ld_for).split(",") if _p.strip()) if _ld_for is not None else frozenset()
    except Exception:
        globals()["ld_display_outputs_for"] = frozenset()
ctx = globals().setdefault("ctx", {})
config = {
    "table_source": {
        "tableName": "renewal_risk_lakehouse.raw_sfdc_dev.Contact"
    }
}
config["meta_state"] = {"is_preview": False, "is_focused_preview": False, "is_disabled": False}
inputs = {}
if config["meta_state"]["is_disabled"]:
    def _lb_limit_zero(value):
        if hasattr(value, "limit"):
            return value.limit(0)
        if isinstance(value, list):
            return [_lb_limit_zero(item) for item in value]
        return value
    inputs = {k: _lb_limit_zero(v) for k, v in inputs.items()}
out = run(config, inputs, spark)
if config["meta_state"]["is_disabled"]:
    out = {k: _lb_limit_zero(v) for k, v in out.items()}
ctx["source_b40f8c24.data"] = out["data"]
if globals().get("ld_display_outputs", False) or "source_b40f8c24" in globals().get("ld_display_outputs_for", frozenset()):
    display(ctx["source_b40f8c24.data"])

In [0]:
"""
id: source_c21ea952
template: source
templateVersion: 2.0.0
name: src_opportunity
position:
  x: 0
  y: 119
description:
  text: Retrieve all data from the renewal risk source table.
  hash: d1eec4f5
previewCodeHash: d7f22c7356569f07
config:
  table_source:
    tableName: renewal_risk_lakehouse.raw_sfdc_dev.Opportunity
input: []
"""

# generated from the system
from typing import Dict, Any, List

def _strip_sql_quotes(s):
    if isinstance(s, str) and len(s) >= 2:
        if (s[0] == '"' and s[-1] == '"') or (s[0] == "'" and s[-1] == "'"):
            return s[1:-1]
    return s

def _build_metric_view_sql(
    table_name: str, dims: List[str], measures: List[str]
) -> str:
    def q(n: str) -> str:
        return "`" + n.replace("`", "``") + "`"

    select_parts = [q(d) for d in dims] + [f"MEASURE({q(m)}) AS {q(m)}" for m in measures]
    if not select_parts:
        return f"SELECT * FROM {table_name}"
    sql = f"SELECT {', '.join(select_parts)} FROM {table_name}"
    if dims and measures:
        sql += " GROUP BY " + ", ".join(q(d) for d in dims)
    elif dims and not measures:
        sql = f"SELECT DISTINCT {', '.join(q(d) for d in dims)} FROM {table_name}"
    return sql

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    file_source = config.get("file_source")
    table_source = config.get("table_source")

    if file_source:
        path = file_source.get("path")
        if not path:
            raise ValueError("Source: 'path' is required for file source")
        options = []
        for key, value in file_source.items():
            if key == "path":
                continue
            if key == "headerRows" and isinstance(value, bool):
                options.append(f"{key}=>{1 if value else 0}")
            elif isinstance(value, bool):
                options.append(f'{key}=>{"true" if value else "false"}')
            elif isinstance(value, (int, float)):
                options.append(f"{key}=>{value}")
            else:
                clean = _strip_sql_quotes(str(value))
                if key == "dataAddress" and clean.startswith("!"):
                    clean = clean[1:]
                options.append(f'{key}=>"{clean}"')
        opts = ", ".join(options)
        sql = f'SELECT * FROM read_files("{path}", {opts})' if opts else f'SELECT * FROM read_files("{path}")'
        out = spark.sql(sql)
    elif table_source:
        table_name = table_source.get("tableName")
        if not table_name:
            raise ValueError("Source: 'tableName' is required for table source")

        mv_selection = table_source.get("metricView")
        if mv_selection is not None:
            dims = mv_selection.get("dimensions")
            measures = mv_selection.get("measures")
            if dims is None or measures is None:
                raise ValueError(
                    "Source: metricView selection is incomplete (both "
                    "'dimensions' and 'measures' must be explicit lists). "
                    "Re-open the source node and select the metric view "
                    "again to re-seed the picker."
                )
            sql = _build_metric_view_sql(table_name, list(dims), list(measures))
            out = spark.sql(sql)
        elif table_source.get("isExpression"):
            out = spark.sql(table_name)
        else:
            out = spark.table(table_name)
    else:
        raise ValueError("Source: either 'file_source' or 'table_source' must be configured")

    return {"data": out}

# generated from the system
if "ld_display_outputs" not in globals():
    try:
        _ld_param = dbutils.widgets.getAll().get("ld_display_outputs")
        if _ld_param is not None and str(_ld_param).strip() != "":
            globals()["ld_display_outputs"] = str(_ld_param).strip().lower() not in ("false", "0", "no", "off")
        else:
            try:
                from dbruntime.databricks_repl_context import get_context
                globals()["ld_display_outputs"] = not get_context().isInJob
            except Exception:
                globals()["ld_display_outputs"] = True
    except Exception:
        globals()["ld_display_outputs"] = False
if "ld_display_outputs_for" not in globals():
    try:
        _ld_for = dbutils.widgets.getAll().get("ld_display_outputs_for")
        globals()["ld_display_outputs_for"] = frozenset(_p.strip() for _p in str(_ld_for).split(",") if _p.strip()) if _ld_for is not None else frozenset()
    except Exception:
        globals()["ld_display_outputs_for"] = frozenset()
ctx = globals().setdefault("ctx", {})
config = {
    "table_source": {
        "tableName": "renewal_risk_lakehouse.raw_sfdc_dev.Opportunity"
    }
}
config["meta_state"] = {"is_preview": False, "is_focused_preview": False, "is_disabled": False}
inputs = {}
if config["meta_state"]["is_disabled"]:
    def _lb_limit_zero(value):
        if hasattr(value, "limit"):
            return value.limit(0)
        if isinstance(value, list):
            return [_lb_limit_zero(item) for item in value]
        return value
    inputs = {k: _lb_limit_zero(v) for k, v in inputs.items()}
out = run(config, inputs, spark)
if config["meta_state"]["is_disabled"]:
    out = {k: _lb_limit_zero(v) for k, v in out.items()}
ctx["source_c21ea952.data"] = out["data"]
if globals().get("ld_display_outputs", False) or "source_c21ea952" in globals().get("ld_display_outputs_for", frozenset()):
    display(ctx["source_c21ea952.data"])

In [0]:
"""
id: source_e212f0ab
template: source
templateVersion: 2.0.0
name: src_case
position:
  x: 300
  y: 310
description:
  text: Get all records from the Case table in the renewal risk lakehouse raw Salesforce development database.
  hash: 7b3b730d
previewCodeHash: 0a0ba3209bf7b1a8
config:
  table_source:
    tableName: renewal_risk_lakehouse.raw_sfdc_dev.`Case`
input: []
"""

# generated from the system
from typing import Dict, Any, List

def _strip_sql_quotes(s):
    if isinstance(s, str) and len(s) >= 2:
        if (s[0] == '"' and s[-1] == '"') or (s[0] == "'" and s[-1] == "'"):
            return s[1:-1]
    return s

def _build_metric_view_sql(
    table_name: str, dims: List[str], measures: List[str]
) -> str:
    def q(n: str) -> str:
        return "`" + n.replace("`", "``") + "`"

    select_parts = [q(d) for d in dims] + [f"MEASURE({q(m)}) AS {q(m)}" for m in measures]
    if not select_parts:
        return f"SELECT * FROM {table_name}"
    sql = f"SELECT {', '.join(select_parts)} FROM {table_name}"
    if dims and measures:
        sql += " GROUP BY " + ", ".join(q(d) for d in dims)
    elif dims and not measures:
        sql = f"SELECT DISTINCT {', '.join(q(d) for d in dims)} FROM {table_name}"
    return sql

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    file_source = config.get("file_source")
    table_source = config.get("table_source")

    if file_source:
        path = file_source.get("path")
        if not path:
            raise ValueError("Source: 'path' is required for file source")
        options = []
        for key, value in file_source.items():
            if key == "path":
                continue
            if key == "headerRows" and isinstance(value, bool):
                options.append(f"{key}=>{1 if value else 0}")
            elif isinstance(value, bool):
                options.append(f'{key}=>{"true" if value else "false"}')
            elif isinstance(value, (int, float)):
                options.append(f"{key}=>{value}")
            else:
                clean = _strip_sql_quotes(str(value))
                if key == "dataAddress" and clean.startswith("!"):
                    clean = clean[1:]
                options.append(f'{key}=>"{clean}"')
        opts = ", ".join(options)
        sql = f'SELECT * FROM read_files("{path}", {opts})' if opts else f'SELECT * FROM read_files("{path}")'
        out = spark.sql(sql)
    elif table_source:
        table_name = table_source.get("tableName")
        if not table_name:
            raise ValueError("Source: 'tableName' is required for table source")

        mv_selection = table_source.get("metricView")
        if mv_selection is not None:
            dims = mv_selection.get("dimensions")
            measures = mv_selection.get("measures")
            if dims is None or measures is None:
                raise ValueError(
                    "Source: metricView selection is incomplete (both "
                    "'dimensions' and 'measures' must be explicit lists). "
                    "Re-open the source node and select the metric view "
                    "again to re-seed the picker."
                )
            sql = _build_metric_view_sql(table_name, list(dims), list(measures))
            out = spark.sql(sql)
        elif table_source.get("isExpression"):
            out = spark.sql(table_name)
        else:
            out = spark.table(table_name)
    else:
        raise ValueError("Source: either 'file_source' or 'table_source' must be configured")

    return {"data": out}

# generated from the system
if "ld_display_outputs" not in globals():
    try:
        _ld_param = dbutils.widgets.getAll().get("ld_display_outputs")
        if _ld_param is not None and str(_ld_param).strip() != "":
            globals()["ld_display_outputs"] = str(_ld_param).strip().lower() not in ("false", "0", "no", "off")
        else:
            try:
                from dbruntime.databricks_repl_context import get_context
                globals()["ld_display_outputs"] = not get_context().isInJob
            except Exception:
                globals()["ld_display_outputs"] = True
    except Exception:
        globals()["ld_display_outputs"] = False
if "ld_display_outputs_for" not in globals():
    try:
        _ld_for = dbutils.widgets.getAll().get("ld_display_outputs_for")
        globals()["ld_display_outputs_for"] = frozenset(_p.strip() for _p in str(_ld_for).split(",") if _p.strip()) if _ld_for is not None else frozenset()
    except Exception:
        globals()["ld_display_outputs_for"] = frozenset()
ctx = globals().setdefault("ctx", {})
config = {
    "table_source": {
        "tableName": "renewal_risk_lakehouse.raw_sfdc_dev.`Case`"
    }
}
config["meta_state"] = {"is_preview": False, "is_focused_preview": False, "is_disabled": False}
inputs = {}
if config["meta_state"]["is_disabled"]:
    def _lb_limit_zero(value):
        if hasattr(value, "limit"):
            return value.limit(0)
        if isinstance(value, list):
            return [_lb_limit_zero(item) for item in value]
        return value
    inputs = {k: _lb_limit_zero(v) for k, v in inputs.items()}
out = run(config, inputs, spark)
if config["meta_state"]["is_disabled"]:
    out = {k: _lb_limit_zero(v) for k, v in out.items()}
ctx["source_e212f0ab.data"] = out["data"]
if globals().get("ld_display_outputs", False) or "source_e212f0ab" in globals().get("ld_display_outputs_for", frozenset()):
    display(ctx["source_e212f0ab.data"])

In [0]:
"""
id: prepare_a0f1c2d3
template: prepare
templateVersion: 1.0.0
name: prep_accounts
position:
  x: 260
  y: 260
description:
  text: Create new columns mapping existing ones, convert types, and trim spaces on account number and name.
  hash: 4ec8c451
previewCodeHash: 0bc9d7d27d485d7f
config:
  actions:
    - type: formula
      target: account_id
      expression: Id
    - type: formula
      target: account_number
      expression: AccountNumber
    - type: formula
      target: account_name
      expression: Name
    - type: formula
      target: industry
      expression: Industry
    - type: formula
      target: customer_segment
      expression: Customer_Segment__c
    - type: formula
      target: arr
      expression: ARR__c
    - type: formula
      target: renewal_date
      expression: Renewal_Date__c
    - type: cast
      column: arr
      to: double
      on_error: "null"
    - type: cast
      column: renewal_date
      to: date
      on_error: "null"
    - type: trim
      column: account_number
      side: both
    - type: trim
      column: account_name
      side: both
input:
  - node: source_954edab1
    input_port: data
    output_port: data
"""

# generated from the system
from typing import Any, Callable, Dict, List
import pyspark.sql.functions as F

TEXT_CASE_FUNCTIONS: Dict[str, Callable] = {
    "lower": F.lower,
    "upper": F.upper,
    "title": F.initcap,
}

TRIM_FUNCTIONS: Dict[str, Callable] = {
    "both": F.trim,
    "left": F.ltrim,
    "right": F.rtrim,
}

VALID_MATCH_MODES = {"exact", "contains", "prefix", "suffix", "regex"}

def _require_column(df, column: str, action_type: str) -> None:
    if not column:
        raise ValueError(f"{action_type}: 'column' is required")
    if column not in df.columns:
        raise ValueError(
            f"{action_type}: column '{column}' not found in input data. "
            f"Available columns: {df.columns}"
        )

def _column_dtype(df, column: str) -> str:
    return df.schema[column].dataType.simpleString()

def _quoted_ident(name: str) -> str:
    return "`" + name.replace("`", "``") + "`"

def _col(name: str):
    """Reference a column by exact name.

    F.col parses its argument as a column expression, so a name containing
    a '.' (e.g. "a.b") is otherwise read as struct-field access and fails
    even though the column exists. Backtick-quoting forces an exact-name
    lookup; quoting plain names is harmless.
    """
    return F.col(_quoted_ident(name))

def _coerced_lit(value: Any, target_dtype: str):
    """Wrap a user-provided value as a literal cast to the target column's type.

    UI always feeds text per the operator spec; this is where the
    type coercion happens so the per-action UI stays simple.
    """
    return F.lit(value).cast(target_dtype)

def _apply_formula(df, action: Dict[str, Any]):
    target = action.get("target", "")
    expression = action.get("expression", "")
    if not target:
        raise ValueError("formula: 'target' is required")
    if not expression:
        raise ValueError("formula: 'expression' is required")
    return df.withColumn(target, F.expr(expression))

def _apply_cast(df, action: Dict[str, Any]):
    column = action.get("column", "")
    to_type = action.get("to", "")
    on_error = action.get("on_error", "null")
    _require_column(df, column, "cast")
    if not to_type:
        raise ValueError("cast: 'to' (target type) is required")
    if on_error == "null":
        return df.withColumn(
            column,
            F.expr(f"try_cast({_quoted_ident(column)} as {to_type})"),
        )
    return df.withColumn(column, _col(column).cast(to_type))

def _apply_replace_value(df, action: Dict[str, Any]):
    column = action.get("column", "")
    match_mode = action.get("match_mode", "exact")
    match = action.get("match", "")
    with_val = action.get("with", "")
    case_sensitive = bool(action.get("case_sensitive", False))
    _require_column(df, column, "replace_value")
    if match_mode not in VALID_MATCH_MODES:
        raise ValueError(
            f"replace_value: unsupported match_mode '{match_mode}'. "
            f"Choose one of: {sorted(VALID_MATCH_MODES)}"
        )
    dtype = _column_dtype(df, column)
    col_as_string = _col(column).cast("string")
    match_lit = F.lit(match)
    if case_sensitive:
        cmp_col = col_as_string
        cmp_lit = match_lit
    else:
        cmp_col = F.lower(col_as_string)
        cmp_lit = F.lower(match_lit)

    if match_mode == "exact":
        predicate = cmp_col == cmp_lit
    elif match_mode == "contains":
        predicate = cmp_col.contains(cmp_lit)
    elif match_mode == "prefix":
        predicate = cmp_col.startswith(cmp_lit)
    elif match_mode == "suffix":
        predicate = cmp_col.endswith(cmp_lit)
    else:  # regex
        pattern = match if case_sensitive else f"(?i){match}"
        predicate = col_as_string.rlike(pattern)

    replacement = _coerced_lit(with_val, dtype)
    return df.withColumn(
        column,
        F.when(predicate, replacement).otherwise(_col(column)),
    )

def _apply_fill_null(df, action: Dict[str, Any]):
    column = action.get("column", "")
    with_val = action.get("with", "")
    _require_column(df, column, "fill_null")
    dtype = _column_dtype(df, column)
    replacement = _coerced_lit(with_val, dtype)
    return df.withColumn(
        column,
        F.when(_col(column).isNull(), replacement).otherwise(_col(column)),
    )

def _apply_text_case(df, action: Dict[str, Any]):
    column = action.get("column", "")
    case = action.get("case", "")
    _require_column(df, column, "text_case")
    fn = TEXT_CASE_FUNCTIONS.get(case)
    if fn is None:
        raise ValueError(
            f"text_case: unsupported case '{case}'. "
            f"Choose one of: {sorted(TEXT_CASE_FUNCTIONS.keys())}"
        )
    return df.withColumn(column, fn(_col(column)))

def _apply_trim(df, action: Dict[str, Any]):
    column = action.get("column", "")
    side = action.get("side", "both")
    _require_column(df, column, "trim")
    fn = TRIM_FUNCTIONS.get(side)
    if fn is None:
        raise ValueError(
            f"trim: unsupported side '{side}'. "
            f"Choose one of: {sorted(TRIM_FUNCTIONS.keys())}"
        )
    return df.withColumn(column, fn(_col(column)))

def _apply_regex_replace(df, action: Dict[str, Any]):
    column = action.get("column", "")
    pattern = action.get("pattern", "")
    replacement = action.get("replacement", "")
    _require_column(df, column, "regex_replace")
    return df.withColumn(
        column,
        F.regexp_replace(_col(column), pattern, replacement),
    )

def _apply_extract(df, action: Dict[str, Any]):
    column = action.get("column", "")
    pattern = action.get("pattern", "")
    target = action.get("target") or column
    group_raw = action.get("group", 0)
    _require_column(df, column, "extract")
    try:
        group_idx = int(group_raw)
    except (TypeError, ValueError) as exc:
        raise ValueError(
            f"extract: 'group' must be an integer, got {group_raw!r}"
        ) from exc
    return df.withColumn(
        target,
        F.regexp_extract(_col(column), pattern, group_idx),
    )

def _apply_parse_date(df, action: Dict[str, Any]):
    column = action.get("column", "")
    kind = action.get("kind", "date")
    fmt = action.get("format") or None
    on_error = action.get("on_error", "null")
    _require_column(df, column, "parse_date")
    if kind not in ("date", "timestamp"):
        raise ValueError(
            f"parse_date: unsupported kind '{kind}'. Choose date or timestamp."
        )
    # PySpark exposes F.try_to_timestamp from Spark 3.5 but only adds
    # F.try_to_date in Spark 4.0. Current Databricks Runtimes still ship
    # Spark 3.5, where referencing F.try_to_date raises AttributeError
    # before any data is touched. Route the null-on-error path through
    # primitives that exist in both 3.5 and 4.0:
    #   - F.try_to_timestamp  (PySpark 3.5+, returns NULL under ANSI too)
    #   - SQL try_cast        (Spark 3.5+, returns NULL under ANSI too)
    #   - F.to_date           (PySpark 3.5+; returns NULL on parse failure
    #                          under the default non-ANSI mode, which is
    #                          the Databricks default. Under ANSI mode it
    #                          raises — accepted limitation until
    #                          try_to_date lands on every supported DBR.)
    if on_error == "null":
        if kind == "timestamp":
            if fmt:
                return df.withColumn(
                    column, F.try_to_timestamp(_col(column), F.lit(fmt))
                )
            return df.withColumn(column, F.try_to_timestamp(_col(column)))
        # kind == "date"
        if fmt:
            return df.withColumn(column, F.to_date(_col(column), fmt))
        return df.withColumn(
            column, F.expr(f"try_cast({_quoted_ident(column)} as date)")
        )
    # on_error == "error": surface parse failures as Spark exceptions.
    fn = F.to_date if kind == "date" else F.to_timestamp
    if fmt:
        return df.withColumn(column, fn(_col(column), fmt))
    return df.withColumn(column, fn(_col(column)))

ACTION_DISPATCH: Dict[str, Callable] = {
    "formula": _apply_formula,
    "cast": _apply_cast,
    "replace_value": _apply_replace_value,
    "fill_null": _apply_fill_null,
    "text_case": _apply_text_case,
    "trim": _apply_trim,
    "regex_replace": _apply_regex_replace,
    "extract": _apply_extract,
    "parse_date": _apply_parse_date,
}

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs.get("data")
    actions: List[Dict[str, Any]] = config.get("actions", []) or []

    if not actions:
        return {"prepared_data": df}

    for index, action in enumerate(actions):
        if not isinstance(action, dict):
            raise ValueError(
                f"actions[{index}]: expected an object, got {type(action).__name__}"
            )
        if action.get("enabled", True) is False:
            continue
        action_type = action.get("type", "")
        fn = ACTION_DISPATCH.get(action_type)
        if fn is None:
            raise ValueError(
                f"actions[{index}]: unsupported action type {action_type!r}. "
                f"Choose one of: {sorted(ACTION_DISPATCH.keys())}"
            )
        df = fn(df, action)
    return {"prepared_data": df}

# generated from the system
if "ld_display_outputs" not in globals():
    try:
        _ld_param = dbutils.widgets.getAll().get("ld_display_outputs")
        if _ld_param is not None and str(_ld_param).strip() != "":
            globals()["ld_display_outputs"] = str(_ld_param).strip().lower() not in ("false", "0", "no", "off")
        else:
            try:
                from dbruntime.databricks_repl_context import get_context
                globals()["ld_display_outputs"] = not get_context().isInJob
            except Exception:
                globals()["ld_display_outputs"] = True
    except Exception:
        globals()["ld_display_outputs"] = False
if "ld_display_outputs_for" not in globals():
    try:
        _ld_for = dbutils.widgets.getAll().get("ld_display_outputs_for")
        globals()["ld_display_outputs_for"] = frozenset(_p.strip() for _p in str(_ld_for).split(",") if _p.strip()) if _ld_for is not None else frozenset()
    except Exception:
        globals()["ld_display_outputs_for"] = frozenset()
ctx = globals().setdefault("ctx", {})
config = {
    "actions": [
        {
            "type": "formula",
            "target": "account_id",
            "expression": "Id"
        },
        {
            "type": "formula",
            "target": "account_number",
            "expression": "AccountNumber"
        },
        {
            "type": "formula",
            "target": "account_name",
            "expression": "Name"
        },
        {
            "type": "formula",
            "target": "industry",
            "expression": "Industry"
        },
        {
            "type": "formula",
            "target": "customer_segment",
            "expression": "Customer_Segment__c"
        },
        {
            "type": "formula",
            "target": "arr",
            "expression": "ARR__c"
        },
        {
            "type": "formula",
            "target": "renewal_date",
            "expression": "Renewal_Date__c"
        },
        {
            "type": "cast",
            "column": "arr",
            "to": "double",
            "on_error": "null"
        },
        {
            "type": "cast",
            "column": "renewal_date",
            "to": "date",
            "on_error": "null"
        },
        {
            "type": "trim",
            "column": "account_number",
            "side": "both"
        },
        {
            "type": "trim",
            "column": "account_name",
            "side": "both"
        }
    ]
}
config["meta_state"] = {"is_preview": False, "is_focused_preview": False, "is_disabled": False}
inputs = {
    "data": ctx["source_954edab1.data"]
}
if config["meta_state"]["is_disabled"]:
    def _lb_limit_zero(value):
        if hasattr(value, "limit"):
            return value.limit(0)
        if isinstance(value, list):
            return [_lb_limit_zero(item) for item in value]
        return value
    inputs = {k: _lb_limit_zero(v) for k, v in inputs.items()}
out = run(config, inputs, spark)
if config["meta_state"]["is_disabled"]:
    out = {k: _lb_limit_zero(v) for k, v in out.items()}
ctx["prepare_a0f1c2d3.prepared_data"] = out["prepared_data"]
if globals().get("ld_display_outputs", False) or "prepare_a0f1c2d3" in globals().get("ld_display_outputs_for", frozenset()):
    display(ctx["prepare_a0f1c2d3.prepared_data"])

In [0]:
"""
id: prepare_538c1f8d
template: prepare
templateVersion: 1.0.0
name: prep_product_usage
position:
  x: 260
  y: 680
description:
  text: Create new columns from existing ones, trim spaces from account numbers, and convert date and numeric columns to correct types.
  hash: 1d93c3bb
previewCodeHash: 447f154424be772b
config:
  actions:
    - type: formula
      target: account_number
      expression: Account_Number__c
    - type: formula
      target: usage_date
      expression: Usage_Date__c
    - type: formula
      target: active_users
      expression: Active_Users__c
    - type: formula
      target: api_calls
      expression: Api_Calls__c
    - type: formula
      target: failed_api_calls
      expression: Failed_Api_Calls__c
    - type: formula
      target: key_feature_events
      expression: Key_Feature_Events__c
    - type: trim
      column: account_number
      side: both
    - type: cast
      column: usage_date
      to: date
      on_error: "null"
    - type: cast
      column: active_users
      to: double
      on_error: "null"
    - type: cast
      column: api_calls
      to: double
      on_error: "null"
    - type: cast
      column: failed_api_calls
      to: double
      on_error: "null"
    - type: cast
      column: key_feature_events
      to: double
      on_error: "null"
input:
  - node: source_ae97be2b
    input_port: data
    output_port: data
"""

# generated from the system
from typing import Any, Callable, Dict, List
import pyspark.sql.functions as F

TEXT_CASE_FUNCTIONS: Dict[str, Callable] = {
    "lower": F.lower,
    "upper": F.upper,
    "title": F.initcap,
}

TRIM_FUNCTIONS: Dict[str, Callable] = {
    "both": F.trim,
    "left": F.ltrim,
    "right": F.rtrim,
}

VALID_MATCH_MODES = {"exact", "contains", "prefix", "suffix", "regex"}

def _require_column(df, column: str, action_type: str) -> None:
    if not column:
        raise ValueError(f"{action_type}: 'column' is required")
    if column not in df.columns:
        raise ValueError(
            f"{action_type}: column '{column}' not found in input data. "
            f"Available columns: {df.columns}"
        )

def _column_dtype(df, column: str) -> str:
    return df.schema[column].dataType.simpleString()

def _quoted_ident(name: str) -> str:
    return "`" + name.replace("`", "``") + "`"

def _col(name: str):
    """Reference a column by exact name.

    F.col parses its argument as a column expression, so a name containing
    a '.' (e.g. "a.b") is otherwise read as struct-field access and fails
    even though the column exists. Backtick-quoting forces an exact-name
    lookup; quoting plain names is harmless.
    """
    return F.col(_quoted_ident(name))

def _coerced_lit(value: Any, target_dtype: str):
    """Wrap a user-provided value as a literal cast to the target column's type.

    UI always feeds text per the operator spec; this is where the
    type coercion happens so the per-action UI stays simple.
    """
    return F.lit(value).cast(target_dtype)

def _apply_formula(df, action: Dict[str, Any]):
    target = action.get("target", "")
    expression = action.get("expression", "")
    if not target:
        raise ValueError("formula: 'target' is required")
    if not expression:
        raise ValueError("formula: 'expression' is required")
    return df.withColumn(target, F.expr(expression))

def _apply_cast(df, action: Dict[str, Any]):
    column = action.get("column", "")
    to_type = action.get("to", "")
    on_error = action.get("on_error", "null")
    _require_column(df, column, "cast")
    if not to_type:
        raise ValueError("cast: 'to' (target type) is required")
    if on_error == "null":
        return df.withColumn(
            column,
            F.expr(f"try_cast({_quoted_ident(column)} as {to_type})"),
        )
    return df.withColumn(column, _col(column).cast(to_type))

def _apply_replace_value(df, action: Dict[str, Any]):
    column = action.get("column", "")
    match_mode = action.get("match_mode", "exact")
    match = action.get("match", "")
    with_val = action.get("with", "")
    case_sensitive = bool(action.get("case_sensitive", False))
    _require_column(df, column, "replace_value")
    if match_mode not in VALID_MATCH_MODES:
        raise ValueError(
            f"replace_value: unsupported match_mode '{match_mode}'. "
            f"Choose one of: {sorted(VALID_MATCH_MODES)}"
        )
    dtype = _column_dtype(df, column)
    col_as_string = _col(column).cast("string")
    match_lit = F.lit(match)
    if case_sensitive:
        cmp_col = col_as_string
        cmp_lit = match_lit
    else:
        cmp_col = F.lower(col_as_string)
        cmp_lit = F.lower(match_lit)

    if match_mode == "exact":
        predicate = cmp_col == cmp_lit
    elif match_mode == "contains":
        predicate = cmp_col.contains(cmp_lit)
    elif match_mode == "prefix":
        predicate = cmp_col.startswith(cmp_lit)
    elif match_mode == "suffix":
        predicate = cmp_col.endswith(cmp_lit)
    else:  # regex
        pattern = match if case_sensitive else f"(?i){match}"
        predicate = col_as_string.rlike(pattern)

    replacement = _coerced_lit(with_val, dtype)
    return df.withColumn(
        column,
        F.when(predicate, replacement).otherwise(_col(column)),
    )

def _apply_fill_null(df, action: Dict[str, Any]):
    column = action.get("column", "")
    with_val = action.get("with", "")
    _require_column(df, column, "fill_null")
    dtype = _column_dtype(df, column)
    replacement = _coerced_lit(with_val, dtype)
    return df.withColumn(
        column,
        F.when(_col(column).isNull(), replacement).otherwise(_col(column)),
    )

def _apply_text_case(df, action: Dict[str, Any]):
    column = action.get("column", "")
    case = action.get("case", "")
    _require_column(df, column, "text_case")
    fn = TEXT_CASE_FUNCTIONS.get(case)
    if fn is None:
        raise ValueError(
            f"text_case: unsupported case '{case}'. "
            f"Choose one of: {sorted(TEXT_CASE_FUNCTIONS.keys())}"
        )
    return df.withColumn(column, fn(_col(column)))

def _apply_trim(df, action: Dict[str, Any]):
    column = action.get("column", "")
    side = action.get("side", "both")
    _require_column(df, column, "trim")
    fn = TRIM_FUNCTIONS.get(side)
    if fn is None:
        raise ValueError(
            f"trim: unsupported side '{side}'. "
            f"Choose one of: {sorted(TRIM_FUNCTIONS.keys())}"
        )
    return df.withColumn(column, fn(_col(column)))

def _apply_regex_replace(df, action: Dict[str, Any]):
    column = action.get("column", "")
    pattern = action.get("pattern", "")
    replacement = action.get("replacement", "")
    _require_column(df, column, "regex_replace")
    return df.withColumn(
        column,
        F.regexp_replace(_col(column), pattern, replacement),
    )

def _apply_extract(df, action: Dict[str, Any]):
    column = action.get("column", "")
    pattern = action.get("pattern", "")
    target = action.get("target") or column
    group_raw = action.get("group", 0)
    _require_column(df, column, "extract")
    try:
        group_idx = int(group_raw)
    except (TypeError, ValueError) as exc:
        raise ValueError(
            f"extract: 'group' must be an integer, got {group_raw!r}"
        ) from exc
    return df.withColumn(
        target,
        F.regexp_extract(_col(column), pattern, group_idx),
    )

def _apply_parse_date(df, action: Dict[str, Any]):
    column = action.get("column", "")
    kind = action.get("kind", "date")
    fmt = action.get("format") or None
    on_error = action.get("on_error", "null")
    _require_column(df, column, "parse_date")
    if kind not in ("date", "timestamp"):
        raise ValueError(
            f"parse_date: unsupported kind '{kind}'. Choose date or timestamp."
        )
    # PySpark exposes F.try_to_timestamp from Spark 3.5 but only adds
    # F.try_to_date in Spark 4.0. Current Databricks Runtimes still ship
    # Spark 3.5, where referencing F.try_to_date raises AttributeError
    # before any data is touched. Route the null-on-error path through
    # primitives that exist in both 3.5 and 4.0:
    #   - F.try_to_timestamp  (PySpark 3.5+, returns NULL under ANSI too)
    #   - SQL try_cast        (Spark 3.5+, returns NULL under ANSI too)
    #   - F.to_date           (PySpark 3.5+; returns NULL on parse failure
    #                          under the default non-ANSI mode, which is
    #                          the Databricks default. Under ANSI mode it
    #                          raises — accepted limitation until
    #                          try_to_date lands on every supported DBR.)
    if on_error == "null":
        if kind == "timestamp":
            if fmt:
                return df.withColumn(
                    column, F.try_to_timestamp(_col(column), F.lit(fmt))
                )
            return df.withColumn(column, F.try_to_timestamp(_col(column)))
        # kind == "date"
        if fmt:
            return df.withColumn(column, F.to_date(_col(column), fmt))
        return df.withColumn(
            column, F.expr(f"try_cast({_quoted_ident(column)} as date)")
        )
    # on_error == "error": surface parse failures as Spark exceptions.
    fn = F.to_date if kind == "date" else F.to_timestamp
    if fmt:
        return df.withColumn(column, fn(_col(column), fmt))
    return df.withColumn(column, fn(_col(column)))

ACTION_DISPATCH: Dict[str, Callable] = {
    "formula": _apply_formula,
    "cast": _apply_cast,
    "replace_value": _apply_replace_value,
    "fill_null": _apply_fill_null,
    "text_case": _apply_text_case,
    "trim": _apply_trim,
    "regex_replace": _apply_regex_replace,
    "extract": _apply_extract,
    "parse_date": _apply_parse_date,
}

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs.get("data")
    actions: List[Dict[str, Any]] = config.get("actions", []) or []

    if not actions:
        return {"prepared_data": df}

    for index, action in enumerate(actions):
        if not isinstance(action, dict):
            raise ValueError(
                f"actions[{index}]: expected an object, got {type(action).__name__}"
            )
        if action.get("enabled", True) is False:
            continue
        action_type = action.get("type", "")
        fn = ACTION_DISPATCH.get(action_type)
        if fn is None:
            raise ValueError(
                f"actions[{index}]: unsupported action type {action_type!r}. "
                f"Choose one of: {sorted(ACTION_DISPATCH.keys())}"
            )
        df = fn(df, action)
    return {"prepared_data": df}

# generated from the system
if "ld_display_outputs" not in globals():
    try:
        _ld_param = dbutils.widgets.getAll().get("ld_display_outputs")
        if _ld_param is not None and str(_ld_param).strip() != "":
            globals()["ld_display_outputs"] = str(_ld_param).strip().lower() not in ("false", "0", "no", "off")
        else:
            try:
                from dbruntime.databricks_repl_context import get_context
                globals()["ld_display_outputs"] = not get_context().isInJob
            except Exception:
                globals()["ld_display_outputs"] = True
    except Exception:
        globals()["ld_display_outputs"] = False
if "ld_display_outputs_for" not in globals():
    try:
        _ld_for = dbutils.widgets.getAll().get("ld_display_outputs_for")
        globals()["ld_display_outputs_for"] = frozenset(_p.strip() for _p in str(_ld_for).split(",") if _p.strip()) if _ld_for is not None else frozenset()
    except Exception:
        globals()["ld_display_outputs_for"] = frozenset()
ctx = globals().setdefault("ctx", {})
config = {
    "actions": [
        {
            "type": "formula",
            "target": "account_number",
            "expression": "Account_Number__c"
        },
        {
            "type": "formula",
            "target": "usage_date",
            "expression": "Usage_Date__c"
        },
        {
            "type": "formula",
            "target": "active_users",
            "expression": "Active_Users__c"
        },
        {
            "type": "formula",
            "target": "api_calls",
            "expression": "Api_Calls__c"
        },
        {
            "type": "formula",
            "target": "failed_api_calls",
            "expression": "Failed_Api_Calls__c"
        },
        {
            "type": "formula",
            "target": "key_feature_events",
            "expression": "Key_Feature_Events__c"
        },
        {
            "type": "trim",
            "column": "account_number",
            "side": "both"
        },
        {
            "type": "cast",
            "column": "usage_date",
            "to": "date",
            "on_error": "null"
        },
        {
            "type": "cast",
            "column": "active_users",
            "to": "double",
            "on_error": "null"
        },
        {
            "type": "cast",
            "column": "api_calls",
            "to": "double",
            "on_error": "null"
        },
        {
            "type": "cast",
            "column": "failed_api_calls",
            "to": "double",
            "on_error": "null"
        },
        {
            "type": "cast",
            "column": "key_feature_events",
            "to": "double",
            "on_error": "null"
        }
    ]
}
config["meta_state"] = {"is_preview": False, "is_focused_preview": False, "is_disabled": False}
inputs = {
    "data": ctx["source_ae97be2b.data"]
}
if config["meta_state"]["is_disabled"]:
    def _lb_limit_zero(value):
        if hasattr(value, "limit"):
            return value.limit(0)
        if isinstance(value, list):
            return [_lb_limit_zero(item) for item in value]
        return value
    inputs = {k: _lb_limit_zero(v) for k, v in inputs.items()}
out = run(config, inputs, spark)
if config["meta_state"]["is_disabled"]:
    out = {k: _lb_limit_zero(v) for k, v in out.items()}
ctx["prepare_538c1f8d.prepared_data"] = out["prepared_data"]
if globals().get("ld_display_outputs", False) or "prepare_538c1f8d" in globals().get("ld_display_outputs_for", frozenset()):
    display(ctx["prepare_538c1f8d.prepared_data"])

In [0]:
"""
id: prepare_b334c3fa
template: prepare
templateVersion: 1.0.0
name: prep_contacts
position:
  x: 260
  y: 70
description:
  text: Rename columns and trim whitespace from email, title, and department. Convert decision maker flag to boolean.
  hash: 4368df60
previewCodeHash: 2955a9a0c178f2a6
config:
  actions:
    - type: formula
      target: contact_id
      expression: Id
    - type: formula
      target: account_id
      expression: AccountId
    - type: formula
      target: email
      expression: Email
    - type: formula
      target: title
      expression: Title
    - type: formula
      target: department
      expression: Department
    - type: formula
      target: is_decision_maker
      expression: Is_Decision_Maker__c
    - type: trim
      column: email
      side: both
    - type: trim
      column: title
      side: both
    - type: trim
      column: department
      side: both
    - type: cast
      column: is_decision_maker
      to: boolean
      on_error: "null"
input:
  - node: source_b40f8c24
    input_port: data
    output_port: data
"""

# generated from the system
from typing import Any, Callable, Dict, List
import pyspark.sql.functions as F

TEXT_CASE_FUNCTIONS: Dict[str, Callable] = {
    "lower": F.lower,
    "upper": F.upper,
    "title": F.initcap,
}

TRIM_FUNCTIONS: Dict[str, Callable] = {
    "both": F.trim,
    "left": F.ltrim,
    "right": F.rtrim,
}

VALID_MATCH_MODES = {"exact", "contains", "prefix", "suffix", "regex"}

def _require_column(df, column: str, action_type: str) -> None:
    if not column:
        raise ValueError(f"{action_type}: 'column' is required")
    if column not in df.columns:
        raise ValueError(
            f"{action_type}: column '{column}' not found in input data. "
            f"Available columns: {df.columns}"
        )

def _column_dtype(df, column: str) -> str:
    return df.schema[column].dataType.simpleString()

def _quoted_ident(name: str) -> str:
    return "`" + name.replace("`", "``") + "`"

def _col(name: str):
    """Reference a column by exact name.

    F.col parses its argument as a column expression, so a name containing
    a '.' (e.g. "a.b") is otherwise read as struct-field access and fails
    even though the column exists. Backtick-quoting forces an exact-name
    lookup; quoting plain names is harmless.
    """
    return F.col(_quoted_ident(name))

def _coerced_lit(value: Any, target_dtype: str):
    """Wrap a user-provided value as a literal cast to the target column's type.

    UI always feeds text per the operator spec; this is where the
    type coercion happens so the per-action UI stays simple.
    """
    return F.lit(value).cast(target_dtype)

def _apply_formula(df, action: Dict[str, Any]):
    target = action.get("target", "")
    expression = action.get("expression", "")
    if not target:
        raise ValueError("formula: 'target' is required")
    if not expression:
        raise ValueError("formula: 'expression' is required")
    return df.withColumn(target, F.expr(expression))

def _apply_cast(df, action: Dict[str, Any]):
    column = action.get("column", "")
    to_type = action.get("to", "")
    on_error = action.get("on_error", "null")
    _require_column(df, column, "cast")
    if not to_type:
        raise ValueError("cast: 'to' (target type) is required")
    if on_error == "null":
        return df.withColumn(
            column,
            F.expr(f"try_cast({_quoted_ident(column)} as {to_type})"),
        )
    return df.withColumn(column, _col(column).cast(to_type))

def _apply_replace_value(df, action: Dict[str, Any]):
    column = action.get("column", "")
    match_mode = action.get("match_mode", "exact")
    match = action.get("match", "")
    with_val = action.get("with", "")
    case_sensitive = bool(action.get("case_sensitive", False))
    _require_column(df, column, "replace_value")
    if match_mode not in VALID_MATCH_MODES:
        raise ValueError(
            f"replace_value: unsupported match_mode '{match_mode}'. "
            f"Choose one of: {sorted(VALID_MATCH_MODES)}"
        )
    dtype = _column_dtype(df, column)
    col_as_string = _col(column).cast("string")
    match_lit = F.lit(match)
    if case_sensitive:
        cmp_col = col_as_string
        cmp_lit = match_lit
    else:
        cmp_col = F.lower(col_as_string)
        cmp_lit = F.lower(match_lit)

    if match_mode == "exact":
        predicate = cmp_col == cmp_lit
    elif match_mode == "contains":
        predicate = cmp_col.contains(cmp_lit)
    elif match_mode == "prefix":
        predicate = cmp_col.startswith(cmp_lit)
    elif match_mode == "suffix":
        predicate = cmp_col.endswith(cmp_lit)
    else:  # regex
        pattern = match if case_sensitive else f"(?i){match}"
        predicate = col_as_string.rlike(pattern)

    replacement = _coerced_lit(with_val, dtype)
    return df.withColumn(
        column,
        F.when(predicate, replacement).otherwise(_col(column)),
    )

def _apply_fill_null(df, action: Dict[str, Any]):
    column = action.get("column", "")
    with_val = action.get("with", "")
    _require_column(df, column, "fill_null")
    dtype = _column_dtype(df, column)
    replacement = _coerced_lit(with_val, dtype)
    return df.withColumn(
        column,
        F.when(_col(column).isNull(), replacement).otherwise(_col(column)),
    )

def _apply_text_case(df, action: Dict[str, Any]):
    column = action.get("column", "")
    case = action.get("case", "")
    _require_column(df, column, "text_case")
    fn = TEXT_CASE_FUNCTIONS.get(case)
    if fn is None:
        raise ValueError(
            f"text_case: unsupported case '{case}'. "
            f"Choose one of: {sorted(TEXT_CASE_FUNCTIONS.keys())}"
        )
    return df.withColumn(column, fn(_col(column)))

def _apply_trim(df, action: Dict[str, Any]):
    column = action.get("column", "")
    side = action.get("side", "both")
    _require_column(df, column, "trim")
    fn = TRIM_FUNCTIONS.get(side)
    if fn is None:
        raise ValueError(
            f"trim: unsupported side '{side}'. "
            f"Choose one of: {sorted(TRIM_FUNCTIONS.keys())}"
        )
    return df.withColumn(column, fn(_col(column)))

def _apply_regex_replace(df, action: Dict[str, Any]):
    column = action.get("column", "")
    pattern = action.get("pattern", "")
    replacement = action.get("replacement", "")
    _require_column(df, column, "regex_replace")
    return df.withColumn(
        column,
        F.regexp_replace(_col(column), pattern, replacement),
    )

def _apply_extract(df, action: Dict[str, Any]):
    column = action.get("column", "")
    pattern = action.get("pattern", "")
    target = action.get("target") or column
    group_raw = action.get("group", 0)
    _require_column(df, column, "extract")
    try:
        group_idx = int(group_raw)
    except (TypeError, ValueError) as exc:
        raise ValueError(
            f"extract: 'group' must be an integer, got {group_raw!r}"
        ) from exc
    return df.withColumn(
        target,
        F.regexp_extract(_col(column), pattern, group_idx),
    )

def _apply_parse_date(df, action: Dict[str, Any]):
    column = action.get("column", "")
    kind = action.get("kind", "date")
    fmt = action.get("format") or None
    on_error = action.get("on_error", "null")
    _require_column(df, column, "parse_date")
    if kind not in ("date", "timestamp"):
        raise ValueError(
            f"parse_date: unsupported kind '{kind}'. Choose date or timestamp."
        )
    # PySpark exposes F.try_to_timestamp from Spark 3.5 but only adds
    # F.try_to_date in Spark 4.0. Current Databricks Runtimes still ship
    # Spark 3.5, where referencing F.try_to_date raises AttributeError
    # before any data is touched. Route the null-on-error path through
    # primitives that exist in both 3.5 and 4.0:
    #   - F.try_to_timestamp  (PySpark 3.5+, returns NULL under ANSI too)
    #   - SQL try_cast        (Spark 3.5+, returns NULL under ANSI too)
    #   - F.to_date           (PySpark 3.5+; returns NULL on parse failure
    #                          under the default non-ANSI mode, which is
    #                          the Databricks default. Under ANSI mode it
    #                          raises — accepted limitation until
    #                          try_to_date lands on every supported DBR.)
    if on_error == "null":
        if kind == "timestamp":
            if fmt:
                return df.withColumn(
                    column, F.try_to_timestamp(_col(column), F.lit(fmt))
                )
            return df.withColumn(column, F.try_to_timestamp(_col(column)))
        # kind == "date"
        if fmt:
            return df.withColumn(column, F.to_date(_col(column), fmt))
        return df.withColumn(
            column, F.expr(f"try_cast({_quoted_ident(column)} as date)")
        )
    # on_error == "error": surface parse failures as Spark exceptions.
    fn = F.to_date if kind == "date" else F.to_timestamp
    if fmt:
        return df.withColumn(column, fn(_col(column), fmt))
    return df.withColumn(column, fn(_col(column)))

ACTION_DISPATCH: Dict[str, Callable] = {
    "formula": _apply_formula,
    "cast": _apply_cast,
    "replace_value": _apply_replace_value,
    "fill_null": _apply_fill_null,
    "text_case": _apply_text_case,
    "trim": _apply_trim,
    "regex_replace": _apply_regex_replace,
    "extract": _apply_extract,
    "parse_date": _apply_parse_date,
}

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs.get("data")
    actions: List[Dict[str, Any]] = config.get("actions", []) or []

    if not actions:
        return {"prepared_data": df}

    for index, action in enumerate(actions):
        if not isinstance(action, dict):
            raise ValueError(
                f"actions[{index}]: expected an object, got {type(action).__name__}"
            )
        if action.get("enabled", True) is False:
            continue
        action_type = action.get("type", "")
        fn = ACTION_DISPATCH.get(action_type)
        if fn is None:
            raise ValueError(
                f"actions[{index}]: unsupported action type {action_type!r}. "
                f"Choose one of: {sorted(ACTION_DISPATCH.keys())}"
            )
        df = fn(df, action)
    return {"prepared_data": df}

# generated from the system
if "ld_display_outputs" not in globals():
    try:
        _ld_param = dbutils.widgets.getAll().get("ld_display_outputs")
        if _ld_param is not None and str(_ld_param).strip() != "":
            globals()["ld_display_outputs"] = str(_ld_param).strip().lower() not in ("false", "0", "no", "off")
        else:
            try:
                from dbruntime.databricks_repl_context import get_context
                globals()["ld_display_outputs"] = not get_context().isInJob
            except Exception:
                globals()["ld_display_outputs"] = True
    except Exception:
        globals()["ld_display_outputs"] = False
if "ld_display_outputs_for" not in globals():
    try:
        _ld_for = dbutils.widgets.getAll().get("ld_display_outputs_for")
        globals()["ld_display_outputs_for"] = frozenset(_p.strip() for _p in str(_ld_for).split(",") if _p.strip()) if _ld_for is not None else frozenset()
    except Exception:
        globals()["ld_display_outputs_for"] = frozenset()
ctx = globals().setdefault("ctx", {})
config = {
    "actions": [
        {
            "type": "formula",
            "target": "contact_id",
            "expression": "Id"
        },
        {
            "type": "formula",
            "target": "account_id",
            "expression": "AccountId"
        },
        {
            "type": "formula",
            "target": "email",
            "expression": "Email"
        },
        {
            "type": "formula",
            "target": "title",
            "expression": "Title"
        },
        {
            "type": "formula",
            "target": "department",
            "expression": "Department"
        },
        {
            "type": "formula",
            "target": "is_decision_maker",
            "expression": "Is_Decision_Maker__c"
        },
        {
            "type": "trim",
            "column": "email",
            "side": "both"
        },
        {
            "type": "trim",
            "column": "title",
            "side": "both"
        },
        {
            "type": "trim",
            "column": "department",
            "side": "both"
        },
        {
            "type": "cast",
            "column": "is_decision_maker",
            "to": "boolean",
            "on_error": "null"
        }
    ]
}
config["meta_state"] = {"is_preview": False, "is_focused_preview": False, "is_disabled": False}
inputs = {
    "data": ctx["source_b40f8c24.data"]
}
if config["meta_state"]["is_disabled"]:
    def _lb_limit_zero(value):
        if hasattr(value, "limit"):
            return value.limit(0)
        if isinstance(value, list):
            return [_lb_limit_zero(item) for item in value]
        return value
    inputs = {k: _lb_limit_zero(v) for k, v in inputs.items()}
out = run(config, inputs, spark)
if config["meta_state"]["is_disabled"]:
    out = {k: _lb_limit_zero(v) for k, v in out.items()}
ctx["prepare_b334c3fa.prepared_data"] = out["prepared_data"]
if globals().get("ld_display_outputs", False) or "prepare_b334c3fa" in globals().get("ld_display_outputs_for", frozenset()):
    display(ctx["prepare_b334c3fa.prepared_data"])

In [0]:
"""
id: prepare_998f4cab
template: prepare
templateVersion: 1.0.0
name: prep_opportunities
position:
  x: 260
  y: 155
description:
  text: Create new columns by copying existing ones and convert data types for consistency.
  hash: "710e8353"
previewCodeHash: 39b17f6c5040a586
config:
  actions:
    - type: formula
      target: opportunity_id
      expression: Id
    - type: formula
      target: account_id
      expression: AccountId
    - type: formula
      target: stage_name
      expression: StageName
    - type: formula
      target: amount
      expression: Amount
    - type: formula
      target: close_date
      expression: CloseDate
    - type: formula
      target: is_renewal_opportunity
      expression: Renewal_Opportunity__c
    - type: formula
      target: is_expansion_opportunity
      expression: Expansion_Candidate__c
    - type: formula
      target: days_in_stage
      expression: Days_In_Stage__c
    - type: cast
      column: amount
      to: double
      on_error: "null"
    - type: cast
      column: days_in_stage
      to: double
      on_error: "null"
    - type: cast
      column: close_date
      to: date
      on_error: "null"
    - type: cast
      column: is_renewal_opportunity
      to: boolean
      on_error: "null"
    - type: cast
      column: is_expansion_opportunity
      to: boolean
      on_error: "null"
input:
  - node: source_c21ea952
    input_port: data
    output_port: data
"""

# generated from the system
from typing import Any, Callable, Dict, List
import pyspark.sql.functions as F

TEXT_CASE_FUNCTIONS: Dict[str, Callable] = {
    "lower": F.lower,
    "upper": F.upper,
    "title": F.initcap,
}

TRIM_FUNCTIONS: Dict[str, Callable] = {
    "both": F.trim,
    "left": F.ltrim,
    "right": F.rtrim,
}

VALID_MATCH_MODES = {"exact", "contains", "prefix", "suffix", "regex"}

def _require_column(df, column: str, action_type: str) -> None:
    if not column:
        raise ValueError(f"{action_type}: 'column' is required")
    if column not in df.columns:
        raise ValueError(
            f"{action_type}: column '{column}' not found in input data. "
            f"Available columns: {df.columns}"
        )

def _column_dtype(df, column: str) -> str:
    return df.schema[column].dataType.simpleString()

def _quoted_ident(name: str) -> str:
    return "`" + name.replace("`", "``") + "`"

def _col(name: str):
    """Reference a column by exact name.

    F.col parses its argument as a column expression, so a name containing
    a '.' (e.g. "a.b") is otherwise read as struct-field access and fails
    even though the column exists. Backtick-quoting forces an exact-name
    lookup; quoting plain names is harmless.
    """
    return F.col(_quoted_ident(name))

def _coerced_lit(value: Any, target_dtype: str):
    """Wrap a user-provided value as a literal cast to the target column's type.

    UI always feeds text per the operator spec; this is where the
    type coercion happens so the per-action UI stays simple.
    """
    return F.lit(value).cast(target_dtype)

def _apply_formula(df, action: Dict[str, Any]):
    target = action.get("target", "")
    expression = action.get("expression", "")
    if not target:
        raise ValueError("formula: 'target' is required")
    if not expression:
        raise ValueError("formula: 'expression' is required")
    return df.withColumn(target, F.expr(expression))

def _apply_cast(df, action: Dict[str, Any]):
    column = action.get("column", "")
    to_type = action.get("to", "")
    on_error = action.get("on_error", "null")
    _require_column(df, column, "cast")
    if not to_type:
        raise ValueError("cast: 'to' (target type) is required")
    if on_error == "null":
        return df.withColumn(
            column,
            F.expr(f"try_cast({_quoted_ident(column)} as {to_type})"),
        )
    return df.withColumn(column, _col(column).cast(to_type))

def _apply_replace_value(df, action: Dict[str, Any]):
    column = action.get("column", "")
    match_mode = action.get("match_mode", "exact")
    match = action.get("match", "")
    with_val = action.get("with", "")
    case_sensitive = bool(action.get("case_sensitive", False))
    _require_column(df, column, "replace_value")
    if match_mode not in VALID_MATCH_MODES:
        raise ValueError(
            f"replace_value: unsupported match_mode '{match_mode}'. "
            f"Choose one of: {sorted(VALID_MATCH_MODES)}"
        )
    dtype = _column_dtype(df, column)
    col_as_string = _col(column).cast("string")
    match_lit = F.lit(match)
    if case_sensitive:
        cmp_col = col_as_string
        cmp_lit = match_lit
    else:
        cmp_col = F.lower(col_as_string)
        cmp_lit = F.lower(match_lit)

    if match_mode == "exact":
        predicate = cmp_col == cmp_lit
    elif match_mode == "contains":
        predicate = cmp_col.contains(cmp_lit)
    elif match_mode == "prefix":
        predicate = cmp_col.startswith(cmp_lit)
    elif match_mode == "suffix":
        predicate = cmp_col.endswith(cmp_lit)
    else:  # regex
        pattern = match if case_sensitive else f"(?i){match}"
        predicate = col_as_string.rlike(pattern)

    replacement = _coerced_lit(with_val, dtype)
    return df.withColumn(
        column,
        F.when(predicate, replacement).otherwise(_col(column)),
    )

def _apply_fill_null(df, action: Dict[str, Any]):
    column = action.get("column", "")
    with_val = action.get("with", "")
    _require_column(df, column, "fill_null")
    dtype = _column_dtype(df, column)
    replacement = _coerced_lit(with_val, dtype)
    return df.withColumn(
        column,
        F.when(_col(column).isNull(), replacement).otherwise(_col(column)),
    )

def _apply_text_case(df, action: Dict[str, Any]):
    column = action.get("column", "")
    case = action.get("case", "")
    _require_column(df, column, "text_case")
    fn = TEXT_CASE_FUNCTIONS.get(case)
    if fn is None:
        raise ValueError(
            f"text_case: unsupported case '{case}'. "
            f"Choose one of: {sorted(TEXT_CASE_FUNCTIONS.keys())}"
        )
    return df.withColumn(column, fn(_col(column)))

def _apply_trim(df, action: Dict[str, Any]):
    column = action.get("column", "")
    side = action.get("side", "both")
    _require_column(df, column, "trim")
    fn = TRIM_FUNCTIONS.get(side)
    if fn is None:
        raise ValueError(
            f"trim: unsupported side '{side}'. "
            f"Choose one of: {sorted(TRIM_FUNCTIONS.keys())}"
        )
    return df.withColumn(column, fn(_col(column)))

def _apply_regex_replace(df, action: Dict[str, Any]):
    column = action.get("column", "")
    pattern = action.get("pattern", "")
    replacement = action.get("replacement", "")
    _require_column(df, column, "regex_replace")
    return df.withColumn(
        column,
        F.regexp_replace(_col(column), pattern, replacement),
    )

def _apply_extract(df, action: Dict[str, Any]):
    column = action.get("column", "")
    pattern = action.get("pattern", "")
    target = action.get("target") or column
    group_raw = action.get("group", 0)
    _require_column(df, column, "extract")
    try:
        group_idx = int(group_raw)
    except (TypeError, ValueError) as exc:
        raise ValueError(
            f"extract: 'group' must be an integer, got {group_raw!r}"
        ) from exc
    return df.withColumn(
        target,
        F.regexp_extract(_col(column), pattern, group_idx),
    )

def _apply_parse_date(df, action: Dict[str, Any]):
    column = action.get("column", "")
    kind = action.get("kind", "date")
    fmt = action.get("format") or None
    on_error = action.get("on_error", "null")
    _require_column(df, column, "parse_date")
    if kind not in ("date", "timestamp"):
        raise ValueError(
            f"parse_date: unsupported kind '{kind}'. Choose date or timestamp."
        )
    # PySpark exposes F.try_to_timestamp from Spark 3.5 but only adds
    # F.try_to_date in Spark 4.0. Current Databricks Runtimes still ship
    # Spark 3.5, where referencing F.try_to_date raises AttributeError
    # before any data is touched. Route the null-on-error path through
    # primitives that exist in both 3.5 and 4.0:
    #   - F.try_to_timestamp  (PySpark 3.5+, returns NULL under ANSI too)
    #   - SQL try_cast        (Spark 3.5+, returns NULL under ANSI too)
    #   - F.to_date           (PySpark 3.5+; returns NULL on parse failure
    #                          under the default non-ANSI mode, which is
    #                          the Databricks default. Under ANSI mode it
    #                          raises — accepted limitation until
    #                          try_to_date lands on every supported DBR.)
    if on_error == "null":
        if kind == "timestamp":
            if fmt:
                return df.withColumn(
                    column, F.try_to_timestamp(_col(column), F.lit(fmt))
                )
            return df.withColumn(column, F.try_to_timestamp(_col(column)))
        # kind == "date"
        if fmt:
            return df.withColumn(column, F.to_date(_col(column), fmt))
        return df.withColumn(
            column, F.expr(f"try_cast({_quoted_ident(column)} as date)")
        )
    # on_error == "error": surface parse failures as Spark exceptions.
    fn = F.to_date if kind == "date" else F.to_timestamp
    if fmt:
        return df.withColumn(column, fn(_col(column), fmt))
    return df.withColumn(column, fn(_col(column)))

ACTION_DISPATCH: Dict[str, Callable] = {
    "formula": _apply_formula,
    "cast": _apply_cast,
    "replace_value": _apply_replace_value,
    "fill_null": _apply_fill_null,
    "text_case": _apply_text_case,
    "trim": _apply_trim,
    "regex_replace": _apply_regex_replace,
    "extract": _apply_extract,
    "parse_date": _apply_parse_date,
}

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs.get("data")
    actions: List[Dict[str, Any]] = config.get("actions", []) or []

    if not actions:
        return {"prepared_data": df}

    for index, action in enumerate(actions):
        if not isinstance(action, dict):
            raise ValueError(
                f"actions[{index}]: expected an object, got {type(action).__name__}"
            )
        if action.get("enabled", True) is False:
            continue
        action_type = action.get("type", "")
        fn = ACTION_DISPATCH.get(action_type)
        if fn is None:
            raise ValueError(
                f"actions[{index}]: unsupported action type {action_type!r}. "
                f"Choose one of: {sorted(ACTION_DISPATCH.keys())}"
            )
        df = fn(df, action)
    return {"prepared_data": df}

# generated from the system
if "ld_display_outputs" not in globals():
    try:
        _ld_param = dbutils.widgets.getAll().get("ld_display_outputs")
        if _ld_param is not None and str(_ld_param).strip() != "":
            globals()["ld_display_outputs"] = str(_ld_param).strip().lower() not in ("false", "0", "no", "off")
        else:
            try:
                from dbruntime.databricks_repl_context import get_context
                globals()["ld_display_outputs"] = not get_context().isInJob
            except Exception:
                globals()["ld_display_outputs"] = True
    except Exception:
        globals()["ld_display_outputs"] = False
if "ld_display_outputs_for" not in globals():
    try:
        _ld_for = dbutils.widgets.getAll().get("ld_display_outputs_for")
        globals()["ld_display_outputs_for"] = frozenset(_p.strip() for _p in str(_ld_for).split(",") if _p.strip()) if _ld_for is not None else frozenset()
    except Exception:
        globals()["ld_display_outputs_for"] = frozenset()
ctx = globals().setdefault("ctx", {})
config = {
    "actions": [
        {
            "type": "formula",
            "target": "opportunity_id",
            "expression": "Id"
        },
        {
            "type": "formula",
            "target": "account_id",
            "expression": "AccountId"
        },
        {
            "type": "formula",
            "target": "stage_name",
            "expression": "StageName"
        },
        {
            "type": "formula",
            "target": "amount",
            "expression": "Amount"
        },
        {
            "type": "formula",
            "target": "close_date",
            "expression": "CloseDate"
        },
        {
            "type": "formula",
            "target": "is_renewal_opportunity",
            "expression": "Renewal_Opportunity__c"
        },
        {
            "type": "formula",
            "target": "is_expansion_opportunity",
            "expression": "Expansion_Candidate__c"
        },
        {
            "type": "formula",
            "target": "days_in_stage",
            "expression": "Days_In_Stage__c"
        },
        {
            "type": "cast",
            "column": "amount",
            "to": "double",
            "on_error": "null"
        },
        {
            "type": "cast",
            "column": "days_in_stage",
            "to": "double",
            "on_error": "null"
        },
        {
            "type": "cast",
            "column": "close_date",
            "to": "date",
            "on_error": "null"
        },
        {
            "type": "cast",
            "column": "is_renewal_opportunity",
            "to": "boolean",
            "on_error": "null"
        },
        {
            "type": "cast",
            "column": "is_expansion_opportunity",
            "to": "boolean",
            "on_error": "null"
        }
    ]
}
config["meta_state"] = {"is_preview": False, "is_focused_preview": False, "is_disabled": False}
inputs = {
    "data": ctx["source_c21ea952.data"]
}
if config["meta_state"]["is_disabled"]:
    def _lb_limit_zero(value):
        if hasattr(value, "limit"):
            return value.limit(0)
        if isinstance(value, list):
            return [_lb_limit_zero(item) for item in value]
        return value
    inputs = {k: _lb_limit_zero(v) for k, v in inputs.items()}
out = run(config, inputs, spark)
if config["meta_state"]["is_disabled"]:
    out = {k: _lb_limit_zero(v) for k, v in out.items()}
ctx["prepare_998f4cab.prepared_data"] = out["prepared_data"]
if globals().get("ld_display_outputs", False) or "prepare_998f4cab" in globals().get("ld_display_outputs_for", frozenset()):
    display(ctx["prepare_998f4cab.prepared_data"])

In [0]:
"""
id: prepare_688e7190
template: prepare
templateVersion: 1.0.0
name: prep_cases
position:
  x: 260
  y: 420
description:
  text: Create new columns from existing data, convert date and boolean columns, add a flag for open cases, and calculate case age in days.
  hash: "16684885"
previewCodeHash: b971f5437b0c8a89
config:
  actions:
    - type: formula
      target: case_id
      expression: Id
    - type: formula
      target: account_id
      expression: AccountId
    - type: formula
      target: case_number
      expression: CaseNumber
    - type: formula
      target: priority
      expression: Priority
    - type: formula
      target: status
      expression: Status
    - type: formula
      target: reported_date
      expression: Reported_Date__c
    - type: formula
      target: resolved_date
      expression: Resolved_Date__c
    - type: formula
      target: sla_breached
      expression: SLA_Breached__c
    - type: cast
      column: reported_date
      to: date
      on_error: "null"
    - type: cast
      column: resolved_date
      to: date
      on_error: "null"
    - type: cast
      column: sla_breached
      to: boolean
      on_error: "null"
    - type: formula
      target: is_open
      expression: status <> 'Closed'
    - type: formula
      target: case_age_days
      expression: DATEDIFF(current_date(), reported_date)
input:
  - node: source_e212f0ab
    input_port: data
    output_port: data
"""

# generated from the system
from typing import Any, Callable, Dict, List
import pyspark.sql.functions as F

TEXT_CASE_FUNCTIONS: Dict[str, Callable] = {
    "lower": F.lower,
    "upper": F.upper,
    "title": F.initcap,
}

TRIM_FUNCTIONS: Dict[str, Callable] = {
    "both": F.trim,
    "left": F.ltrim,
    "right": F.rtrim,
}

VALID_MATCH_MODES = {"exact", "contains", "prefix", "suffix", "regex"}

def _require_column(df, column: str, action_type: str) -> None:
    if not column:
        raise ValueError(f"{action_type}: 'column' is required")
    if column not in df.columns:
        raise ValueError(
            f"{action_type}: column '{column}' not found in input data. "
            f"Available columns: {df.columns}"
        )

def _column_dtype(df, column: str) -> str:
    return df.schema[column].dataType.simpleString()

def _quoted_ident(name: str) -> str:
    return "`" + name.replace("`", "``") + "`"

def _col(name: str):
    """Reference a column by exact name.

    F.col parses its argument as a column expression, so a name containing
    a '.' (e.g. "a.b") is otherwise read as struct-field access and fails
    even though the column exists. Backtick-quoting forces an exact-name
    lookup; quoting plain names is harmless.
    """
    return F.col(_quoted_ident(name))

def _coerced_lit(value: Any, target_dtype: str):
    """Wrap a user-provided value as a literal cast to the target column's type.

    UI always feeds text per the operator spec; this is where the
    type coercion happens so the per-action UI stays simple.
    """
    return F.lit(value).cast(target_dtype)

def _apply_formula(df, action: Dict[str, Any]):
    target = action.get("target", "")
    expression = action.get("expression", "")
    if not target:
        raise ValueError("formula: 'target' is required")
    if not expression:
        raise ValueError("formula: 'expression' is required")
    return df.withColumn(target, F.expr(expression))

def _apply_cast(df, action: Dict[str, Any]):
    column = action.get("column", "")
    to_type = action.get("to", "")
    on_error = action.get("on_error", "null")
    _require_column(df, column, "cast")
    if not to_type:
        raise ValueError("cast: 'to' (target type) is required")
    if on_error == "null":
        return df.withColumn(
            column,
            F.expr(f"try_cast({_quoted_ident(column)} as {to_type})"),
        )
    return df.withColumn(column, _col(column).cast(to_type))

def _apply_replace_value(df, action: Dict[str, Any]):
    column = action.get("column", "")
    match_mode = action.get("match_mode", "exact")
    match = action.get("match", "")
    with_val = action.get("with", "")
    case_sensitive = bool(action.get("case_sensitive", False))
    _require_column(df, column, "replace_value")
    if match_mode not in VALID_MATCH_MODES:
        raise ValueError(
            f"replace_value: unsupported match_mode '{match_mode}'. "
            f"Choose one of: {sorted(VALID_MATCH_MODES)}"
        )
    dtype = _column_dtype(df, column)
    col_as_string = _col(column).cast("string")
    match_lit = F.lit(match)
    if case_sensitive:
        cmp_col = col_as_string
        cmp_lit = match_lit
    else:
        cmp_col = F.lower(col_as_string)
        cmp_lit = F.lower(match_lit)

    if match_mode == "exact":
        predicate = cmp_col == cmp_lit
    elif match_mode == "contains":
        predicate = cmp_col.contains(cmp_lit)
    elif match_mode == "prefix":
        predicate = cmp_col.startswith(cmp_lit)
    elif match_mode == "suffix":
        predicate = cmp_col.endswith(cmp_lit)
    else:  # regex
        pattern = match if case_sensitive else f"(?i){match}"
        predicate = col_as_string.rlike(pattern)

    replacement = _coerced_lit(with_val, dtype)
    return df.withColumn(
        column,
        F.when(predicate, replacement).otherwise(_col(column)),
    )

def _apply_fill_null(df, action: Dict[str, Any]):
    column = action.get("column", "")
    with_val = action.get("with", "")
    _require_column(df, column, "fill_null")
    dtype = _column_dtype(df, column)
    replacement = _coerced_lit(with_val, dtype)
    return df.withColumn(
        column,
        F.when(_col(column).isNull(), replacement).otherwise(_col(column)),
    )

def _apply_text_case(df, action: Dict[str, Any]):
    column = action.get("column", "")
    case = action.get("case", "")
    _require_column(df, column, "text_case")
    fn = TEXT_CASE_FUNCTIONS.get(case)
    if fn is None:
        raise ValueError(
            f"text_case: unsupported case '{case}'. "
            f"Choose one of: {sorted(TEXT_CASE_FUNCTIONS.keys())}"
        )
    return df.withColumn(column, fn(_col(column)))

def _apply_trim(df, action: Dict[str, Any]):
    column = action.get("column", "")
    side = action.get("side", "both")
    _require_column(df, column, "trim")
    fn = TRIM_FUNCTIONS.get(side)
    if fn is None:
        raise ValueError(
            f"trim: unsupported side '{side}'. "
            f"Choose one of: {sorted(TRIM_FUNCTIONS.keys())}"
        )
    return df.withColumn(column, fn(_col(column)))

def _apply_regex_replace(df, action: Dict[str, Any]):
    column = action.get("column", "")
    pattern = action.get("pattern", "")
    replacement = action.get("replacement", "")
    _require_column(df, column, "regex_replace")
    return df.withColumn(
        column,
        F.regexp_replace(_col(column), pattern, replacement),
    )

def _apply_extract(df, action: Dict[str, Any]):
    column = action.get("column", "")
    pattern = action.get("pattern", "")
    target = action.get("target") or column
    group_raw = action.get("group", 0)
    _require_column(df, column, "extract")
    try:
        group_idx = int(group_raw)
    except (TypeError, ValueError) as exc:
        raise ValueError(
            f"extract: 'group' must be an integer, got {group_raw!r}"
        ) from exc
    return df.withColumn(
        target,
        F.regexp_extract(_col(column), pattern, group_idx),
    )

def _apply_parse_date(df, action: Dict[str, Any]):
    column = action.get("column", "")
    kind = action.get("kind", "date")
    fmt = action.get("format") or None
    on_error = action.get("on_error", "null")
    _require_column(df, column, "parse_date")
    if kind not in ("date", "timestamp"):
        raise ValueError(
            f"parse_date: unsupported kind '{kind}'. Choose date or timestamp."
        )
    # PySpark exposes F.try_to_timestamp from Spark 3.5 but only adds
    # F.try_to_date in Spark 4.0. Current Databricks Runtimes still ship
    # Spark 3.5, where referencing F.try_to_date raises AttributeError
    # before any data is touched. Route the null-on-error path through
    # primitives that exist in both 3.5 and 4.0:
    #   - F.try_to_timestamp  (PySpark 3.5+, returns NULL under ANSI too)
    #   - SQL try_cast        (Spark 3.5+, returns NULL under ANSI too)
    #   - F.to_date           (PySpark 3.5+; returns NULL on parse failure
    #                          under the default non-ANSI mode, which is
    #                          the Databricks default. Under ANSI mode it
    #                          raises — accepted limitation until
    #                          try_to_date lands on every supported DBR.)
    if on_error == "null":
        if kind == "timestamp":
            if fmt:
                return df.withColumn(
                    column, F.try_to_timestamp(_col(column), F.lit(fmt))
                )
            return df.withColumn(column, F.try_to_timestamp(_col(column)))
        # kind == "date"
        if fmt:
            return df.withColumn(column, F.to_date(_col(column), fmt))
        return df.withColumn(
            column, F.expr(f"try_cast({_quoted_ident(column)} as date)")
        )
    # on_error == "error": surface parse failures as Spark exceptions.
    fn = F.to_date if kind == "date" else F.to_timestamp
    if fmt:
        return df.withColumn(column, fn(_col(column), fmt))
    return df.withColumn(column, fn(_col(column)))

ACTION_DISPATCH: Dict[str, Callable] = {
    "formula": _apply_formula,
    "cast": _apply_cast,
    "replace_value": _apply_replace_value,
    "fill_null": _apply_fill_null,
    "text_case": _apply_text_case,
    "trim": _apply_trim,
    "regex_replace": _apply_regex_replace,
    "extract": _apply_extract,
    "parse_date": _apply_parse_date,
}

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs.get("data")
    actions: List[Dict[str, Any]] = config.get("actions", []) or []

    if not actions:
        return {"prepared_data": df}

    for index, action in enumerate(actions):
        if not isinstance(action, dict):
            raise ValueError(
                f"actions[{index}]: expected an object, got {type(action).__name__}"
            )
        if action.get("enabled", True) is False:
            continue
        action_type = action.get("type", "")
        fn = ACTION_DISPATCH.get(action_type)
        if fn is None:
            raise ValueError(
                f"actions[{index}]: unsupported action type {action_type!r}. "
                f"Choose one of: {sorted(ACTION_DISPATCH.keys())}"
            )
        df = fn(df, action)
    return {"prepared_data": df}

# generated from the system
if "ld_display_outputs" not in globals():
    try:
        _ld_param = dbutils.widgets.getAll().get("ld_display_outputs")
        if _ld_param is not None and str(_ld_param).strip() != "":
            globals()["ld_display_outputs"] = str(_ld_param).strip().lower() not in ("false", "0", "no", "off")
        else:
            try:
                from dbruntime.databricks_repl_context import get_context
                globals()["ld_display_outputs"] = not get_context().isInJob
            except Exception:
                globals()["ld_display_outputs"] = True
    except Exception:
        globals()["ld_display_outputs"] = False
if "ld_display_outputs_for" not in globals():
    try:
        _ld_for = dbutils.widgets.getAll().get("ld_display_outputs_for")
        globals()["ld_display_outputs_for"] = frozenset(_p.strip() for _p in str(_ld_for).split(",") if _p.strip()) if _ld_for is not None else frozenset()
    except Exception:
        globals()["ld_display_outputs_for"] = frozenset()
ctx = globals().setdefault("ctx", {})
config = {
    "actions": [
        {
            "type": "formula",
            "target": "case_id",
            "expression": "Id"
        },
        {
            "type": "formula",
            "target": "account_id",
            "expression": "AccountId"
        },
        {
            "type": "formula",
            "target": "case_number",
            "expression": "CaseNumber"
        },
        {
            "type": "formula",
            "target": "priority",
            "expression": "Priority"
        },
        {
            "type": "formula",
            "target": "status",
            "expression": "Status"
        },
        {
            "type": "formula",
            "target": "reported_date",
            "expression": "Reported_Date__c"
        },
        {
            "type": "formula",
            "target": "resolved_date",
            "expression": "Resolved_Date__c"
        },
        {
            "type": "formula",
            "target": "sla_breached",
            "expression": "SLA_Breached__c"
        },
        {
            "type": "cast",
            "column": "reported_date",
            "to": "date",
            "on_error": "null"
        },
        {
            "type": "cast",
            "column": "resolved_date",
            "to": "date",
            "on_error": "null"
        },
        {
            "type": "cast",
            "column": "sla_breached",
            "to": "boolean",
            "on_error": "null"
        },
        {
            "type": "formula",
            "target": "is_open",
            "expression": "status <> 'Closed'"
        },
        {
            "type": "formula",
            "target": "case_age_days",
            "expression": "DATEDIFF(current_date(), reported_date)"
        }
    ]
}
config["meta_state"] = {"is_preview": False, "is_focused_preview": False, "is_disabled": False}
inputs = {
    "data": ctx["source_e212f0ab.data"]
}
if config["meta_state"]["is_disabled"]:
    def _lb_limit_zero(value):
        if hasattr(value, "limit"):
            return value.limit(0)
        if isinstance(value, list):
            return [_lb_limit_zero(item) for item in value]
        return value
    inputs = {k: _lb_limit_zero(v) for k, v in inputs.items()}
out = run(config, inputs, spark)
if config["meta_state"]["is_disabled"]:
    out = {k: _lb_limit_zero(v) for k, v in out.items()}
ctx["prepare_688e7190.prepared_data"] = out["prepared_data"]
if globals().get("ld_display_outputs", False) or "prepare_688e7190" in globals().get("ld_display_outputs_for", frozenset()):
    display(ctx["prepare_688e7190.prepared_data"])

In [0]:
"""
id: filter_b72e1f84
template: filter
templateVersion: 2.0.0
name: filter_valid_accounts
position:
  x: 520
  y: 260
description:
  text: Keep rows where account_id and account_number are not missing; separate the rest.
  hash: 6c777a50
previewCodeHash: 49c486cf4eb7ce4e
config:
  condition: account_id IS NOT NULL AND account_number IS NOT NULL
input:
  - node: prepare_a0f1c2d3
    input_port: data
    output_port: prepared_data
"""

# generated from the system
from typing import Dict, Any

from pyspark.sql import functions as F

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs["data"]
    condition = config.get("condition", "")

    if not condition:
        return {"filtered_data": df, "excluded_data": spark.createDataFrame([], df.schema)}

    keep = F.coalesce(F.expr(condition), F.lit(False))
    return {"filtered_data": df.filter(keep), "excluded_data": df.filter(~keep)}

# generated from the system
if "ld_display_outputs" not in globals():
    try:
        _ld_param = dbutils.widgets.getAll().get("ld_display_outputs")
        if _ld_param is not None and str(_ld_param).strip() != "":
            globals()["ld_display_outputs"] = str(_ld_param).strip().lower() not in ("false", "0", "no", "off")
        else:
            try:
                from dbruntime.databricks_repl_context import get_context
                globals()["ld_display_outputs"] = not get_context().isInJob
            except Exception:
                globals()["ld_display_outputs"] = True
    except Exception:
        globals()["ld_display_outputs"] = False
if "ld_display_outputs_for" not in globals():
    try:
        _ld_for = dbutils.widgets.getAll().get("ld_display_outputs_for")
        globals()["ld_display_outputs_for"] = frozenset(_p.strip() for _p in str(_ld_for).split(",") if _p.strip()) if _ld_for is not None else frozenset()
    except Exception:
        globals()["ld_display_outputs_for"] = frozenset()
ctx = globals().setdefault("ctx", {})
config = {
    "condition": "account_id IS NOT NULL AND account_number IS NOT NULL"
}
config["meta_state"] = {"is_preview": False, "is_focused_preview": False, "is_disabled": False}
inputs = {
    "data": ctx["prepare_a0f1c2d3.prepared_data"]
}
if config["meta_state"]["is_disabled"]:
    def _lb_limit_zero(value):
        if hasattr(value, "limit"):
            return value.limit(0)
        if isinstance(value, list):
            return [_lb_limit_zero(item) for item in value]
        return value
    inputs = {k: _lb_limit_zero(v) for k, v in inputs.items()}
out = run(config, inputs, spark)
if config["meta_state"]["is_disabled"]:
    out = {k: _lb_limit_zero(v) for k, v in out.items()}
ctx["filter_b72e1f84.filtered_data"] = out["filtered_data"]
ctx["filter_b72e1f84.excluded_data"] = out["excluded_data"]
if globals().get("ld_display_outputs", False) or "filter_b72e1f84" in globals().get("ld_display_outputs_for", frozenset()):
    display(ctx["filter_b72e1f84.filtered_data"])
    display(ctx["filter_b72e1f84.excluded_data"])

In [0]:
"""
id: filter_950cd717
template: filter
templateVersion: 2.0.0
name: filter_valid_usage
position:
  x: 520
  y: 680
description:
  text: "Split data into two parts based on a condition: one that meets the condition and one that does not."
  hash: 3e9139bc
previewCodeHash: ebc7a4603b7de9aa
config:
  condition: account_number IS NOT NULL AND usage_date IS NOT NULL AND active_users >= 0 AND api_calls >= 0 AND failed_api_calls >= 0
input:
  - node: prepare_538c1f8d
    input_port: data
    output_port: prepared_data
"""

# generated from the system
from typing import Dict, Any

from pyspark.sql import functions as F

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs["data"]
    condition = config.get("condition", "")

    if not condition:
        return {"filtered_data": df, "excluded_data": spark.createDataFrame([], df.schema)}

    keep = F.coalesce(F.expr(condition), F.lit(False))
    return {"filtered_data": df.filter(keep), "excluded_data": df.filter(~keep)}

# generated from the system
if "ld_display_outputs" not in globals():
    try:
        _ld_param = dbutils.widgets.getAll().get("ld_display_outputs")
        if _ld_param is not None and str(_ld_param).strip() != "":
            globals()["ld_display_outputs"] = str(_ld_param).strip().lower() not in ("false", "0", "no", "off")
        else:
            try:
                from dbruntime.databricks_repl_context import get_context
                globals()["ld_display_outputs"] = not get_context().isInJob
            except Exception:
                globals()["ld_display_outputs"] = True
    except Exception:
        globals()["ld_display_outputs"] = False
if "ld_display_outputs_for" not in globals():
    try:
        _ld_for = dbutils.widgets.getAll().get("ld_display_outputs_for")
        globals()["ld_display_outputs_for"] = frozenset(_p.strip() for _p in str(_ld_for).split(",") if _p.strip()) if _ld_for is not None else frozenset()
    except Exception:
        globals()["ld_display_outputs_for"] = frozenset()
ctx = globals().setdefault("ctx", {})
config = {
    "condition": "account_number IS NOT NULL AND usage_date IS NOT NULL AND active_users >= 0 AND api_calls >= 0 AND failed_api_calls >= 0"
}
config["meta_state"] = {"is_preview": False, "is_focused_preview": False, "is_disabled": False}
inputs = {
    "data": ctx["prepare_538c1f8d.prepared_data"]
}
if config["meta_state"]["is_disabled"]:
    def _lb_limit_zero(value):
        if hasattr(value, "limit"):
            return value.limit(0)
        if isinstance(value, list):
            return [_lb_limit_zero(item) for item in value]
        return value
    inputs = {k: _lb_limit_zero(v) for k, v in inputs.items()}
out = run(config, inputs, spark)
if config["meta_state"]["is_disabled"]:
    out = {k: _lb_limit_zero(v) for k, v in out.items()}
ctx["filter_950cd717.filtered_data"] = out["filtered_data"]
ctx["filter_950cd717.excluded_data"] = out["excluded_data"]
if globals().get("ld_display_outputs", False) or "filter_950cd717" in globals().get("ld_display_outputs_for", frozenset()):
    display(ctx["filter_950cd717.filtered_data"])
    display(ctx["filter_950cd717.excluded_data"])

In [0]:
"""
id: filter_e52e6b11
template: filter
templateVersion: 2.0.0
name: filter_valid_contacts
position:
  x: 520
  y: 70
description:
  text: Keep rows where account_id exists and email contains '@'; separate the rest.
  hash: b4e97e4a
previewCodeHash: 35d26ac88109d210
config:
  condition: account_id IS NOT NULL AND email LIKE '%@%'
input:
  - node: prepare_b334c3fa
    input_port: data
    output_port: prepared_data
"""

# generated from the system
from typing import Dict, Any

from pyspark.sql import functions as F

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs["data"]
    condition = config.get("condition", "")

    if not condition:
        return {"filtered_data": df, "excluded_data": spark.createDataFrame([], df.schema)}

    keep = F.coalesce(F.expr(condition), F.lit(False))
    return {"filtered_data": df.filter(keep), "excluded_data": df.filter(~keep)}

# generated from the system
if "ld_display_outputs" not in globals():
    try:
        _ld_param = dbutils.widgets.getAll().get("ld_display_outputs")
        if _ld_param is not None and str(_ld_param).strip() != "":
            globals()["ld_display_outputs"] = str(_ld_param).strip().lower() not in ("false", "0", "no", "off")
        else:
            try:
                from dbruntime.databricks_repl_context import get_context
                globals()["ld_display_outputs"] = not get_context().isInJob
            except Exception:
                globals()["ld_display_outputs"] = True
    except Exception:
        globals()["ld_display_outputs"] = False
if "ld_display_outputs_for" not in globals():
    try:
        _ld_for = dbutils.widgets.getAll().get("ld_display_outputs_for")
        globals()["ld_display_outputs_for"] = frozenset(_p.strip() for _p in str(_ld_for).split(",") if _p.strip()) if _ld_for is not None else frozenset()
    except Exception:
        globals()["ld_display_outputs_for"] = frozenset()
ctx = globals().setdefault("ctx", {})
config = {
    "condition": "account_id IS NOT NULL AND email LIKE '%@%'"
}
config["meta_state"] = {"is_preview": False, "is_focused_preview": False, "is_disabled": False}
inputs = {
    "data": ctx["prepare_b334c3fa.prepared_data"]
}
if config["meta_state"]["is_disabled"]:
    def _lb_limit_zero(value):
        if hasattr(value, "limit"):
            return value.limit(0)
        if isinstance(value, list):
            return [_lb_limit_zero(item) for item in value]
        return value
    inputs = {k: _lb_limit_zero(v) for k, v in inputs.items()}
out = run(config, inputs, spark)
if config["meta_state"]["is_disabled"]:
    out = {k: _lb_limit_zero(v) for k, v in out.items()}
ctx["filter_e52e6b11.filtered_data"] = out["filtered_data"]
ctx["filter_e52e6b11.excluded_data"] = out["excluded_data"]
if globals().get("ld_display_outputs", False) or "filter_e52e6b11" in globals().get("ld_display_outputs_for", frozenset()):
    display(ctx["filter_e52e6b11.filtered_data"])
    display(ctx["filter_e52e6b11.excluded_data"])

In [0]:
"""
id: filter_81ec30bb
template: filter
templateVersion: 2.0.0
name: filter_valid_opportunities
position:
  x: 520
  y: 155
description:
  text: Keep rows where account_id is not null.
  hash: ad550678
previewCodeHash: cf39f293feccade7
config:
  condition: account_id IS NOT NULL
input:
  - node: prepare_998f4cab
    input_port: data
    output_port: prepared_data
"""

# generated from the system
from typing import Dict, Any

from pyspark.sql import functions as F

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs["data"]
    condition = config.get("condition", "")

    if not condition:
        return {"filtered_data": df, "excluded_data": spark.createDataFrame([], df.schema)}

    keep = F.coalesce(F.expr(condition), F.lit(False))
    return {"filtered_data": df.filter(keep), "excluded_data": df.filter(~keep)}

# generated from the system
if "ld_display_outputs" not in globals():
    try:
        _ld_param = dbutils.widgets.getAll().get("ld_display_outputs")
        if _ld_param is not None and str(_ld_param).strip() != "":
            globals()["ld_display_outputs"] = str(_ld_param).strip().lower() not in ("false", "0", "no", "off")
        else:
            try:
                from dbruntime.databricks_repl_context import get_context
                globals()["ld_display_outputs"] = not get_context().isInJob
            except Exception:
                globals()["ld_display_outputs"] = True
    except Exception:
        globals()["ld_display_outputs"] = False
if "ld_display_outputs_for" not in globals():
    try:
        _ld_for = dbutils.widgets.getAll().get("ld_display_outputs_for")
        globals()["ld_display_outputs_for"] = frozenset(_p.strip() for _p in str(_ld_for).split(",") if _p.strip()) if _ld_for is not None else frozenset()
    except Exception:
        globals()["ld_display_outputs_for"] = frozenset()
ctx = globals().setdefault("ctx", {})
config = {
    "condition": "account_id IS NOT NULL"
}
config["meta_state"] = {"is_preview": False, "is_focused_preview": False, "is_disabled": False}
inputs = {
    "data": ctx["prepare_998f4cab.prepared_data"]
}
if config["meta_state"]["is_disabled"]:
    def _lb_limit_zero(value):
        if hasattr(value, "limit"):
            return value.limit(0)
        if isinstance(value, list):
            return [_lb_limit_zero(item) for item in value]
        return value
    inputs = {k: _lb_limit_zero(v) for k, v in inputs.items()}
out = run(config, inputs, spark)
if config["meta_state"]["is_disabled"]:
    out = {k: _lb_limit_zero(v) for k, v in out.items()}
ctx["filter_81ec30bb.filtered_data"] = out["filtered_data"]
ctx["filter_81ec30bb.excluded_data"] = out["excluded_data"]
if globals().get("ld_display_outputs", False) or "filter_81ec30bb" in globals().get("ld_display_outputs_for", frozenset()):
    display(ctx["filter_81ec30bb.filtered_data"])
    display(ctx["filter_81ec30bb.excluded_data"])

In [0]:
"""
id: filter_998c35a1
template: filter
templateVersion: 2.0.0
name: filter_valid_cases
position:
  x: 520
  y: 420
description:
  text: Keep rows where account ID exists and priority is Low, Medium, High, or Critical; exclude others.
  hash: ad07079e
previewCodeHash: 532d41e3881772e1
config:
  condition: account_id IS NOT NULL AND priority IN ('Low', 'Medium', 'High', 'Critical')
input:
  - node: prepare_688e7190
    input_port: data
    output_port: prepared_data
"""

# generated from the system
from typing import Dict, Any

from pyspark.sql import functions as F

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs["data"]
    condition = config.get("condition", "")

    if not condition:
        return {"filtered_data": df, "excluded_data": spark.createDataFrame([], df.schema)}

    keep = F.coalesce(F.expr(condition), F.lit(False))
    return {"filtered_data": df.filter(keep), "excluded_data": df.filter(~keep)}

# generated from the system
if "ld_display_outputs" not in globals():
    try:
        _ld_param = dbutils.widgets.getAll().get("ld_display_outputs")
        if _ld_param is not None and str(_ld_param).strip() != "":
            globals()["ld_display_outputs"] = str(_ld_param).strip().lower() not in ("false", "0", "no", "off")
        else:
            try:
                from dbruntime.databricks_repl_context import get_context
                globals()["ld_display_outputs"] = not get_context().isInJob
            except Exception:
                globals()["ld_display_outputs"] = True
    except Exception:
        globals()["ld_display_outputs"] = False
if "ld_display_outputs_for" not in globals():
    try:
        _ld_for = dbutils.widgets.getAll().get("ld_display_outputs_for")
        globals()["ld_display_outputs_for"] = frozenset(_p.strip() for _p in str(_ld_for).split(",") if _p.strip()) if _ld_for is not None else frozenset()
    except Exception:
        globals()["ld_display_outputs_for"] = frozenset()
ctx = globals().setdefault("ctx", {})
config = {
    "condition": "account_id IS NOT NULL AND priority IN ('Low', 'Medium', 'High', 'Critical')"
}
config["meta_state"] = {"is_preview": False, "is_focused_preview": False, "is_disabled": False}
inputs = {
    "data": ctx["prepare_688e7190.prepared_data"]
}
if config["meta_state"]["is_disabled"]:
    def _lb_limit_zero(value):
        if hasattr(value, "limit"):
            return value.limit(0)
        if isinstance(value, list):
            return [_lb_limit_zero(item) for item in value]
        return value
    inputs = {k: _lb_limit_zero(v) for k, v in inputs.items()}
out = run(config, inputs, spark)
if config["meta_state"]["is_disabled"]:
    out = {k: _lb_limit_zero(v) for k, v in out.items()}
ctx["filter_998c35a1.filtered_data"] = out["filtered_data"]
ctx["filter_998c35a1.excluded_data"] = out["excluded_data"]
if globals().get("ld_display_outputs", False) or "filter_998c35a1" in globals().get("ld_display_outputs_for", frozenset()):
    display(ctx["filter_998c35a1.filtered_data"])
    display(ctx["filter_998c35a1.excluded_data"])

In [0]:
"""
id: join_56dc73ae
template: join
templateVersion: 3.0.0
name: join_usage_accounts
position:
  x: 900
  y: 906.25
description:
  text: Combine two tables by matching on the account number column, keeping only matching records.
  hash: 997944a1
previewCodeHash: be3c990d539682f4
config:
  join_type: inner
  match_case: false
  multi_table: false
  conditions:
    - left_input: 0
      right_input: 1
      join_keys:
        - left: account_number
          right: account_number
      join_conditions: ""
  columns:
    - edits: []
      ordered: []
    - edits: []
      ordered: []
input:
  - node: filter_950cd717
    input_port: left
    output_port: filtered_data
  - node: filter_b72e1f84
    input_port: right
    output_port: filtered_data
"""

# generated from the system
from typing import Any, Dict, List, Optional
import pyspark.sql.functions as F
from pyspark.sql.types import StringType

def _is_checked(item: Dict[str, Any]) -> bool:
    return item.get("checked", True) is not False

def _qualified_col(alias: str, name: str):
    escaped = name.replace("`", "``")
    return F.col("`" + alias + "`.`" + escaped + "`")

def _ordered_names(df_cols: List[str], cfg: Dict[str, Any]) -> List[str]:
    ordered: List[str] = cfg.get("ordered") or []
    col_set = set(df_cols)
    placed = set()
    result: List[str] = []
    for name in ordered:
        if name in placed or name not in col_set:
            continue
        placed.add(name)
        result.append(name)
    for name in df_cols:
        if name in placed:
            continue
        placed.add(name)
        result.append(name)
    return result

def _edits_by_column(cfg: Dict[str, Any]) -> Dict[str, Dict[str, Any]]:
    by_col: Dict[str, Dict[str, Any]] = {}
    for item in cfg.get("edits", []) or []:
        col = item.get("column")
        if col is not None and col not in by_col:
            by_col[col] = item
    return by_col

def _projection(frames: List[Any]):
    out = []
    seen: Dict[str, int] = {}
    n = len(frames)
    for idx, (alias, df, cfg) in enumerate(frames):
        edits = _edits_by_column(cfg)
        for name in _ordered_names(list(df.columns), cfg):
            edit = edits.get(name)
            if edit is not None and not _is_checked(edit):
                continue
            col = _qualified_col(alias, name)
            user_alias = edit.get("alias") if edit else None
            if user_alias:
                out.append(col.alias(user_alias))
                seen[user_alias.lower()] = seen.get(user_alias.lower(), 0) + 1
                continue
            key = name.lower()
            if key not in seen:
                seen[key] = 1
                out.append(col.alias(name))
                continue
            new_name = ("right_" + name) if (n == 2 and idx == 1) else (name + "_" + str(idx + 1))
            while new_name.lower() in seen:
                new_name = new_name + "_dup"
            seen[new_name.lower()] = 1
            out.append(col.alias(new_name))
    return out

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    dfs: List[Any] = []
    for port in ("left", "right"):
        df = inputs.get(port)
        if df is not None:
            dfs.append(df)
    dfs.extend(inputs.get("data") or [])
    if len(dfs) < 2:
        raise ValueError(
            f"Join requires at least 2 inputs, got {len(dfs)}. "
            "Wire 2 or more upstream operators into this Join."
        )

    n = len(dfs)
    join_type = config.get("join_type") or "split_join"
    match_case = bool(config.get("match_case", False))
    conditions: List[Dict[str, Any]] = config.get("conditions") or []
    columns: List[Dict[str, Any]] = config.get("columns") or []

    aliased = [df.alias("t" + str(i)) for i, df in enumerate(dfs)]
    types = [{f.name.lower(): f.dataType for f in df.schema} for df in aliased]

    def key_col(idx: int, name: str):
        col = _qualified_col("t" + str(idx), name)
        if not match_case and isinstance(types[idx].get(name.lower()), StringType):
            return F.upper(F.trim(col))
        return col

    def edge_predicate(i: int, j: int, edge: Dict[str, Any]) -> Optional[Any]:
        predicates = []
        for key in edge.get("join_keys") or []:
            predicates.append(key_col(i, key["left"]) == key_col(j, key["right"]))
        condition = edge.get("join_conditions") or ""
        if condition:
            predicates.append(F.expr(condition))
        join_expr = None
        for predicate in predicates:
            join_expr = predicate if join_expr is None else join_expr & predicate
        return join_expr

    edges: List[Any] = [
        (int(edge.get("left_input", 0)), int(edge.get("right_input", 0)), edge)
        for edge in conditions
    ]

    predicates_by_step: Dict[int, List[Any]] = {}
    for i, j, edge in edges:
        predicate = edge_predicate(i, j, edge)
        if predicate is not None:
            predicates_by_step.setdefault(max(i, j), []).append(predicate)

    def predicate_at(step: int) -> Optional[Any]:
        join_expr = None
        for predicate in predicates_by_step.get(step, []):
            join_expr = predicate if join_expr is None else join_expr & predicate
        return join_expr

    def cfg(i: int) -> Dict[str, Any]:
        return columns[i] if i < len(columns) else {}

    is_two = n == 2
    is_split = is_two and join_type == "split_join"

    if is_split:
        join_expr = predicate_at(1)
        left_df, right_df = aliased[0], aliased[1]
        if join_expr is None:
            matched = left_df.join(right_df, how="inner")
            left_unmatched = left_df.join(right_df, how="left_anti")
            right_unmatched = right_df.join(left_df, how="left_anti")
        else:
            matched = left_df.join(right_df, join_expr, how="inner")
            left_unmatched = left_df.join(right_df, join_expr, how="left_anti")
            right_unmatched = right_df.join(left_df, join_expr, how="left_anti")
        projection = _projection([("t0", left_df, cfg(0)), ("t1", right_df, cfg(1))])
        if projection:
            matched = matched.select(*projection)
        return {
            "joined_data": matched,
            "left_unmatched": left_unmatched,
            "right_unmatched": right_unmatched,
        }

    how = join_type if is_two else "inner"
    if is_two:
        acc = aliased[0]
        predicate = predicate_at(1)
        if predicate is None:
            acc = acc.join(aliased[1], how=how)
        else:
            acc = acc.join(aliased[1], predicate, how=how)
    else:
        adjacency: Dict[int, List[int]] = {}
        for idx, (i, j, _edge) in enumerate(edges):
            adjacency.setdefault(i, []).append(idx)
            adjacency.setdefault(j, []).append(idx)
        visited = [False] * n
        visited[0] = True
        acc = aliased[0]
        used_edges = set()
        queue: List[int] = [0]
        while queue:
            cur = queue.pop(0)
            for idx in adjacency.get(cur, []):
                i, j, edge = edges[idx]
                nxt = j if i == cur else i
                if visited[nxt]:
                    continue
                visited[nxt] = True
                used_edges.add(idx)
                predicate = edge_predicate(i, j, edge)
                if predicate is None:
                    acc = acc.join(aliased[nxt], how=how)
                else:
                    acc = acc.join(aliased[nxt], predicate, how=how)
                queue.append(nxt)
        for idx, (i, j, edge) in enumerate(edges):
            if idx in used_edges:
                continue
            predicate = edge_predicate(i, j, edge)
            if predicate is not None:
                acc = acc.where(predicate)

    projection = _projection([("t" + str(i), aliased[i], cfg(i)) for i in range(n)])
    if projection:
        acc = acc.select(*projection)

    return {
        "joined_data": acc,
        "left_unmatched": spark.createDataFrame([], aliased[0].schema),
        "right_unmatched": spark.createDataFrame([], aliased[n - 1].schema),
    }

# generated from the system
if "ld_display_outputs" not in globals():
    try:
        _ld_param = dbutils.widgets.getAll().get("ld_display_outputs")
        if _ld_param is not None and str(_ld_param).strip() != "":
            globals()["ld_display_outputs"] = str(_ld_param).strip().lower() not in ("false", "0", "no", "off")
        else:
            try:
                from dbruntime.databricks_repl_context import get_context
                globals()["ld_display_outputs"] = not get_context().isInJob
            except Exception:
                globals()["ld_display_outputs"] = True
    except Exception:
        globals()["ld_display_outputs"] = False
if "ld_display_outputs_for" not in globals():
    try:
        _ld_for = dbutils.widgets.getAll().get("ld_display_outputs_for")
        globals()["ld_display_outputs_for"] = frozenset(_p.strip() for _p in str(_ld_for).split(",") if _p.strip()) if _ld_for is not None else frozenset()
    except Exception:
        globals()["ld_display_outputs_for"] = frozenset()
ctx = globals().setdefault("ctx", {})
config = {
    "join_type": "inner",
    "match_case": False,
    "multi_table": False,
    "conditions": [
        {
            "left_input": 0,
            "right_input": 1,
            "join_keys": [
                {
                    "left": "account_number",
                    "right": "account_number"
                }
            ],
            "join_conditions": ""
        }
    ],
    "columns": [
        {
            "edits": [],
            "ordered": []
        },
        {
            "edits": [],
            "ordered": []
        }
    ]
}
config["meta_state"] = {"is_preview": False, "is_focused_preview": False, "is_disabled": False}
inputs = {
    "left": ctx["filter_950cd717.filtered_data"],
    "right": ctx["filter_b72e1f84.filtered_data"]
}
if config["meta_state"]["is_disabled"]:
    def _lb_limit_zero(value):
        if hasattr(value, "limit"):
            return value.limit(0)
        if isinstance(value, list):
            return [_lb_limit_zero(item) for item in value]
        return value
    inputs = {k: _lb_limit_zero(v) for k, v in inputs.items()}
out = run(config, inputs, spark)
if config["meta_state"]["is_disabled"]:
    out = {k: _lb_limit_zero(v) for k, v in out.items()}
ctx["join_56dc73ae.joined_data"] = out["joined_data"]
ctx["join_56dc73ae.left_unmatched"] = out["left_unmatched"]
ctx["join_56dc73ae.right_unmatched"] = out["right_unmatched"]
if globals().get("ld_display_outputs", False) or "join_56dc73ae" in globals().get("ld_display_outputs_for", frozenset()):
    display(ctx["join_56dc73ae.joined_data"])
    display(ctx["join_56dc73ae.left_unmatched"])
    display(ctx["join_56dc73ae.right_unmatched"])

In [0]:
"""
id: aggregate_97c95dc7
template: aggregate
templateVersion: 2.0.0
name: agg_decision_makers
position:
  x: 2100
  y: 552.5
description:
  text: Group by account_id and count contacts, sum and max decision maker flags.
  hash: e3fa37dd
previewCodeHash: 3c752dfc50c460cd
config:
  group_bys:
    - expr: account_id
      type: expr
  aggregations:
    - columnExpr:
        expr: contact_id
        type: expr
      fn: COUNT
      alias: contact_count
    - columnExpr:
        expr: IF(is_decision_maker, 1, 0)
        type: expr
      fn: SUM
      alias: decision_maker_contact_count
    - columnExpr:
        expr: IF(is_decision_maker, 1, 0)
        type: expr
      fn: MAX
      alias: has_decision_maker
input:
  - node: filter_e52e6b11
    input_port: data
    output_port: filtered_data
"""

# generated from the system
import math
from typing import Dict, Any
import pyspark.sql.functions as F

DEFAULT_PERCENTILE = 0.5

DEFAULT_CONCAT_SEPARATOR = ", "

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs.get("data")
    group_bys = config.get("group_bys", [])
    aggregations = config.get("aggregations", [])

    group_by_set = set(e for gb in group_bys if (e := gb.get("expr", "")))

    agg_exprs = []
    for agg_def in aggregations:
        col_expr = agg_def.get("columnExpr", {})
        raw_expr = col_expr.get("expr", "")
        fn = agg_def.get("fn", "-")
        alias = agg_def.get("alias")

        if (fn == "-" or fn == "_") and not alias and raw_expr in group_by_set:
            continue

        fn_map = {
            "SUM": F.sum,
            "AVG": F.avg,
            "COUNT": F.count,
            "MIN": F.min,
            "MAX": F.max,
            "MEAN": F.mean,
            "MEDIAN": F.median,
            "STDDEV": F.stddev,
            "VARIANCE": F.variance,
            "FIRST": F.first,
            "LAST": F.last,
        }

        agg_fn = fn_map.get(fn)
        if agg_fn:
            arg = F.expr(raw_expr) if col_expr.get("type") == "expr" else raw_expr
            col = agg_fn(arg)
        elif fn == "-" or fn == "_":
            col = F.expr(raw_expr) if col_expr.get("type") == "expr" else F.col(raw_expr)
        elif fn == "PERCENTILE":
            raw_pct = agg_def.get("percentage")
            if (
                isinstance(raw_pct, (int, float))
                and not isinstance(raw_pct, bool)
                and math.isfinite(raw_pct)
            ):
                pct = max(0.0, min(1.0, float(raw_pct)))
            else:
                pct = DEFAULT_PERCENTILE
            col = F.expr(f"PERCENTILE({raw_expr}, {pct})")
        elif fn == "CONCAT":
            raw_sep = agg_def.get("separator")
            sep = raw_sep if isinstance(raw_sep, str) else DEFAULT_CONCAT_SEPARATOR
            concat_arg = F.expr(raw_expr) if col_expr.get("type") == "expr" else raw_expr
            col = F.concat_ws(sep, F.collect_list(concat_arg))
        elif fn == "COUNT_DISTINCT":
            arg = F.expr(raw_expr) if col_expr.get("type") != "column" else raw_expr
            col = F.count_distinct(arg)
        else:
            col = F.expr(f"{fn}({raw_expr})")

        if alias:
            col = col.alias(alias)

        agg_exprs.append(col)

    group_cols = [
        gb.get("expr", "") for gb in group_bys if gb.get("expr", "")
    ]

    if not agg_exprs:
        if group_cols:
            result = df.select(*group_cols).distinct()
            return {"aggregated_data": result}
        return {"aggregated_data": df}

    if group_cols:
        result = df.groupBy(*group_cols).agg(*agg_exprs)
    else:
        result = df.agg(*agg_exprs)

    return {"aggregated_data": result}

# generated from the system
if "ld_display_outputs" not in globals():
    try:
        _ld_param = dbutils.widgets.getAll().get("ld_display_outputs")
        if _ld_param is not None and str(_ld_param).strip() != "":
            globals()["ld_display_outputs"] = str(_ld_param).strip().lower() not in ("false", "0", "no", "off")
        else:
            try:
                from dbruntime.databricks_repl_context import get_context
                globals()["ld_display_outputs"] = not get_context().isInJob
            except Exception:
                globals()["ld_display_outputs"] = True
    except Exception:
        globals()["ld_display_outputs"] = False
if "ld_display_outputs_for" not in globals():
    try:
        _ld_for = dbutils.widgets.getAll().get("ld_display_outputs_for")
        globals()["ld_display_outputs_for"] = frozenset(_p.strip() for _p in str(_ld_for).split(",") if _p.strip()) if _ld_for is not None else frozenset()
    except Exception:
        globals()["ld_display_outputs_for"] = frozenset()
ctx = globals().setdefault("ctx", {})
config = {
    "group_bys": [
        {
            "expr": "account_id",
            "type": "expr"
        }
    ],
    "aggregations": [
        {
            "columnExpr": {
                "expr": "contact_id",
                "type": "expr"
            },
            "fn": "COUNT",
            "alias": "contact_count",
            "withAsKeyword": None
        },
        {
            "columnExpr": {
                "expr": "IF(is_decision_maker, 1, 0)",
                "type": "expr"
            },
            "fn": "SUM",
            "alias": "decision_maker_contact_count",
            "withAsKeyword": None
        },
        {
            "columnExpr": {
                "expr": "IF(is_decision_maker, 1, 0)",
                "type": "expr"
            },
            "fn": "MAX",
            "alias": "has_decision_maker",
            "withAsKeyword": None
        }
    ]
}
config["meta_state"] = {"is_preview": False, "is_focused_preview": False, "is_disabled": False}
inputs = {
    "data": ctx["filter_e52e6b11.filtered_data"]
}
if config["meta_state"]["is_disabled"]:
    def _lb_limit_zero(value):
        if hasattr(value, "limit"):
            return value.limit(0)
        if isinstance(value, list):
            return [_lb_limit_zero(item) for item in value]
        return value
    inputs = {k: _lb_limit_zero(v) for k, v in inputs.items()}
out = run(config, inputs, spark)
if config["meta_state"]["is_disabled"]:
    out = {k: _lb_limit_zero(v) for k, v in out.items()}
ctx["aggregate_97c95dc7.aggregated_data"] = out["aggregated_data"]
if globals().get("ld_display_outputs", False) or "aggregate_97c95dc7" in globals().get("ld_display_outputs_for", frozenset()):
    display(ctx["aggregate_97c95dc7.aggregated_data"])

In [0]:
"""
id: aggregate_87c4fde9
template: aggregate
templateVersion: 2.0.0
name: agg_pipeline_health
position:
  x: 1500
  y: 441.25
description:
  text: Group data by account and calculate sums and max values for renewal and expansion opportunity metrics.
  hash: d6a20dc1
previewCodeHash: 626b778926cdfae4
config:
  group_bys:
    - expr: account_id
      type: expr
  aggregations:
    - columnExpr:
        expr: IF(is_renewal_opportunity, amount, 0)
        type: expr
      fn: SUM
      alias: renewal_pipeline_amount
    - columnExpr:
        expr: IF(is_renewal_opportunity AND stage_name NOT IN ('Closed Won', 'Closed Lost') AND days_in_stage > 30, 1, 0)
        type: expr
      fn: SUM
      alias: stalled_renewal_opportunity_count
    - columnExpr:
        expr: IF(is_expansion_opportunity AND stage_name NOT IN ('Closed Won', 'Closed Lost'), 1, 0)
        type: expr
      fn: SUM
      alias: active_expansion_opportunity_count
    - columnExpr:
        expr: IF(is_renewal_opportunity, close_date, NULL)
        type: expr
      fn: MAX
      alias: latest_renewal_close_date
input:
  - node: filter_81ec30bb
    input_port: data
    output_port: filtered_data
"""

# generated from the system
import math
from typing import Dict, Any
import pyspark.sql.functions as F

DEFAULT_PERCENTILE = 0.5

DEFAULT_CONCAT_SEPARATOR = ", "

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs.get("data")
    group_bys = config.get("group_bys", [])
    aggregations = config.get("aggregations", [])

    group_by_set = set(e for gb in group_bys if (e := gb.get("expr", "")))

    agg_exprs = []
    for agg_def in aggregations:
        col_expr = agg_def.get("columnExpr", {})
        raw_expr = col_expr.get("expr", "")
        fn = agg_def.get("fn", "-")
        alias = agg_def.get("alias")

        if (fn == "-" or fn == "_") and not alias and raw_expr in group_by_set:
            continue

        fn_map = {
            "SUM": F.sum,
            "AVG": F.avg,
            "COUNT": F.count,
            "MIN": F.min,
            "MAX": F.max,
            "MEAN": F.mean,
            "MEDIAN": F.median,
            "STDDEV": F.stddev,
            "VARIANCE": F.variance,
            "FIRST": F.first,
            "LAST": F.last,
        }

        agg_fn = fn_map.get(fn)
        if agg_fn:
            arg = F.expr(raw_expr) if col_expr.get("type") == "expr" else raw_expr
            col = agg_fn(arg)
        elif fn == "-" or fn == "_":
            col = F.expr(raw_expr) if col_expr.get("type") == "expr" else F.col(raw_expr)
        elif fn == "PERCENTILE":
            raw_pct = agg_def.get("percentage")
            if (
                isinstance(raw_pct, (int, float))
                and not isinstance(raw_pct, bool)
                and math.isfinite(raw_pct)
            ):
                pct = max(0.0, min(1.0, float(raw_pct)))
            else:
                pct = DEFAULT_PERCENTILE
            col = F.expr(f"PERCENTILE({raw_expr}, {pct})")
        elif fn == "CONCAT":
            raw_sep = agg_def.get("separator")
            sep = raw_sep if isinstance(raw_sep, str) else DEFAULT_CONCAT_SEPARATOR
            concat_arg = F.expr(raw_expr) if col_expr.get("type") == "expr" else raw_expr
            col = F.concat_ws(sep, F.collect_list(concat_arg))
        elif fn == "COUNT_DISTINCT":
            arg = F.expr(raw_expr) if col_expr.get("type") != "column" else raw_expr
            col = F.count_distinct(arg)
        else:
            col = F.expr(f"{fn}({raw_expr})")

        if alias:
            col = col.alias(alias)

        agg_exprs.append(col)

    group_cols = [
        gb.get("expr", "") for gb in group_bys if gb.get("expr", "")
    ]

    if not agg_exprs:
        if group_cols:
            result = df.select(*group_cols).distinct()
            return {"aggregated_data": result}
        return {"aggregated_data": df}

    if group_cols:
        result = df.groupBy(*group_cols).agg(*agg_exprs)
    else:
        result = df.agg(*agg_exprs)

    return {"aggregated_data": result}

# generated from the system
if "ld_display_outputs" not in globals():
    try:
        _ld_param = dbutils.widgets.getAll().get("ld_display_outputs")
        if _ld_param is not None and str(_ld_param).strip() != "":
            globals()["ld_display_outputs"] = str(_ld_param).strip().lower() not in ("false", "0", "no", "off")
        else:
            try:
                from dbruntime.databricks_repl_context import get_context
                globals()["ld_display_outputs"] = not get_context().isInJob
            except Exception:
                globals()["ld_display_outputs"] = True
    except Exception:
        globals()["ld_display_outputs"] = False
if "ld_display_outputs_for" not in globals():
    try:
        _ld_for = dbutils.widgets.getAll().get("ld_display_outputs_for")
        globals()["ld_display_outputs_for"] = frozenset(_p.strip() for _p in str(_ld_for).split(",") if _p.strip()) if _ld_for is not None else frozenset()
    except Exception:
        globals()["ld_display_outputs_for"] = frozenset()
ctx = globals().setdefault("ctx", {})
config = {
    "group_bys": [
        {
            "expr": "account_id",
            "type": "expr"
        }
    ],
    "aggregations": [
        {
            "columnExpr": {
                "expr": "IF(is_renewal_opportunity, amount, 0)",
                "type": "expr"
            },
            "fn": "SUM",
            "alias": "renewal_pipeline_amount",
            "withAsKeyword": None
        },
        {
            "columnExpr": {
                "expr": "IF(is_renewal_opportunity AND stage_name NOT IN ('Closed Won', 'Closed Lost') AND days_in_stage > 30, 1, 0)",
                "type": "expr"
            },
            "fn": "SUM",
            "alias": "stalled_renewal_opportunity_count",
            "withAsKeyword": None
        },
        {
            "columnExpr": {
                "expr": "IF(is_expansion_opportunity AND stage_name NOT IN ('Closed Won', 'Closed Lost'), 1, 0)",
                "type": "expr"
            },
            "fn": "SUM",
            "alias": "active_expansion_opportunity_count",
            "withAsKeyword": None
        },
        {
            "columnExpr": {
                "expr": "IF(is_renewal_opportunity, close_date, NULL)",
                "type": "expr"
            },
            "fn": "MAX",
            "alias": "latest_renewal_close_date",
            "withAsKeyword": None
        }
    ]
}
config["meta_state"] = {"is_preview": False, "is_focused_preview": False, "is_disabled": False}
inputs = {
    "data": ctx["filter_81ec30bb.filtered_data"]
}
if config["meta_state"]["is_disabled"]:
    def _lb_limit_zero(value):
        if hasattr(value, "limit"):
            return value.limit(0)
        if isinstance(value, list):
            return [_lb_limit_zero(item) for item in value]
        return value
    inputs = {k: _lb_limit_zero(v) for k, v in inputs.items()}
out = run(config, inputs, spark)
if config["meta_state"]["is_disabled"]:
    out = {k: _lb_limit_zero(v) for k, v in out.items()}
ctx["aggregate_87c4fde9.aggregated_data"] = out["aggregated_data"]
if globals().get("ld_display_outputs", False) or "aggregate_87c4fde9" in globals().get("ld_display_outputs_for", frozenset()):
    display(ctx["aggregate_87c4fde9.aggregated_data"])

In [0]:
"""
id: aggregate_2e5617bb
template: aggregate
templateVersion: 2.0.0
name: agg_support_exposure
position:
  x: 1200
  y: 310
description:
  text: Group data by account ID and calculate counts of open cases, high priority open cases, SLA breach open cases, and the maximum age of open cases.
  hash: 6d0da60e
previewCodeHash: 5c6381bf459ed8d8
config:
  group_bys:
    - expr: account_id
      type: expr
  aggregations:
    - columnExpr:
        expr: IF(is_open, 1, 0)
        type: expr
      fn: SUM
      alias: open_case_count
    - columnExpr:
        expr: IF(is_open AND priority IN ('High', 'Critical'), 1, 0)
        type: expr
      fn: SUM
      alias: open_high_priority_case_count
    - columnExpr:
        expr: IF(is_open AND sla_breached = true, 1, 0)
        type: expr
      fn: SUM
      alias: open_sla_breach_count
    - columnExpr:
        expr: IF(is_open, case_age_days, NULL)
        type: expr
      fn: MAX
      alias: max_open_case_age_days
input:
  - node: filter_998c35a1
    input_port: data
    output_port: filtered_data
"""

# generated from the system
import math
from typing import Dict, Any
import pyspark.sql.functions as F

DEFAULT_PERCENTILE = 0.5

DEFAULT_CONCAT_SEPARATOR = ", "

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs.get("data")
    group_bys = config.get("group_bys", [])
    aggregations = config.get("aggregations", [])

    group_by_set = set(e for gb in group_bys if (e := gb.get("expr", "")))

    agg_exprs = []
    for agg_def in aggregations:
        col_expr = agg_def.get("columnExpr", {})
        raw_expr = col_expr.get("expr", "")
        fn = agg_def.get("fn", "-")
        alias = agg_def.get("alias")

        if (fn == "-" or fn == "_") and not alias and raw_expr in group_by_set:
            continue

        fn_map = {
            "SUM": F.sum,
            "AVG": F.avg,
            "COUNT": F.count,
            "MIN": F.min,
            "MAX": F.max,
            "MEAN": F.mean,
            "MEDIAN": F.median,
            "STDDEV": F.stddev,
            "VARIANCE": F.variance,
            "FIRST": F.first,
            "LAST": F.last,
        }

        agg_fn = fn_map.get(fn)
        if agg_fn:
            arg = F.expr(raw_expr) if col_expr.get("type") == "expr" else raw_expr
            col = agg_fn(arg)
        elif fn == "-" or fn == "_":
            col = F.expr(raw_expr) if col_expr.get("type") == "expr" else F.col(raw_expr)
        elif fn == "PERCENTILE":
            raw_pct = agg_def.get("percentage")
            if (
                isinstance(raw_pct, (int, float))
                and not isinstance(raw_pct, bool)
                and math.isfinite(raw_pct)
            ):
                pct = max(0.0, min(1.0, float(raw_pct)))
            else:
                pct = DEFAULT_PERCENTILE
            col = F.expr(f"PERCENTILE({raw_expr}, {pct})")
        elif fn == "CONCAT":
            raw_sep = agg_def.get("separator")
            sep = raw_sep if isinstance(raw_sep, str) else DEFAULT_CONCAT_SEPARATOR
            concat_arg = F.expr(raw_expr) if col_expr.get("type") == "expr" else raw_expr
            col = F.concat_ws(sep, F.collect_list(concat_arg))
        elif fn == "COUNT_DISTINCT":
            arg = F.expr(raw_expr) if col_expr.get("type") != "column" else raw_expr
            col = F.count_distinct(arg)
        else:
            col = F.expr(f"{fn}({raw_expr})")

        if alias:
            col = col.alias(alias)

        agg_exprs.append(col)

    group_cols = [
        gb.get("expr", "") for gb in group_bys if gb.get("expr", "")
    ]

    if not agg_exprs:
        if group_cols:
            result = df.select(*group_cols).distinct()
            return {"aggregated_data": result}
        return {"aggregated_data": df}

    if group_cols:
        result = df.groupBy(*group_cols).agg(*agg_exprs)
    else:
        result = df.agg(*agg_exprs)

    return {"aggregated_data": result}

# generated from the system
if "ld_display_outputs" not in globals():
    try:
        _ld_param = dbutils.widgets.getAll().get("ld_display_outputs")
        if _ld_param is not None and str(_ld_param).strip() != "":
            globals()["ld_display_outputs"] = str(_ld_param).strip().lower() not in ("false", "0", "no", "off")
        else:
            try:
                from dbruntime.databricks_repl_context import get_context
                globals()["ld_display_outputs"] = not get_context().isInJob
            except Exception:
                globals()["ld_display_outputs"] = True
    except Exception:
        globals()["ld_display_outputs"] = False
if "ld_display_outputs_for" not in globals():
    try:
        _ld_for = dbutils.widgets.getAll().get("ld_display_outputs_for")
        globals()["ld_display_outputs_for"] = frozenset(_p.strip() for _p in str(_ld_for).split(",") if _p.strip()) if _ld_for is not None else frozenset()
    except Exception:
        globals()["ld_display_outputs_for"] = frozenset()
ctx = globals().setdefault("ctx", {})
config = {
    "group_bys": [
        {
            "expr": "account_id",
            "type": "expr"
        }
    ],
    "aggregations": [
        {
            "columnExpr": {
                "expr": "IF(is_open, 1, 0)",
                "type": "expr"
            },
            "fn": "SUM",
            "alias": "open_case_count",
            "withAsKeyword": None
        },
        {
            "columnExpr": {
                "expr": "IF(is_open AND priority IN ('High', 'Critical'), 1, 0)",
                "type": "expr"
            },
            "fn": "SUM",
            "alias": "open_high_priority_case_count",
            "withAsKeyword": None
        },
        {
            "columnExpr": {
                "expr": "IF(is_open AND sla_breached = true, 1, 0)",
                "type": "expr"
            },
            "fn": "SUM",
            "alias": "open_sla_breach_count",
            "withAsKeyword": None
        },
        {
            "columnExpr": {
                "expr": "IF(is_open, case_age_days, NULL)",
                "type": "expr"
            },
            "fn": "MAX",
            "alias": "max_open_case_age_days",
            "withAsKeyword": None
        }
    ]
}
config["meta_state"] = {"is_preview": False, "is_focused_preview": False, "is_disabled": False}
inputs = {
    "data": ctx["filter_998c35a1.filtered_data"]
}
if config["meta_state"]["is_disabled"]:
    def _lb_limit_zero(value):
        if hasattr(value, "limit"):
            return value.limit(0)
        if isinstance(value, list):
            return [_lb_limit_zero(item) for item in value]
        return value
    inputs = {k: _lb_limit_zero(v) for k, v in inputs.items()}
out = run(config, inputs, spark)
if config["meta_state"]["is_disabled"]:
    out = {k: _lb_limit_zero(v) for k, v in out.items()}
ctx["aggregate_2e5617bb.aggregated_data"] = out["aggregated_data"]
if globals().get("ld_display_outputs", False) or "aggregate_2e5617bb" in globals().get("ld_display_outputs_for", frozenset()):
    display(ctx["aggregate_2e5617bb.aggregated_data"])

In [0]:
"""
id: prepare_b02cb6b8
template: prepare
templateVersion: 1.0.0
name: prep_usage_trend_windows
position:
  x: 1200
  y: 906.25
description:
  text: Add columns to flag latest and previous 14-day windows based on the maximum usage date.
  hash: 2b7dbfd8
previewCodeHash: 5dae60f94c1cc1f9
config:
  actions:
    - type: formula
      target: max_usage_date
      expression: MAX(usage_date) OVER ()
    - type: formula
      target: latest_window_flag
      expression: usage_date >= DATE_SUB(max_usage_date, 13) AND usage_date <= max_usage_date
    - type: formula
      target: previous_window_flag
      expression: usage_date >= DATE_SUB(max_usage_date, 27) AND usage_date < DATE_SUB(max_usage_date, 13)
input:
  - node: join_56dc73ae
    input_port: data
    output_port: joined_data
"""

# generated from the system
from typing import Any, Callable, Dict, List
import pyspark.sql.functions as F

TEXT_CASE_FUNCTIONS: Dict[str, Callable] = {
    "lower": F.lower,
    "upper": F.upper,
    "title": F.initcap,
}

TRIM_FUNCTIONS: Dict[str, Callable] = {
    "both": F.trim,
    "left": F.ltrim,
    "right": F.rtrim,
}

VALID_MATCH_MODES = {"exact", "contains", "prefix", "suffix", "regex"}

def _require_column(df, column: str, action_type: str) -> None:
    if not column:
        raise ValueError(f"{action_type}: 'column' is required")
    if column not in df.columns:
        raise ValueError(
            f"{action_type}: column '{column}' not found in input data. "
            f"Available columns: {df.columns}"
        )

def _column_dtype(df, column: str) -> str:
    return df.schema[column].dataType.simpleString()

def _quoted_ident(name: str) -> str:
    return "`" + name.replace("`", "``") + "`"

def _col(name: str):
    """Reference a column by exact name.

    F.col parses its argument as a column expression, so a name containing
    a '.' (e.g. "a.b") is otherwise read as struct-field access and fails
    even though the column exists. Backtick-quoting forces an exact-name
    lookup; quoting plain names is harmless.
    """
    return F.col(_quoted_ident(name))

def _coerced_lit(value: Any, target_dtype: str):
    """Wrap a user-provided value as a literal cast to the target column's type.

    UI always feeds text per the operator spec; this is where the
    type coercion happens so the per-action UI stays simple.
    """
    return F.lit(value).cast(target_dtype)

def _apply_formula(df, action: Dict[str, Any]):
    target = action.get("target", "")
    expression = action.get("expression", "")
    if not target:
        raise ValueError("formula: 'target' is required")
    if not expression:
        raise ValueError("formula: 'expression' is required")
    return df.withColumn(target, F.expr(expression))

def _apply_cast(df, action: Dict[str, Any]):
    column = action.get("column", "")
    to_type = action.get("to", "")
    on_error = action.get("on_error", "null")
    _require_column(df, column, "cast")
    if not to_type:
        raise ValueError("cast: 'to' (target type) is required")
    if on_error == "null":
        return df.withColumn(
            column,
            F.expr(f"try_cast({_quoted_ident(column)} as {to_type})"),
        )
    return df.withColumn(column, _col(column).cast(to_type))

def _apply_replace_value(df, action: Dict[str, Any]):
    column = action.get("column", "")
    match_mode = action.get("match_mode", "exact")
    match = action.get("match", "")
    with_val = action.get("with", "")
    case_sensitive = bool(action.get("case_sensitive", False))
    _require_column(df, column, "replace_value")
    if match_mode not in VALID_MATCH_MODES:
        raise ValueError(
            f"replace_value: unsupported match_mode '{match_mode}'. "
            f"Choose one of: {sorted(VALID_MATCH_MODES)}"
        )
    dtype = _column_dtype(df, column)
    col_as_string = _col(column).cast("string")
    match_lit = F.lit(match)
    if case_sensitive:
        cmp_col = col_as_string
        cmp_lit = match_lit
    else:
        cmp_col = F.lower(col_as_string)
        cmp_lit = F.lower(match_lit)

    if match_mode == "exact":
        predicate = cmp_col == cmp_lit
    elif match_mode == "contains":
        predicate = cmp_col.contains(cmp_lit)
    elif match_mode == "prefix":
        predicate = cmp_col.startswith(cmp_lit)
    elif match_mode == "suffix":
        predicate = cmp_col.endswith(cmp_lit)
    else:  # regex
        pattern = match if case_sensitive else f"(?i){match}"
        predicate = col_as_string.rlike(pattern)

    replacement = _coerced_lit(with_val, dtype)
    return df.withColumn(
        column,
        F.when(predicate, replacement).otherwise(_col(column)),
    )

def _apply_fill_null(df, action: Dict[str, Any]):
    column = action.get("column", "")
    with_val = action.get("with", "")
    _require_column(df, column, "fill_null")
    dtype = _column_dtype(df, column)
    replacement = _coerced_lit(with_val, dtype)
    return df.withColumn(
        column,
        F.when(_col(column).isNull(), replacement).otherwise(_col(column)),
    )

def _apply_text_case(df, action: Dict[str, Any]):
    column = action.get("column", "")
    case = action.get("case", "")
    _require_column(df, column, "text_case")
    fn = TEXT_CASE_FUNCTIONS.get(case)
    if fn is None:
        raise ValueError(
            f"text_case: unsupported case '{case}'. "
            f"Choose one of: {sorted(TEXT_CASE_FUNCTIONS.keys())}"
        )
    return df.withColumn(column, fn(_col(column)))

def _apply_trim(df, action: Dict[str, Any]):
    column = action.get("column", "")
    side = action.get("side", "both")
    _require_column(df, column, "trim")
    fn = TRIM_FUNCTIONS.get(side)
    if fn is None:
        raise ValueError(
            f"trim: unsupported side '{side}'. "
            f"Choose one of: {sorted(TRIM_FUNCTIONS.keys())}"
        )
    return df.withColumn(column, fn(_col(column)))

def _apply_regex_replace(df, action: Dict[str, Any]):
    column = action.get("column", "")
    pattern = action.get("pattern", "")
    replacement = action.get("replacement", "")
    _require_column(df, column, "regex_replace")
    return df.withColumn(
        column,
        F.regexp_replace(_col(column), pattern, replacement),
    )

def _apply_extract(df, action: Dict[str, Any]):
    column = action.get("column", "")
    pattern = action.get("pattern", "")
    target = action.get("target") or column
    group_raw = action.get("group", 0)
    _require_column(df, column, "extract")
    try:
        group_idx = int(group_raw)
    except (TypeError, ValueError) as exc:
        raise ValueError(
            f"extract: 'group' must be an integer, got {group_raw!r}"
        ) from exc
    return df.withColumn(
        target,
        F.regexp_extract(_col(column), pattern, group_idx),
    )

def _apply_parse_date(df, action: Dict[str, Any]):
    column = action.get("column", "")
    kind = action.get("kind", "date")
    fmt = action.get("format") or None
    on_error = action.get("on_error", "null")
    _require_column(df, column, "parse_date")
    if kind not in ("date", "timestamp"):
        raise ValueError(
            f"parse_date: unsupported kind '{kind}'. Choose date or timestamp."
        )
    # PySpark exposes F.try_to_timestamp from Spark 3.5 but only adds
    # F.try_to_date in Spark 4.0. Current Databricks Runtimes still ship
    # Spark 3.5, where referencing F.try_to_date raises AttributeError
    # before any data is touched. Route the null-on-error path through
    # primitives that exist in both 3.5 and 4.0:
    #   - F.try_to_timestamp  (PySpark 3.5+, returns NULL under ANSI too)
    #   - SQL try_cast        (Spark 3.5+, returns NULL under ANSI too)
    #   - F.to_date           (PySpark 3.5+; returns NULL on parse failure
    #                          under the default non-ANSI mode, which is
    #                          the Databricks default. Under ANSI mode it
    #                          raises — accepted limitation until
    #                          try_to_date lands on every supported DBR.)
    if on_error == "null":
        if kind == "timestamp":
            if fmt:
                return df.withColumn(
                    column, F.try_to_timestamp(_col(column), F.lit(fmt))
                )
            return df.withColumn(column, F.try_to_timestamp(_col(column)))
        # kind == "date"
        if fmt:
            return df.withColumn(column, F.to_date(_col(column), fmt))
        return df.withColumn(
            column, F.expr(f"try_cast({_quoted_ident(column)} as date)")
        )
    # on_error == "error": surface parse failures as Spark exceptions.
    fn = F.to_date if kind == "date" else F.to_timestamp
    if fmt:
        return df.withColumn(column, fn(_col(column), fmt))
    return df.withColumn(column, fn(_col(column)))

ACTION_DISPATCH: Dict[str, Callable] = {
    "formula": _apply_formula,
    "cast": _apply_cast,
    "replace_value": _apply_replace_value,
    "fill_null": _apply_fill_null,
    "text_case": _apply_text_case,
    "trim": _apply_trim,
    "regex_replace": _apply_regex_replace,
    "extract": _apply_extract,
    "parse_date": _apply_parse_date,
}

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs.get("data")
    actions: List[Dict[str, Any]] = config.get("actions", []) or []

    if not actions:
        return {"prepared_data": df}

    for index, action in enumerate(actions):
        if not isinstance(action, dict):
            raise ValueError(
                f"actions[{index}]: expected an object, got {type(action).__name__}"
            )
        if action.get("enabled", True) is False:
            continue
        action_type = action.get("type", "")
        fn = ACTION_DISPATCH.get(action_type)
        if fn is None:
            raise ValueError(
                f"actions[{index}]: unsupported action type {action_type!r}. "
                f"Choose one of: {sorted(ACTION_DISPATCH.keys())}"
            )
        df = fn(df, action)
    return {"prepared_data": df}

# generated from the system
if "ld_display_outputs" not in globals():
    try:
        _ld_param = dbutils.widgets.getAll().get("ld_display_outputs")
        if _ld_param is not None and str(_ld_param).strip() != "":
            globals()["ld_display_outputs"] = str(_ld_param).strip().lower() not in ("false", "0", "no", "off")
        else:
            try:
                from dbruntime.databricks_repl_context import get_context
                globals()["ld_display_outputs"] = not get_context().isInJob
            except Exception:
                globals()["ld_display_outputs"] = True
    except Exception:
        globals()["ld_display_outputs"] = False
if "ld_display_outputs_for" not in globals():
    try:
        _ld_for = dbutils.widgets.getAll().get("ld_display_outputs_for")
        globals()["ld_display_outputs_for"] = frozenset(_p.strip() for _p in str(_ld_for).split(",") if _p.strip()) if _ld_for is not None else frozenset()
    except Exception:
        globals()["ld_display_outputs_for"] = frozenset()
ctx = globals().setdefault("ctx", {})
config = {
    "actions": [
        {
            "type": "formula",
            "target": "max_usage_date",
            "expression": "MAX(usage_date) OVER ()"
        },
        {
            "type": "formula",
            "target": "latest_window_flag",
            "expression": "usage_date >= DATE_SUB(max_usage_date, 13) AND usage_date <= max_usage_date"
        },
        {
            "type": "formula",
            "target": "previous_window_flag",
            "expression": "usage_date >= DATE_SUB(max_usage_date, 27) AND usage_date < DATE_SUB(max_usage_date, 13)"
        }
    ]
}
config["meta_state"] = {"is_preview": False, "is_focused_preview": False, "is_disabled": False}
inputs = {
    "data": ctx["join_56dc73ae.joined_data"]
}
if config["meta_state"]["is_disabled"]:
    def _lb_limit_zero(value):
        if hasattr(value, "limit"):
            return value.limit(0)
        if isinstance(value, list):
            return [_lb_limit_zero(item) for item in value]
        return value
    inputs = {k: _lb_limit_zero(v) for k, v in inputs.items()}
out = run(config, inputs, spark)
if config["meta_state"]["is_disabled"]:
    out = {k: _lb_limit_zero(v) for k, v in out.items()}
ctx["prepare_b02cb6b8.prepared_data"] = out["prepared_data"]
if globals().get("ld_display_outputs", False) or "prepare_b02cb6b8" in globals().get("ld_display_outputs_for", frozenset()):
    display(ctx["prepare_b02cb6b8.prepared_data"])

In [0]:
"""
id: join_support_exposure_65e02cbb
template: join
templateVersion: 3.0.0
name: join_support_exposure
position:
  x: 1500
  y: 155
description:
  text: Join two inputs by matching account_id, keeping all rows from the left input and corresponding matches from the right.
  hash: b666a9ea
config:
  join_type: left
  match_case: false
  multi_table: false
  conditions:
    - left_input: 0
      right_input: 1
      join_keys:
        - left: account_id
          right: account_id
      join_conditions: ""
  columns:
    - edits: []
      ordered: []
    - edits: []
      ordered: []
input:
  - node: filter_b72e1f84
    input_port: left
    output_port: filtered_data
  - node: aggregate_2e5617bb
    input_port: right
    output_port: aggregated_data
"""

# generated from the system
from typing import Any, Dict, List, Optional
import pyspark.sql.functions as F
from pyspark.sql.types import StringType

def _is_checked(item: Dict[str, Any]) -> bool:
    return item.get("checked", True) is not False

def _qualified_col(alias: str, name: str):
    escaped = name.replace("`", "``")
    return F.col("`" + alias + "`.`" + escaped + "`")

def _ordered_names(df_cols: List[str], cfg: Dict[str, Any]) -> List[str]:
    ordered: List[str] = cfg.get("ordered") or []
    col_set = set(df_cols)
    placed = set()
    result: List[str] = []
    for name in ordered:
        if name in placed or name not in col_set:
            continue
        placed.add(name)
        result.append(name)
    for name in df_cols:
        if name in placed:
            continue
        placed.add(name)
        result.append(name)
    return result

def _edits_by_column(cfg: Dict[str, Any]) -> Dict[str, Dict[str, Any]]:
    by_col: Dict[str, Dict[str, Any]] = {}
    for item in cfg.get("edits", []) or []:
        col = item.get("column")
        if col is not None and col not in by_col:
            by_col[col] = item
    return by_col

def _projection(frames: List[Any]):
    out = []
    seen: Dict[str, int] = {}
    n = len(frames)
    for idx, (alias, df, cfg) in enumerate(frames):
        edits = _edits_by_column(cfg)
        for name in _ordered_names(list(df.columns), cfg):
            edit = edits.get(name)
            if edit is not None and not _is_checked(edit):
                continue
            col = _qualified_col(alias, name)
            user_alias = edit.get("alias") if edit else None
            if user_alias:
                out.append(col.alias(user_alias))
                seen[user_alias.lower()] = seen.get(user_alias.lower(), 0) + 1
                continue
            key = name.lower()
            if key not in seen:
                seen[key] = 1
                out.append(col.alias(name))
                continue
            new_name = ("right_" + name) if (n == 2 and idx == 1) else (name + "_" + str(idx + 1))
            while new_name.lower() in seen:
                new_name = new_name + "_dup"
            seen[new_name.lower()] = 1
            out.append(col.alias(new_name))
    return out

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    dfs: List[Any] = []
    for port in ("left", "right"):
        df = inputs.get(port)
        if df is not None:
            dfs.append(df)
    dfs.extend(inputs.get("data") or [])
    if len(dfs) < 2:
        raise ValueError(
            f"Join requires at least 2 inputs, got {len(dfs)}. "
            "Wire 2 or more upstream operators into this Join."
        )

    n = len(dfs)
    join_type = config.get("join_type") or "split_join"
    match_case = bool(config.get("match_case", False))
    conditions: List[Dict[str, Any]] = config.get("conditions") or []
    columns: List[Dict[str, Any]] = config.get("columns") or []

    aliased = [df.alias("t" + str(i)) for i, df in enumerate(dfs)]
    types = [{f.name.lower(): f.dataType for f in df.schema} for df in aliased]

    def key_col(idx: int, name: str):
        col = _qualified_col("t" + str(idx), name)
        if not match_case and isinstance(types[idx].get(name.lower()), StringType):
            return F.upper(F.trim(col))
        return col

    def edge_predicate(i: int, j: int, edge: Dict[str, Any]) -> Optional[Any]:
        predicates = []
        for key in edge.get("join_keys") or []:
            predicates.append(key_col(i, key["left"]) == key_col(j, key["right"]))
        condition = edge.get("join_conditions") or ""
        if condition:
            predicates.append(F.expr(condition))
        join_expr = None
        for predicate in predicates:
            join_expr = predicate if join_expr is None else join_expr & predicate
        return join_expr

    edges: List[Any] = [
        (int(edge.get("left_input", 0)), int(edge.get("right_input", 0)), edge)
        for edge in conditions
    ]

    predicates_by_step: Dict[int, List[Any]] = {}
    for i, j, edge in edges:
        predicate = edge_predicate(i, j, edge)
        if predicate is not None:
            predicates_by_step.setdefault(max(i, j), []).append(predicate)

    def predicate_at(step: int) -> Optional[Any]:
        join_expr = None
        for predicate in predicates_by_step.get(step, []):
            join_expr = predicate if join_expr is None else join_expr & predicate
        return join_expr

    def cfg(i: int) -> Dict[str, Any]:
        return columns[i] if i < len(columns) else {}

    is_two = n == 2
    is_split = is_two and join_type == "split_join"

    if is_split:
        join_expr = predicate_at(1)
        left_df, right_df = aliased[0], aliased[1]
        if join_expr is None:
            matched = left_df.join(right_df, how="inner")
            left_unmatched = left_df.join(right_df, how="left_anti")
            right_unmatched = right_df.join(left_df, how="left_anti")
        else:
            matched = left_df.join(right_df, join_expr, how="inner")
            left_unmatched = left_df.join(right_df, join_expr, how="left_anti")
            right_unmatched = right_df.join(left_df, join_expr, how="left_anti")
        projection = _projection([("t0", left_df, cfg(0)), ("t1", right_df, cfg(1))])
        if projection:
            matched = matched.select(*projection)
        return {
            "joined_data": matched,
            "left_unmatched": left_unmatched,
            "right_unmatched": right_unmatched,
        }

    how = join_type if is_two else "inner"
    if is_two:
        acc = aliased[0]
        predicate = predicate_at(1)
        if predicate is None:
            acc = acc.join(aliased[1], how=how)
        else:
            acc = acc.join(aliased[1], predicate, how=how)
    else:
        adjacency: Dict[int, List[int]] = {}
        for idx, (i, j, _edge) in enumerate(edges):
            adjacency.setdefault(i, []).append(idx)
            adjacency.setdefault(j, []).append(idx)
        visited = [False] * n
        visited[0] = True
        acc = aliased[0]
        used_edges = set()
        queue: List[int] = [0]
        while queue:
            cur = queue.pop(0)
            for idx in adjacency.get(cur, []):
                i, j, edge = edges[idx]
                nxt = j if i == cur else i
                if visited[nxt]:
                    continue
                visited[nxt] = True
                used_edges.add(idx)
                predicate = edge_predicate(i, j, edge)
                if predicate is None:
                    acc = acc.join(aliased[nxt], how=how)
                else:
                    acc = acc.join(aliased[nxt], predicate, how=how)
                queue.append(nxt)
        for idx, (i, j, edge) in enumerate(edges):
            if idx in used_edges:
                continue
            predicate = edge_predicate(i, j, edge)
            if predicate is not None:
                acc = acc.where(predicate)

    projection = _projection([("t" + str(i), aliased[i], cfg(i)) for i in range(n)])
    if projection:
        acc = acc.select(*projection)

    return {
        "joined_data": acc,
        "left_unmatched": spark.createDataFrame([], aliased[0].schema),
        "right_unmatched": spark.createDataFrame([], aliased[n - 1].schema),
    }

# generated from the system
if "ld_display_outputs" not in globals():
    try:
        _ld_param = dbutils.widgets.getAll().get("ld_display_outputs")
        if _ld_param is not None and str(_ld_param).strip() != "":
            globals()["ld_display_outputs"] = str(_ld_param).strip().lower() not in ("false", "0", "no", "off")
        else:
            try:
                from dbruntime.databricks_repl_context import get_context
                globals()["ld_display_outputs"] = not get_context().isInJob
            except Exception:
                globals()["ld_display_outputs"] = True
    except Exception:
        globals()["ld_display_outputs"] = False
if "ld_display_outputs_for" not in globals():
    try:
        _ld_for = dbutils.widgets.getAll().get("ld_display_outputs_for")
        globals()["ld_display_outputs_for"] = frozenset(_p.strip() for _p in str(_ld_for).split(",") if _p.strip()) if _ld_for is not None else frozenset()
    except Exception:
        globals()["ld_display_outputs_for"] = frozenset()
ctx = globals().setdefault("ctx", {})
config = {
    "join_type": "left",
    "match_case": False,
    "multi_table": False,
    "conditions": [
        {
            "left_input": 0,
            "right_input": 1,
            "join_keys": [
                {
                    "left": "account_id",
                    "right": "account_id"
                }
            ],
            "join_conditions": ""
        }
    ],
    "columns": [
        {
            "edits": [],
            "ordered": []
        },
        {
            "edits": [],
            "ordered": []
        }
    ]
}
config["meta_state"] = {"is_preview": False, "is_focused_preview": False, "is_disabled": False}
inputs = {
    "left": ctx["filter_b72e1f84.filtered_data"],
    "right": ctx["aggregate_2e5617bb.aggregated_data"]
}
if config["meta_state"]["is_disabled"]:
    def _lb_limit_zero(value):
        if hasattr(value, "limit"):
            return value.limit(0)
        if isinstance(value, list):
            return [_lb_limit_zero(item) for item in value]
        return value
    inputs = {k: _lb_limit_zero(v) for k, v in inputs.items()}
out = run(config, inputs, spark)
if config["meta_state"]["is_disabled"]:
    out = {k: _lb_limit_zero(v) for k, v in out.items()}
ctx["join_support_exposure_65e02cbb.joined_data"] = out["joined_data"]
ctx["join_support_exposure_65e02cbb.left_unmatched"] = out["left_unmatched"]
ctx["join_support_exposure_65e02cbb.right_unmatched"] = out["right_unmatched"]
if globals().get("ld_display_outputs", False) or "join_support_exposure_65e02cbb" in globals().get("ld_display_outputs_for", frozenset()):
    display(ctx["join_support_exposure_65e02cbb.joined_data"])
    display(ctx["join_support_exposure_65e02cbb.left_unmatched"])
    display(ctx["join_support_exposure_65e02cbb.right_unmatched"])

In [0]:
"""
id: aggregate_14b39d7f
template: aggregate
templateVersion: 2.0.0
name: agg_usage_trend
position:
  x: 1500
  y: 751.25
description:
  text: Group by account ID and calculate averages and sums for user activity, API calls, and failed calls in the latest and previous 14-day periods.
  hash: dda6b18c
previewCodeHash: f8634487b0467c51
config:
  group_bys:
    - expr: account_id
      type: expr
  aggregations:
    - columnExpr:
        expr: IF(latest_window_flag, active_users, NULL)
        type: expr
      fn: AVG
      alias: avg_active_users_latest_14_days
    - columnExpr:
        expr: IF(previous_window_flag, active_users, NULL)
        type: expr
      fn: AVG
      alias: avg_active_users_previous_14_days
    - columnExpr:
        expr: IF(latest_window_flag, api_calls, NULL)
        type: expr
      fn: SUM
      alias: api_calls_latest_14_days
    - columnExpr:
        expr: IF(latest_window_flag, failed_api_calls, NULL)
        type: expr
      fn: SUM
      alias: failed_api_calls_latest_14_days
    - columnExpr:
        expr: IF(latest_window_flag, key_feature_events, NULL)
        type: expr
      fn: SUM
      alias: key_feature_events_latest_14_days
input:
  - node: prepare_b02cb6b8
    input_port: data
    output_port: prepared_data
"""

# generated from the system
import math
from typing import Dict, Any
import pyspark.sql.functions as F

DEFAULT_PERCENTILE = 0.5

DEFAULT_CONCAT_SEPARATOR = ", "

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs.get("data")
    group_bys = config.get("group_bys", [])
    aggregations = config.get("aggregations", [])

    group_by_set = set(e for gb in group_bys if (e := gb.get("expr", "")))

    agg_exprs = []
    for agg_def in aggregations:
        col_expr = agg_def.get("columnExpr", {})
        raw_expr = col_expr.get("expr", "")
        fn = agg_def.get("fn", "-")
        alias = agg_def.get("alias")

        if (fn == "-" or fn == "_") and not alias and raw_expr in group_by_set:
            continue

        fn_map = {
            "SUM": F.sum,
            "AVG": F.avg,
            "COUNT": F.count,
            "MIN": F.min,
            "MAX": F.max,
            "MEAN": F.mean,
            "MEDIAN": F.median,
            "STDDEV": F.stddev,
            "VARIANCE": F.variance,
            "FIRST": F.first,
            "LAST": F.last,
        }

        agg_fn = fn_map.get(fn)
        if agg_fn:
            arg = F.expr(raw_expr) if col_expr.get("type") == "expr" else raw_expr
            col = agg_fn(arg)
        elif fn == "-" or fn == "_":
            col = F.expr(raw_expr) if col_expr.get("type") == "expr" else F.col(raw_expr)
        elif fn == "PERCENTILE":
            raw_pct = agg_def.get("percentage")
            if (
                isinstance(raw_pct, (int, float))
                and not isinstance(raw_pct, bool)
                and math.isfinite(raw_pct)
            ):
                pct = max(0.0, min(1.0, float(raw_pct)))
            else:
                pct = DEFAULT_PERCENTILE
            col = F.expr(f"PERCENTILE({raw_expr}, {pct})")
        elif fn == "CONCAT":
            raw_sep = agg_def.get("separator")
            sep = raw_sep if isinstance(raw_sep, str) else DEFAULT_CONCAT_SEPARATOR
            concat_arg = F.expr(raw_expr) if col_expr.get("type") == "expr" else raw_expr
            col = F.concat_ws(sep, F.collect_list(concat_arg))
        elif fn == "COUNT_DISTINCT":
            arg = F.expr(raw_expr) if col_expr.get("type") != "column" else raw_expr
            col = F.count_distinct(arg)
        else:
            col = F.expr(f"{fn}({raw_expr})")

        if alias:
            col = col.alias(alias)

        agg_exprs.append(col)

    group_cols = [
        gb.get("expr", "") for gb in group_bys if gb.get("expr", "")
    ]

    if not agg_exprs:
        if group_cols:
            result = df.select(*group_cols).distinct()
            return {"aggregated_data": result}
        return {"aggregated_data": df}

    if group_cols:
        result = df.groupBy(*group_cols).agg(*agg_exprs)
    else:
        result = df.agg(*agg_exprs)

    return {"aggregated_data": result}

# generated from the system
if "ld_display_outputs" not in globals():
    try:
        _ld_param = dbutils.widgets.getAll().get("ld_display_outputs")
        if _ld_param is not None and str(_ld_param).strip() != "":
            globals()["ld_display_outputs"] = str(_ld_param).strip().lower() not in ("false", "0", "no", "off")
        else:
            try:
                from dbruntime.databricks_repl_context import get_context
                globals()["ld_display_outputs"] = not get_context().isInJob
            except Exception:
                globals()["ld_display_outputs"] = True
    except Exception:
        globals()["ld_display_outputs"] = False
if "ld_display_outputs_for" not in globals():
    try:
        _ld_for = dbutils.widgets.getAll().get("ld_display_outputs_for")
        globals()["ld_display_outputs_for"] = frozenset(_p.strip() for _p in str(_ld_for).split(",") if _p.strip()) if _ld_for is not None else frozenset()
    except Exception:
        globals()["ld_display_outputs_for"] = frozenset()
ctx = globals().setdefault("ctx", {})
config = {
    "group_bys": [
        {
            "expr": "account_id",
            "type": "expr"
        }
    ],
    "aggregations": [
        {
            "columnExpr": {
                "expr": "IF(latest_window_flag, active_users, NULL)",
                "type": "expr"
            },
            "fn": "AVG",
            "alias": "avg_active_users_latest_14_days",
            "withAsKeyword": None
        },
        {
            "columnExpr": {
                "expr": "IF(previous_window_flag, active_users, NULL)",
                "type": "expr"
            },
            "fn": "AVG",
            "alias": "avg_active_users_previous_14_days",
            "withAsKeyword": None
        },
        {
            "columnExpr": {
                "expr": "IF(latest_window_flag, api_calls, NULL)",
                "type": "expr"
            },
            "fn": "SUM",
            "alias": "api_calls_latest_14_days",
            "withAsKeyword": None
        },
        {
            "columnExpr": {
                "expr": "IF(latest_window_flag, failed_api_calls, NULL)",
                "type": "expr"
            },
            "fn": "SUM",
            "alias": "failed_api_calls_latest_14_days",
            "withAsKeyword": None
        },
        {
            "columnExpr": {
                "expr": "IF(latest_window_flag, key_feature_events, NULL)",
                "type": "expr"
            },
            "fn": "SUM",
            "alias": "key_feature_events_latest_14_days",
            "withAsKeyword": None
        }
    ]
}
config["meta_state"] = {"is_preview": False, "is_focused_preview": False, "is_disabled": False}
inputs = {
    "data": ctx["prepare_b02cb6b8.prepared_data"]
}
if config["meta_state"]["is_disabled"]:
    def _lb_limit_zero(value):
        if hasattr(value, "limit"):
            return value.limit(0)
        if isinstance(value, list):
            return [_lb_limit_zero(item) for item in value]
        return value
    inputs = {k: _lb_limit_zero(v) for k, v in inputs.items()}
out = run(config, inputs, spark)
if config["meta_state"]["is_disabled"]:
    out = {k: _lb_limit_zero(v) for k, v in out.items()}
ctx["aggregate_14b39d7f.aggregated_data"] = out["aggregated_data"]
if globals().get("ld_display_outputs", False) or "aggregate_14b39d7f" in globals().get("ld_display_outputs_for", frozenset()):
    display(ctx["aggregate_14b39d7f.aggregated_data"])

In [0]:
"""
id: aggregate_c0abc43a
template: aggregate
templateVersion: 2.0.0
name: agg_usage_trend
position:
  x: 1500
  y: 906.25
description:
  text: Group by account ID and calculate averages and sums for active users, API calls, failed API calls, and key feature events over specific time windows.
  hash: d9a7a031
previewCodeHash: 97621bc7c347d751
config:
  group_bys:
    - expr: account_id
      type: expr
  aggregations:
    - columnExpr:
        expr: IF(latest_window_flag, active_users, NULL)
        type: expr
      fn: AVG
      alias: avg_active_users_latest_14_days
    - columnExpr:
        expr: IF(previous_window_flag, active_users, NULL)
        type: expr
      fn: AVG
      alias: avg_active_users_previous_14_days
    - columnExpr:
        expr: IF(latest_window_flag, api_calls, NULL)
        type: expr
      fn: SUM
      alias: api_calls_latest_14_days
    - columnExpr:
        expr: IF(latest_window_flag, failed_api_calls, NULL)
        type: expr
      fn: SUM
      alias: failed_api_calls_latest_14_days
    - columnExpr:
        expr: IF(latest_window_flag, key_feature_events, NULL)
        type: expr
      fn: SUM
      alias: key_feature_events_latest_14_days
input:
  - node: prepare_b02cb6b8
    input_port: data
    output_port: prepared_data
"""

# generated from the system
import math
from typing import Dict, Any
import pyspark.sql.functions as F

DEFAULT_PERCENTILE = 0.5

DEFAULT_CONCAT_SEPARATOR = ", "

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs.get("data")
    group_bys = config.get("group_bys", [])
    aggregations = config.get("aggregations", [])

    group_by_set = set(e for gb in group_bys if (e := gb.get("expr", "")))

    agg_exprs = []
    for agg_def in aggregations:
        col_expr = agg_def.get("columnExpr", {})
        raw_expr = col_expr.get("expr", "")
        fn = agg_def.get("fn", "-")
        alias = agg_def.get("alias")

        if (fn == "-" or fn == "_") and not alias and raw_expr in group_by_set:
            continue

        fn_map = {
            "SUM": F.sum,
            "AVG": F.avg,
            "COUNT": F.count,
            "MIN": F.min,
            "MAX": F.max,
            "MEAN": F.mean,
            "MEDIAN": F.median,
            "STDDEV": F.stddev,
            "VARIANCE": F.variance,
            "FIRST": F.first,
            "LAST": F.last,
        }

        agg_fn = fn_map.get(fn)
        if agg_fn:
            arg = F.expr(raw_expr) if col_expr.get("type") == "expr" else raw_expr
            col = agg_fn(arg)
        elif fn == "-" or fn == "_":
            col = F.expr(raw_expr) if col_expr.get("type") == "expr" else F.col(raw_expr)
        elif fn == "PERCENTILE":
            raw_pct = agg_def.get("percentage")
            if (
                isinstance(raw_pct, (int, float))
                and not isinstance(raw_pct, bool)
                and math.isfinite(raw_pct)
            ):
                pct = max(0.0, min(1.0, float(raw_pct)))
            else:
                pct = DEFAULT_PERCENTILE
            col = F.expr(f"PERCENTILE({raw_expr}, {pct})")
        elif fn == "CONCAT":
            raw_sep = agg_def.get("separator")
            sep = raw_sep if isinstance(raw_sep, str) else DEFAULT_CONCAT_SEPARATOR
            concat_arg = F.expr(raw_expr) if col_expr.get("type") == "expr" else raw_expr
            col = F.concat_ws(sep, F.collect_list(concat_arg))
        elif fn == "COUNT_DISTINCT":
            arg = F.expr(raw_expr) if col_expr.get("type") != "column" else raw_expr
            col = F.count_distinct(arg)
        else:
            col = F.expr(f"{fn}({raw_expr})")

        if alias:
            col = col.alias(alias)

        agg_exprs.append(col)

    group_cols = [
        gb.get("expr", "") for gb in group_bys if gb.get("expr", "")
    ]

    if not agg_exprs:
        if group_cols:
            result = df.select(*group_cols).distinct()
            return {"aggregated_data": result}
        return {"aggregated_data": df}

    if group_cols:
        result = df.groupBy(*group_cols).agg(*agg_exprs)
    else:
        result = df.agg(*agg_exprs)

    return {"aggregated_data": result}

# generated from the system
if "ld_display_outputs" not in globals():
    try:
        _ld_param = dbutils.widgets.getAll().get("ld_display_outputs")
        if _ld_param is not None and str(_ld_param).strip() != "":
            globals()["ld_display_outputs"] = str(_ld_param).strip().lower() not in ("false", "0", "no", "off")
        else:
            try:
                from dbruntime.databricks_repl_context import get_context
                globals()["ld_display_outputs"] = not get_context().isInJob
            except Exception:
                globals()["ld_display_outputs"] = True
    except Exception:
        globals()["ld_display_outputs"] = False
if "ld_display_outputs_for" not in globals():
    try:
        _ld_for = dbutils.widgets.getAll().get("ld_display_outputs_for")
        globals()["ld_display_outputs_for"] = frozenset(_p.strip() for _p in str(_ld_for).split(",") if _p.strip()) if _ld_for is not None else frozenset()
    except Exception:
        globals()["ld_display_outputs_for"] = frozenset()
ctx = globals().setdefault("ctx", {})
config = {
    "group_bys": [
        {
            "expr": "account_id",
            "type": "expr"
        }
    ],
    "aggregations": [
        {
            "columnExpr": {
                "expr": "IF(latest_window_flag, active_users, NULL)",
                "type": "expr"
            },
            "fn": "AVG",
            "alias": "avg_active_users_latest_14_days",
            "withAsKeyword": None
        },
        {
            "columnExpr": {
                "expr": "IF(previous_window_flag, active_users, NULL)",
                "type": "expr"
            },
            "fn": "AVG",
            "alias": "avg_active_users_previous_14_days",
            "withAsKeyword": None
        },
        {
            "columnExpr": {
                "expr": "IF(latest_window_flag, api_calls, NULL)",
                "type": "expr"
            },
            "fn": "SUM",
            "alias": "api_calls_latest_14_days",
            "withAsKeyword": None
        },
        {
            "columnExpr": {
                "expr": "IF(latest_window_flag, failed_api_calls, NULL)",
                "type": "expr"
            },
            "fn": "SUM",
            "alias": "failed_api_calls_latest_14_days",
            "withAsKeyword": None
        },
        {
            "columnExpr": {
                "expr": "IF(latest_window_flag, key_feature_events, NULL)",
                "type": "expr"
            },
            "fn": "SUM",
            "alias": "key_feature_events_latest_14_days",
            "withAsKeyword": None
        }
    ]
}
config["meta_state"] = {"is_preview": False, "is_focused_preview": False, "is_disabled": False}
inputs = {
    "data": ctx["prepare_b02cb6b8.prepared_data"]
}
if config["meta_state"]["is_disabled"]:
    def _lb_limit_zero(value):
        if hasattr(value, "limit"):
            return value.limit(0)
        if isinstance(value, list):
            return [_lb_limit_zero(item) for item in value]
        return value
    inputs = {k: _lb_limit_zero(v) for k, v in inputs.items()}
out = run(config, inputs, spark)
if config["meta_state"]["is_disabled"]:
    out = {k: _lb_limit_zero(v) for k, v in out.items()}
ctx["aggregate_c0abc43a.aggregated_data"] = out["aggregated_data"]
if globals().get("ld_display_outputs", False) or "aggregate_c0abc43a" in globals().get("ld_display_outputs_for", frozenset()):
    display(ctx["aggregate_c0abc43a.aggregated_data"])

In [0]:
"""
id: join_pipeline_health_09aadfe6
template: join
templateVersion: 3.0.0
name: join_pipeline_health
position:
  x: 1800
  y: 441.25
description:
  text: Combine two inputs by matching on account_id and keep all records from the left input.
  hash: c529d80a
config:
  join_type: left
  match_case: false
  multi_table: false
  conditions:
    - left_input: 0
      right_input: 1
      join_keys:
        - left: account_id
          right: account_id
      join_conditions: ""
  columns:
    - edits: []
      ordered: []
    - edits: []
      ordered: []
input:
  - node: join_support_exposure_65e02cbb
    input_port: left
    output_port: joined_data
  - node: aggregate_87c4fde9
    input_port: right
    output_port: aggregated_data
"""

# generated from the system
from typing import Any, Dict, List, Optional
import pyspark.sql.functions as F
from pyspark.sql.types import StringType

def _is_checked(item: Dict[str, Any]) -> bool:
    return item.get("checked", True) is not False

def _qualified_col(alias: str, name: str):
    escaped = name.replace("`", "``")
    return F.col("`" + alias + "`.`" + escaped + "`")

def _ordered_names(df_cols: List[str], cfg: Dict[str, Any]) -> List[str]:
    ordered: List[str] = cfg.get("ordered") or []
    col_set = set(df_cols)
    placed = set()
    result: List[str] = []
    for name in ordered:
        if name in placed or name not in col_set:
            continue
        placed.add(name)
        result.append(name)
    for name in df_cols:
        if name in placed:
            continue
        placed.add(name)
        result.append(name)
    return result

def _edits_by_column(cfg: Dict[str, Any]) -> Dict[str, Dict[str, Any]]:
    by_col: Dict[str, Dict[str, Any]] = {}
    for item in cfg.get("edits", []) or []:
        col = item.get("column")
        if col is not None and col not in by_col:
            by_col[col] = item
    return by_col

def _projection(frames: List[Any]):
    out = []
    seen: Dict[str, int] = {}
    n = len(frames)
    for idx, (alias, df, cfg) in enumerate(frames):
        edits = _edits_by_column(cfg)
        for name in _ordered_names(list(df.columns), cfg):
            edit = edits.get(name)
            if edit is not None and not _is_checked(edit):
                continue
            col = _qualified_col(alias, name)
            user_alias = edit.get("alias") if edit else None
            if user_alias:
                out.append(col.alias(user_alias))
                seen[user_alias.lower()] = seen.get(user_alias.lower(), 0) + 1
                continue
            key = name.lower()
            if key not in seen:
                seen[key] = 1
                out.append(col.alias(name))
                continue
            new_name = ("right_" + name) if (n == 2 and idx == 1) else (name + "_" + str(idx + 1))
            while new_name.lower() in seen:
                new_name = new_name + "_dup"
            seen[new_name.lower()] = 1
            out.append(col.alias(new_name))
    return out

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    dfs: List[Any] = []
    for port in ("left", "right"):
        df = inputs.get(port)
        if df is not None:
            dfs.append(df)
    dfs.extend(inputs.get("data") or [])
    if len(dfs) < 2:
        raise ValueError(
            f"Join requires at least 2 inputs, got {len(dfs)}. "
            "Wire 2 or more upstream operators into this Join."
        )

    n = len(dfs)
    join_type = config.get("join_type") or "split_join"
    match_case = bool(config.get("match_case", False))
    conditions: List[Dict[str, Any]] = config.get("conditions") or []
    columns: List[Dict[str, Any]] = config.get("columns") or []

    aliased = [df.alias("t" + str(i)) for i, df in enumerate(dfs)]
    types = [{f.name.lower(): f.dataType for f in df.schema} for df in aliased]

    def key_col(idx: int, name: str):
        col = _qualified_col("t" + str(idx), name)
        if not match_case and isinstance(types[idx].get(name.lower()), StringType):
            return F.upper(F.trim(col))
        return col

    def edge_predicate(i: int, j: int, edge: Dict[str, Any]) -> Optional[Any]:
        predicates = []
        for key in edge.get("join_keys") or []:
            predicates.append(key_col(i, key["left"]) == key_col(j, key["right"]))
        condition = edge.get("join_conditions") or ""
        if condition:
            predicates.append(F.expr(condition))
        join_expr = None
        for predicate in predicates:
            join_expr = predicate if join_expr is None else join_expr & predicate
        return join_expr

    edges: List[Any] = [
        (int(edge.get("left_input", 0)), int(edge.get("right_input", 0)), edge)
        for edge in conditions
    ]

    predicates_by_step: Dict[int, List[Any]] = {}
    for i, j, edge in edges:
        predicate = edge_predicate(i, j, edge)
        if predicate is not None:
            predicates_by_step.setdefault(max(i, j), []).append(predicate)

    def predicate_at(step: int) -> Optional[Any]:
        join_expr = None
        for predicate in predicates_by_step.get(step, []):
            join_expr = predicate if join_expr is None else join_expr & predicate
        return join_expr

    def cfg(i: int) -> Dict[str, Any]:
        return columns[i] if i < len(columns) else {}

    is_two = n == 2
    is_split = is_two and join_type == "split_join"

    if is_split:
        join_expr = predicate_at(1)
        left_df, right_df = aliased[0], aliased[1]
        if join_expr is None:
            matched = left_df.join(right_df, how="inner")
            left_unmatched = left_df.join(right_df, how="left_anti")
            right_unmatched = right_df.join(left_df, how="left_anti")
        else:
            matched = left_df.join(right_df, join_expr, how="inner")
            left_unmatched = left_df.join(right_df, join_expr, how="left_anti")
            right_unmatched = right_df.join(left_df, join_expr, how="left_anti")
        projection = _projection([("t0", left_df, cfg(0)), ("t1", right_df, cfg(1))])
        if projection:
            matched = matched.select(*projection)
        return {
            "joined_data": matched,
            "left_unmatched": left_unmatched,
            "right_unmatched": right_unmatched,
        }

    how = join_type if is_two else "inner"
    if is_two:
        acc = aliased[0]
        predicate = predicate_at(1)
        if predicate is None:
            acc = acc.join(aliased[1], how=how)
        else:
            acc = acc.join(aliased[1], predicate, how=how)
    else:
        adjacency: Dict[int, List[int]] = {}
        for idx, (i, j, _edge) in enumerate(edges):
            adjacency.setdefault(i, []).append(idx)
            adjacency.setdefault(j, []).append(idx)
        visited = [False] * n
        visited[0] = True
        acc = aliased[0]
        used_edges = set()
        queue: List[int] = [0]
        while queue:
            cur = queue.pop(0)
            for idx in adjacency.get(cur, []):
                i, j, edge = edges[idx]
                nxt = j if i == cur else i
                if visited[nxt]:
                    continue
                visited[nxt] = True
                used_edges.add(idx)
                predicate = edge_predicate(i, j, edge)
                if predicate is None:
                    acc = acc.join(aliased[nxt], how=how)
                else:
                    acc = acc.join(aliased[nxt], predicate, how=how)
                queue.append(nxt)
        for idx, (i, j, edge) in enumerate(edges):
            if idx in used_edges:
                continue
            predicate = edge_predicate(i, j, edge)
            if predicate is not None:
                acc = acc.where(predicate)

    projection = _projection([("t" + str(i), aliased[i], cfg(i)) for i in range(n)])
    if projection:
        acc = acc.select(*projection)

    return {
        "joined_data": acc,
        "left_unmatched": spark.createDataFrame([], aliased[0].schema),
        "right_unmatched": spark.createDataFrame([], aliased[n - 1].schema),
    }

# generated from the system
if "ld_display_outputs" not in globals():
    try:
        _ld_param = dbutils.widgets.getAll().get("ld_display_outputs")
        if _ld_param is not None and str(_ld_param).strip() != "":
            globals()["ld_display_outputs"] = str(_ld_param).strip().lower() not in ("false", "0", "no", "off")
        else:
            try:
                from dbruntime.databricks_repl_context import get_context
                globals()["ld_display_outputs"] = not get_context().isInJob
            except Exception:
                globals()["ld_display_outputs"] = True
    except Exception:
        globals()["ld_display_outputs"] = False
if "ld_display_outputs_for" not in globals():
    try:
        _ld_for = dbutils.widgets.getAll().get("ld_display_outputs_for")
        globals()["ld_display_outputs_for"] = frozenset(_p.strip() for _p in str(_ld_for).split(",") if _p.strip()) if _ld_for is not None else frozenset()
    except Exception:
        globals()["ld_display_outputs_for"] = frozenset()
ctx = globals().setdefault("ctx", {})
config = {
    "join_type": "left",
    "match_case": False,
    "multi_table": False,
    "conditions": [
        {
            "left_input": 0,
            "right_input": 1,
            "join_keys": [
                {
                    "left": "account_id",
                    "right": "account_id"
                }
            ],
            "join_conditions": ""
        }
    ],
    "columns": [
        {
            "edits": [],
            "ordered": []
        },
        {
            "edits": [],
            "ordered": []
        }
    ]
}
config["meta_state"] = {"is_preview": False, "is_focused_preview": False, "is_disabled": False}
inputs = {
    "left": ctx["join_support_exposure_65e02cbb.joined_data"],
    "right": ctx["aggregate_87c4fde9.aggregated_data"]
}
if config["meta_state"]["is_disabled"]:
    def _lb_limit_zero(value):
        if hasattr(value, "limit"):
            return value.limit(0)
        if isinstance(value, list):
            return [_lb_limit_zero(item) for item in value]
        return value
    inputs = {k: _lb_limit_zero(v) for k, v in inputs.items()}
out = run(config, inputs, spark)
if config["meta_state"]["is_disabled"]:
    out = {k: _lb_limit_zero(v) for k, v in out.items()}
ctx["join_pipeline_health_09aadfe6.joined_data"] = out["joined_data"]
ctx["join_pipeline_health_09aadfe6.left_unmatched"] = out["left_unmatched"]
ctx["join_pipeline_health_09aadfe6.right_unmatched"] = out["right_unmatched"]
if globals().get("ld_display_outputs", False) or "join_pipeline_health_09aadfe6" in globals().get("ld_display_outputs_for", frozenset()):
    display(ctx["join_pipeline_health_09aadfe6.joined_data"])
    display(ctx["join_pipeline_health_09aadfe6.left_unmatched"])
    display(ctx["join_pipeline_health_09aadfe6.right_unmatched"])

In [0]:
"""
id: prep_usage_trend_metrics_e96b3d85
template: prepare
templateVersion: 1.0.0
name: prep_usage_trend_metrics
position:
  x: 1800
  y: 673.75
description:
  text: Create two new columns calculating API failure rate and active user growth rate over the last 14 days.
  hash: 17e59a57
config:
  actions:
    - type: formula
      target: api_failure_rate_latest_14_days
      expression: CASE WHEN api_calls_latest_14_days = 0 THEN 0 ELSE failed_api_calls_latest_14_days / api_calls_latest_14_days END
    - type: formula
      target: active_user_growth_rate
      expression: CASE WHEN avg_active_users_previous_14_days = 0 THEN 0 ELSE (avg_active_users_latest_14_days - avg_active_users_previous_14_days) / avg_active_users_previous_14_days END
input:
  - node: aggregate_14b39d7f
    input_port: data
    output_port: aggregated_data
"""

# generated from the system
from typing import Any, Callable, Dict, List
import pyspark.sql.functions as F

TEXT_CASE_FUNCTIONS: Dict[str, Callable] = {
    "lower": F.lower,
    "upper": F.upper,
    "title": F.initcap,
}

TRIM_FUNCTIONS: Dict[str, Callable] = {
    "both": F.trim,
    "left": F.ltrim,
    "right": F.rtrim,
}

VALID_MATCH_MODES = {"exact", "contains", "prefix", "suffix", "regex"}

def _require_column(df, column: str, action_type: str) -> None:
    if not column:
        raise ValueError(f"{action_type}: 'column' is required")
    if column not in df.columns:
        raise ValueError(
            f"{action_type}: column '{column}' not found in input data. "
            f"Available columns: {df.columns}"
        )

def _column_dtype(df, column: str) -> str:
    return df.schema[column].dataType.simpleString()

def _quoted_ident(name: str) -> str:
    return "`" + name.replace("`", "``") + "`"

def _col(name: str):
    """Reference a column by exact name.

    F.col parses its argument as a column expression, so a name containing
    a '.' (e.g. "a.b") is otherwise read as struct-field access and fails
    even though the column exists. Backtick-quoting forces an exact-name
    lookup; quoting plain names is harmless.
    """
    return F.col(_quoted_ident(name))

def _coerced_lit(value: Any, target_dtype: str):
    """Wrap a user-provided value as a literal cast to the target column's type.

    UI always feeds text per the operator spec; this is where the
    type coercion happens so the per-action UI stays simple.
    """
    return F.lit(value).cast(target_dtype)

def _apply_formula(df, action: Dict[str, Any]):
    target = action.get("target", "")
    expression = action.get("expression", "")
    if not target:
        raise ValueError("formula: 'target' is required")
    if not expression:
        raise ValueError("formula: 'expression' is required")
    return df.withColumn(target, F.expr(expression))

def _apply_cast(df, action: Dict[str, Any]):
    column = action.get("column", "")
    to_type = action.get("to", "")
    on_error = action.get("on_error", "null")
    _require_column(df, column, "cast")
    if not to_type:
        raise ValueError("cast: 'to' (target type) is required")
    if on_error == "null":
        return df.withColumn(
            column,
            F.expr(f"try_cast({_quoted_ident(column)} as {to_type})"),
        )
    return df.withColumn(column, _col(column).cast(to_type))

def _apply_replace_value(df, action: Dict[str, Any]):
    column = action.get("column", "")
    match_mode = action.get("match_mode", "exact")
    match = action.get("match", "")
    with_val = action.get("with", "")
    case_sensitive = bool(action.get("case_sensitive", False))
    _require_column(df, column, "replace_value")
    if match_mode not in VALID_MATCH_MODES:
        raise ValueError(
            f"replace_value: unsupported match_mode '{match_mode}'. "
            f"Choose one of: {sorted(VALID_MATCH_MODES)}"
        )
    dtype = _column_dtype(df, column)
    col_as_string = _col(column).cast("string")
    match_lit = F.lit(match)
    if case_sensitive:
        cmp_col = col_as_string
        cmp_lit = match_lit
    else:
        cmp_col = F.lower(col_as_string)
        cmp_lit = F.lower(match_lit)

    if match_mode == "exact":
        predicate = cmp_col == cmp_lit
    elif match_mode == "contains":
        predicate = cmp_col.contains(cmp_lit)
    elif match_mode == "prefix":
        predicate = cmp_col.startswith(cmp_lit)
    elif match_mode == "suffix":
        predicate = cmp_col.endswith(cmp_lit)
    else:  # regex
        pattern = match if case_sensitive else f"(?i){match}"
        predicate = col_as_string.rlike(pattern)

    replacement = _coerced_lit(with_val, dtype)
    return df.withColumn(
        column,
        F.when(predicate, replacement).otherwise(_col(column)),
    )

def _apply_fill_null(df, action: Dict[str, Any]):
    column = action.get("column", "")
    with_val = action.get("with", "")
    _require_column(df, column, "fill_null")
    dtype = _column_dtype(df, column)
    replacement = _coerced_lit(with_val, dtype)
    return df.withColumn(
        column,
        F.when(_col(column).isNull(), replacement).otherwise(_col(column)),
    )

def _apply_text_case(df, action: Dict[str, Any]):
    column = action.get("column", "")
    case = action.get("case", "")
    _require_column(df, column, "text_case")
    fn = TEXT_CASE_FUNCTIONS.get(case)
    if fn is None:
        raise ValueError(
            f"text_case: unsupported case '{case}'. "
            f"Choose one of: {sorted(TEXT_CASE_FUNCTIONS.keys())}"
        )
    return df.withColumn(column, fn(_col(column)))

def _apply_trim(df, action: Dict[str, Any]):
    column = action.get("column", "")
    side = action.get("side", "both")
    _require_column(df, column, "trim")
    fn = TRIM_FUNCTIONS.get(side)
    if fn is None:
        raise ValueError(
            f"trim: unsupported side '{side}'. "
            f"Choose one of: {sorted(TRIM_FUNCTIONS.keys())}"
        )
    return df.withColumn(column, fn(_col(column)))

def _apply_regex_replace(df, action: Dict[str, Any]):
    column = action.get("column", "")
    pattern = action.get("pattern", "")
    replacement = action.get("replacement", "")
    _require_column(df, column, "regex_replace")
    return df.withColumn(
        column,
        F.regexp_replace(_col(column), pattern, replacement),
    )

def _apply_extract(df, action: Dict[str, Any]):
    column = action.get("column", "")
    pattern = action.get("pattern", "")
    target = action.get("target") or column
    group_raw = action.get("group", 0)
    _require_column(df, column, "extract")
    try:
        group_idx = int(group_raw)
    except (TypeError, ValueError) as exc:
        raise ValueError(
            f"extract: 'group' must be an integer, got {group_raw!r}"
        ) from exc
    return df.withColumn(
        target,
        F.regexp_extract(_col(column), pattern, group_idx),
    )

def _apply_parse_date(df, action: Dict[str, Any]):
    column = action.get("column", "")
    kind = action.get("kind", "date")
    fmt = action.get("format") or None
    on_error = action.get("on_error", "null")
    _require_column(df, column, "parse_date")
    if kind not in ("date", "timestamp"):
        raise ValueError(
            f"parse_date: unsupported kind '{kind}'. Choose date or timestamp."
        )
    # PySpark exposes F.try_to_timestamp from Spark 3.5 but only adds
    # F.try_to_date in Spark 4.0. Current Databricks Runtimes still ship
    # Spark 3.5, where referencing F.try_to_date raises AttributeError
    # before any data is touched. Route the null-on-error path through
    # primitives that exist in both 3.5 and 4.0:
    #   - F.try_to_timestamp  (PySpark 3.5+, returns NULL under ANSI too)
    #   - SQL try_cast        (Spark 3.5+, returns NULL under ANSI too)
    #   - F.to_date           (PySpark 3.5+; returns NULL on parse failure
    #                          under the default non-ANSI mode, which is
    #                          the Databricks default. Under ANSI mode it
    #                          raises — accepted limitation until
    #                          try_to_date lands on every supported DBR.)
    if on_error == "null":
        if kind == "timestamp":
            if fmt:
                return df.withColumn(
                    column, F.try_to_timestamp(_col(column), F.lit(fmt))
                )
            return df.withColumn(column, F.try_to_timestamp(_col(column)))
        # kind == "date"
        if fmt:
            return df.withColumn(column, F.to_date(_col(column), fmt))
        return df.withColumn(
            column, F.expr(f"try_cast({_quoted_ident(column)} as date)")
        )
    # on_error == "error": surface parse failures as Spark exceptions.
    fn = F.to_date if kind == "date" else F.to_timestamp
    if fmt:
        return df.withColumn(column, fn(_col(column), fmt))
    return df.withColumn(column, fn(_col(column)))

ACTION_DISPATCH: Dict[str, Callable] = {
    "formula": _apply_formula,
    "cast": _apply_cast,
    "replace_value": _apply_replace_value,
    "fill_null": _apply_fill_null,
    "text_case": _apply_text_case,
    "trim": _apply_trim,
    "regex_replace": _apply_regex_replace,
    "extract": _apply_extract,
    "parse_date": _apply_parse_date,
}

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs.get("data")
    actions: List[Dict[str, Any]] = config.get("actions", []) or []

    if not actions:
        return {"prepared_data": df}

    for index, action in enumerate(actions):
        if not isinstance(action, dict):
            raise ValueError(
                f"actions[{index}]: expected an object, got {type(action).__name__}"
            )
        if action.get("enabled", True) is False:
            continue
        action_type = action.get("type", "")
        fn = ACTION_DISPATCH.get(action_type)
        if fn is None:
            raise ValueError(
                f"actions[{index}]: unsupported action type {action_type!r}. "
                f"Choose one of: {sorted(ACTION_DISPATCH.keys())}"
            )
        df = fn(df, action)
    return {"prepared_data": df}

# generated from the system
if "ld_display_outputs" not in globals():
    try:
        _ld_param = dbutils.widgets.getAll().get("ld_display_outputs")
        if _ld_param is not None and str(_ld_param).strip() != "":
            globals()["ld_display_outputs"] = str(_ld_param).strip().lower() not in ("false", "0", "no", "off")
        else:
            try:
                from dbruntime.databricks_repl_context import get_context
                globals()["ld_display_outputs"] = not get_context().isInJob
            except Exception:
                globals()["ld_display_outputs"] = True
    except Exception:
        globals()["ld_display_outputs"] = False
if "ld_display_outputs_for" not in globals():
    try:
        _ld_for = dbutils.widgets.getAll().get("ld_display_outputs_for")
        globals()["ld_display_outputs_for"] = frozenset(_p.strip() for _p in str(_ld_for).split(",") if _p.strip()) if _ld_for is not None else frozenset()
    except Exception:
        globals()["ld_display_outputs_for"] = frozenset()
ctx = globals().setdefault("ctx", {})
config = {
    "actions": [
        {
            "type": "formula",
            "target": "api_failure_rate_latest_14_days",
            "expression": "CASE WHEN api_calls_latest_14_days = 0 THEN 0 ELSE failed_api_calls_latest_14_days / api_calls_latest_14_days END"
        },
        {
            "type": "formula",
            "target": "active_user_growth_rate",
            "expression": "CASE WHEN avg_active_users_previous_14_days = 0 THEN 0 ELSE (avg_active_users_latest_14_days - avg_active_users_previous_14_days) / avg_active_users_previous_14_days END"
        }
    ]
}
config["meta_state"] = {"is_preview": False, "is_focused_preview": False, "is_disabled": False}
inputs = {
    "data": ctx["aggregate_14b39d7f.aggregated_data"]
}
if config["meta_state"]["is_disabled"]:
    def _lb_limit_zero(value):
        if hasattr(value, "limit"):
            return value.limit(0)
        if isinstance(value, list):
            return [_lb_limit_zero(item) for item in value]
        return value
    inputs = {k: _lb_limit_zero(v) for k, v in inputs.items()}
out = run(config, inputs, spark)
if config["meta_state"]["is_disabled"]:
    out = {k: _lb_limit_zero(v) for k, v in out.items()}
ctx["prep_usage_trend_metrics_e96b3d85.prepared_data"] = out["prepared_data"]
if globals().get("ld_display_outputs", False) or "prep_usage_trend_metrics_e96b3d85" in globals().get("ld_display_outputs_for", frozenset()):
    display(ctx["prep_usage_trend_metrics_e96b3d85.prepared_data"])

In [0]:
"""
id: prepare_e96b3d85
template: prepare
templateVersion: 1.0.0
name: prep_usage_trend_metrics
position:
  x: 1800
  y: 828.75
description:
  text: Calculate failure rate of API calls in last 14 days and growth rate of active users.
  hash: dc326a15
previewCodeHash: fce1d080090e0a6e
config:
  actions:
    - type: formula
      target: api_failure_rate_latest_14_days
      expression: CASE WHEN api_calls_latest_14_days = 0 THEN 0 ELSE failed_api_calls_latest_14_days / api_calls_latest_14_days END
    - type: formula
      target: active_user_growth_rate
      expression: CASE WHEN avg_active_users_previous_14_days = 0 THEN 0 ELSE (avg_active_users_latest_14_days - avg_active_users_previous_14_days) / avg_active_users_previous_14_days END
input:
  - node: aggregate_14b39d7f
    input_port: data
    output_port: aggregated_data
"""

# generated from the system
from typing import Any, Callable, Dict, List
import pyspark.sql.functions as F

TEXT_CASE_FUNCTIONS: Dict[str, Callable] = {
    "lower": F.lower,
    "upper": F.upper,
    "title": F.initcap,
}

TRIM_FUNCTIONS: Dict[str, Callable] = {
    "both": F.trim,
    "left": F.ltrim,
    "right": F.rtrim,
}

VALID_MATCH_MODES = {"exact", "contains", "prefix", "suffix", "regex"}

def _require_column(df, column: str, action_type: str) -> None:
    if not column:
        raise ValueError(f"{action_type}: 'column' is required")
    if column not in df.columns:
        raise ValueError(
            f"{action_type}: column '{column}' not found in input data. "
            f"Available columns: {df.columns}"
        )

def _column_dtype(df, column: str) -> str:
    return df.schema[column].dataType.simpleString()

def _quoted_ident(name: str) -> str:
    return "`" + name.replace("`", "``") + "`"

def _col(name: str):
    """Reference a column by exact name.

    F.col parses its argument as a column expression, so a name containing
    a '.' (e.g. "a.b") is otherwise read as struct-field access and fails
    even though the column exists. Backtick-quoting forces an exact-name
    lookup; quoting plain names is harmless.
    """
    return F.col(_quoted_ident(name))

def _coerced_lit(value: Any, target_dtype: str):
    """Wrap a user-provided value as a literal cast to the target column's type.

    UI always feeds text per the operator spec; this is where the
    type coercion happens so the per-action UI stays simple.
    """
    return F.lit(value).cast(target_dtype)

def _apply_formula(df, action: Dict[str, Any]):
    target = action.get("target", "")
    expression = action.get("expression", "")
    if not target:
        raise ValueError("formula: 'target' is required")
    if not expression:
        raise ValueError("formula: 'expression' is required")
    return df.withColumn(target, F.expr(expression))

def _apply_cast(df, action: Dict[str, Any]):
    column = action.get("column", "")
    to_type = action.get("to", "")
    on_error = action.get("on_error", "null")
    _require_column(df, column, "cast")
    if not to_type:
        raise ValueError("cast: 'to' (target type) is required")
    if on_error == "null":
        return df.withColumn(
            column,
            F.expr(f"try_cast({_quoted_ident(column)} as {to_type})"),
        )
    return df.withColumn(column, _col(column).cast(to_type))

def _apply_replace_value(df, action: Dict[str, Any]):
    column = action.get("column", "")
    match_mode = action.get("match_mode", "exact")
    match = action.get("match", "")
    with_val = action.get("with", "")
    case_sensitive = bool(action.get("case_sensitive", False))
    _require_column(df, column, "replace_value")
    if match_mode not in VALID_MATCH_MODES:
        raise ValueError(
            f"replace_value: unsupported match_mode '{match_mode}'. "
            f"Choose one of: {sorted(VALID_MATCH_MODES)}"
        )
    dtype = _column_dtype(df, column)
    col_as_string = _col(column).cast("string")
    match_lit = F.lit(match)
    if case_sensitive:
        cmp_col = col_as_string
        cmp_lit = match_lit
    else:
        cmp_col = F.lower(col_as_string)
        cmp_lit = F.lower(match_lit)

    if match_mode == "exact":
        predicate = cmp_col == cmp_lit
    elif match_mode == "contains":
        predicate = cmp_col.contains(cmp_lit)
    elif match_mode == "prefix":
        predicate = cmp_col.startswith(cmp_lit)
    elif match_mode == "suffix":
        predicate = cmp_col.endswith(cmp_lit)
    else:  # regex
        pattern = match if case_sensitive else f"(?i){match}"
        predicate = col_as_string.rlike(pattern)

    replacement = _coerced_lit(with_val, dtype)
    return df.withColumn(
        column,
        F.when(predicate, replacement).otherwise(_col(column)),
    )

def _apply_fill_null(df, action: Dict[str, Any]):
    column = action.get("column", "")
    with_val = action.get("with", "")
    _require_column(df, column, "fill_null")
    dtype = _column_dtype(df, column)
    replacement = _coerced_lit(with_val, dtype)
    return df.withColumn(
        column,
        F.when(_col(column).isNull(), replacement).otherwise(_col(column)),
    )

def _apply_text_case(df, action: Dict[str, Any]):
    column = action.get("column", "")
    case = action.get("case", "")
    _require_column(df, column, "text_case")
    fn = TEXT_CASE_FUNCTIONS.get(case)
    if fn is None:
        raise ValueError(
            f"text_case: unsupported case '{case}'. "
            f"Choose one of: {sorted(TEXT_CASE_FUNCTIONS.keys())}"
        )
    return df.withColumn(column, fn(_col(column)))

def _apply_trim(df, action: Dict[str, Any]):
    column = action.get("column", "")
    side = action.get("side", "both")
    _require_column(df, column, "trim")
    fn = TRIM_FUNCTIONS.get(side)
    if fn is None:
        raise ValueError(
            f"trim: unsupported side '{side}'. "
            f"Choose one of: {sorted(TRIM_FUNCTIONS.keys())}"
        )
    return df.withColumn(column, fn(_col(column)))

def _apply_regex_replace(df, action: Dict[str, Any]):
    column = action.get("column", "")
    pattern = action.get("pattern", "")
    replacement = action.get("replacement", "")
    _require_column(df, column, "regex_replace")
    return df.withColumn(
        column,
        F.regexp_replace(_col(column), pattern, replacement),
    )

def _apply_extract(df, action: Dict[str, Any]):
    column = action.get("column", "")
    pattern = action.get("pattern", "")
    target = action.get("target") or column
    group_raw = action.get("group", 0)
    _require_column(df, column, "extract")
    try:
        group_idx = int(group_raw)
    except (TypeError, ValueError) as exc:
        raise ValueError(
            f"extract: 'group' must be an integer, got {group_raw!r}"
        ) from exc
    return df.withColumn(
        target,
        F.regexp_extract(_col(column), pattern, group_idx),
    )

def _apply_parse_date(df, action: Dict[str, Any]):
    column = action.get("column", "")
    kind = action.get("kind", "date")
    fmt = action.get("format") or None
    on_error = action.get("on_error", "null")
    _require_column(df, column, "parse_date")
    if kind not in ("date", "timestamp"):
        raise ValueError(
            f"parse_date: unsupported kind '{kind}'. Choose date or timestamp."
        )
    # PySpark exposes F.try_to_timestamp from Spark 3.5 but only adds
    # F.try_to_date in Spark 4.0. Current Databricks Runtimes still ship
    # Spark 3.5, where referencing F.try_to_date raises AttributeError
    # before any data is touched. Route the null-on-error path through
    # primitives that exist in both 3.5 and 4.0:
    #   - F.try_to_timestamp  (PySpark 3.5+, returns NULL under ANSI too)
    #   - SQL try_cast        (Spark 3.5+, returns NULL under ANSI too)
    #   - F.to_date           (PySpark 3.5+; returns NULL on parse failure
    #                          under the default non-ANSI mode, which is
    #                          the Databricks default. Under ANSI mode it
    #                          raises — accepted limitation until
    #                          try_to_date lands on every supported DBR.)
    if on_error == "null":
        if kind == "timestamp":
            if fmt:
                return df.withColumn(
                    column, F.try_to_timestamp(_col(column), F.lit(fmt))
                )
            return df.withColumn(column, F.try_to_timestamp(_col(column)))
        # kind == "date"
        if fmt:
            return df.withColumn(column, F.to_date(_col(column), fmt))
        return df.withColumn(
            column, F.expr(f"try_cast({_quoted_ident(column)} as date)")
        )
    # on_error == "error": surface parse failures as Spark exceptions.
    fn = F.to_date if kind == "date" else F.to_timestamp
    if fmt:
        return df.withColumn(column, fn(_col(column), fmt))
    return df.withColumn(column, fn(_col(column)))

ACTION_DISPATCH: Dict[str, Callable] = {
    "formula": _apply_formula,
    "cast": _apply_cast,
    "replace_value": _apply_replace_value,
    "fill_null": _apply_fill_null,
    "text_case": _apply_text_case,
    "trim": _apply_trim,
    "regex_replace": _apply_regex_replace,
    "extract": _apply_extract,
    "parse_date": _apply_parse_date,
}

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs.get("data")
    actions: List[Dict[str, Any]] = config.get("actions", []) or []

    if not actions:
        return {"prepared_data": df}

    for index, action in enumerate(actions):
        if not isinstance(action, dict):
            raise ValueError(
                f"actions[{index}]: expected an object, got {type(action).__name__}"
            )
        if action.get("enabled", True) is False:
            continue
        action_type = action.get("type", "")
        fn = ACTION_DISPATCH.get(action_type)
        if fn is None:
            raise ValueError(
                f"actions[{index}]: unsupported action type {action_type!r}. "
                f"Choose one of: {sorted(ACTION_DISPATCH.keys())}"
            )
        df = fn(df, action)
    return {"prepared_data": df}

# generated from the system
if "ld_display_outputs" not in globals():
    try:
        _ld_param = dbutils.widgets.getAll().get("ld_display_outputs")
        if _ld_param is not None and str(_ld_param).strip() != "":
            globals()["ld_display_outputs"] = str(_ld_param).strip().lower() not in ("false", "0", "no", "off")
        else:
            try:
                from dbruntime.databricks_repl_context import get_context
                globals()["ld_display_outputs"] = not get_context().isInJob
            except Exception:
                globals()["ld_display_outputs"] = True
    except Exception:
        globals()["ld_display_outputs"] = False
if "ld_display_outputs_for" not in globals():
    try:
        _ld_for = dbutils.widgets.getAll().get("ld_display_outputs_for")
        globals()["ld_display_outputs_for"] = frozenset(_p.strip() for _p in str(_ld_for).split(",") if _p.strip()) if _ld_for is not None else frozenset()
    except Exception:
        globals()["ld_display_outputs_for"] = frozenset()
ctx = globals().setdefault("ctx", {})
config = {
    "actions": [
        {
            "type": "formula",
            "target": "api_failure_rate_latest_14_days",
            "expression": "CASE WHEN api_calls_latest_14_days = 0 THEN 0 ELSE failed_api_calls_latest_14_days / api_calls_latest_14_days END"
        },
        {
            "type": "formula",
            "target": "active_user_growth_rate",
            "expression": "CASE WHEN avg_active_users_previous_14_days = 0 THEN 0 ELSE (avg_active_users_latest_14_days - avg_active_users_previous_14_days) / avg_active_users_previous_14_days END"
        }
    ]
}
config["meta_state"] = {"is_preview": False, "is_focused_preview": False, "is_disabled": False}
inputs = {
    "data": ctx["aggregate_14b39d7f.aggregated_data"]
}
if config["meta_state"]["is_disabled"]:
    def _lb_limit_zero(value):
        if hasattr(value, "limit"):
            return value.limit(0)
        if isinstance(value, list):
            return [_lb_limit_zero(item) for item in value]
        return value
    inputs = {k: _lb_limit_zero(v) for k, v in inputs.items()}
out = run(config, inputs, spark)
if config["meta_state"]["is_disabled"]:
    out = {k: _lb_limit_zero(v) for k, v in out.items()}
ctx["prepare_e96b3d85.prepared_data"] = out["prepared_data"]
if globals().get("ld_display_outputs", False) or "prepare_e96b3d85" in globals().get("ld_display_outputs_for", frozenset()):
    display(ctx["prepare_e96b3d85.prepared_data"])

In [0]:
"""
id: join_usage_trend_b9dc6cd8
template: join
templateVersion: 3.0.0
name: join_usage_trend
position:
  x: 2100
  y: 0
description:
  text: Join two datasets on account_id, keeping all records from the left and matching ones from the right.
  hash: 4e9c2fd7
config:
  join_type: left
  match_case: false
  multi_table: false
  conditions:
    - left_input: 0
      right_input: 1
      join_keys:
        - left: account_id
          right: account_id
      join_conditions: ""
  columns:
    - edits: []
      ordered: []
    - edits: []
      ordered: []
input:
  - node: join_pipeline_health_09aadfe6
    input_port: left
    output_port: joined_data
  - node: prep_usage_trend_metrics_e96b3d85
    input_port: right
    output_port: prepared_data
"""

# generated from the system
from typing import Any, Dict, List, Optional
import pyspark.sql.functions as F
from pyspark.sql.types import StringType

def _is_checked(item: Dict[str, Any]) -> bool:
    return item.get("checked", True) is not False

def _qualified_col(alias: str, name: str):
    escaped = name.replace("`", "``")
    return F.col("`" + alias + "`.`" + escaped + "`")

def _ordered_names(df_cols: List[str], cfg: Dict[str, Any]) -> List[str]:
    ordered: List[str] = cfg.get("ordered") or []
    col_set = set(df_cols)
    placed = set()
    result: List[str] = []
    for name in ordered:
        if name in placed or name not in col_set:
            continue
        placed.add(name)
        result.append(name)
    for name in df_cols:
        if name in placed:
            continue
        placed.add(name)
        result.append(name)
    return result

def _edits_by_column(cfg: Dict[str, Any]) -> Dict[str, Dict[str, Any]]:
    by_col: Dict[str, Dict[str, Any]] = {}
    for item in cfg.get("edits", []) or []:
        col = item.get("column")
        if col is not None and col not in by_col:
            by_col[col] = item
    return by_col

def _projection(frames: List[Any]):
    out = []
    seen: Dict[str, int] = {}
    n = len(frames)
    for idx, (alias, df, cfg) in enumerate(frames):
        edits = _edits_by_column(cfg)
        for name in _ordered_names(list(df.columns), cfg):
            edit = edits.get(name)
            if edit is not None and not _is_checked(edit):
                continue
            col = _qualified_col(alias, name)
            user_alias = edit.get("alias") if edit else None
            if user_alias:
                out.append(col.alias(user_alias))
                seen[user_alias.lower()] = seen.get(user_alias.lower(), 0) + 1
                continue
            key = name.lower()
            if key not in seen:
                seen[key] = 1
                out.append(col.alias(name))
                continue
            new_name = ("right_" + name) if (n == 2 and idx == 1) else (name + "_" + str(idx + 1))
            while new_name.lower() in seen:
                new_name = new_name + "_dup"
            seen[new_name.lower()] = 1
            out.append(col.alias(new_name))
    return out

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    dfs: List[Any] = []
    for port in ("left", "right"):
        df = inputs.get(port)
        if df is not None:
            dfs.append(df)
    dfs.extend(inputs.get("data") or [])
    if len(dfs) < 2:
        raise ValueError(
            f"Join requires at least 2 inputs, got {len(dfs)}. "
            "Wire 2 or more upstream operators into this Join."
        )

    n = len(dfs)
    join_type = config.get("join_type") or "split_join"
    match_case = bool(config.get("match_case", False))
    conditions: List[Dict[str, Any]] = config.get("conditions") or []
    columns: List[Dict[str, Any]] = config.get("columns") or []

    aliased = [df.alias("t" + str(i)) for i, df in enumerate(dfs)]
    types = [{f.name.lower(): f.dataType for f in df.schema} for df in aliased]

    def key_col(idx: int, name: str):
        col = _qualified_col("t" + str(idx), name)
        if not match_case and isinstance(types[idx].get(name.lower()), StringType):
            return F.upper(F.trim(col))
        return col

    def edge_predicate(i: int, j: int, edge: Dict[str, Any]) -> Optional[Any]:
        predicates = []
        for key in edge.get("join_keys") or []:
            predicates.append(key_col(i, key["left"]) == key_col(j, key["right"]))
        condition = edge.get("join_conditions") or ""
        if condition:
            predicates.append(F.expr(condition))
        join_expr = None
        for predicate in predicates:
            join_expr = predicate if join_expr is None else join_expr & predicate
        return join_expr

    edges: List[Any] = [
        (int(edge.get("left_input", 0)), int(edge.get("right_input", 0)), edge)
        for edge in conditions
    ]

    predicates_by_step: Dict[int, List[Any]] = {}
    for i, j, edge in edges:
        predicate = edge_predicate(i, j, edge)
        if predicate is not None:
            predicates_by_step.setdefault(max(i, j), []).append(predicate)

    def predicate_at(step: int) -> Optional[Any]:
        join_expr = None
        for predicate in predicates_by_step.get(step, []):
            join_expr = predicate if join_expr is None else join_expr & predicate
        return join_expr

    def cfg(i: int) -> Dict[str, Any]:
        return columns[i] if i < len(columns) else {}

    is_two = n == 2
    is_split = is_two and join_type == "split_join"

    if is_split:
        join_expr = predicate_at(1)
        left_df, right_df = aliased[0], aliased[1]
        if join_expr is None:
            matched = left_df.join(right_df, how="inner")
            left_unmatched = left_df.join(right_df, how="left_anti")
            right_unmatched = right_df.join(left_df, how="left_anti")
        else:
            matched = left_df.join(right_df, join_expr, how="inner")
            left_unmatched = left_df.join(right_df, join_expr, how="left_anti")
            right_unmatched = right_df.join(left_df, join_expr, how="left_anti")
        projection = _projection([("t0", left_df, cfg(0)), ("t1", right_df, cfg(1))])
        if projection:
            matched = matched.select(*projection)
        return {
            "joined_data": matched,
            "left_unmatched": left_unmatched,
            "right_unmatched": right_unmatched,
        }

    how = join_type if is_two else "inner"
    if is_two:
        acc = aliased[0]
        predicate = predicate_at(1)
        if predicate is None:
            acc = acc.join(aliased[1], how=how)
        else:
            acc = acc.join(aliased[1], predicate, how=how)
    else:
        adjacency: Dict[int, List[int]] = {}
        for idx, (i, j, _edge) in enumerate(edges):
            adjacency.setdefault(i, []).append(idx)
            adjacency.setdefault(j, []).append(idx)
        visited = [False] * n
        visited[0] = True
        acc = aliased[0]
        used_edges = set()
        queue: List[int] = [0]
        while queue:
            cur = queue.pop(0)
            for idx in adjacency.get(cur, []):
                i, j, edge = edges[idx]
                nxt = j if i == cur else i
                if visited[nxt]:
                    continue
                visited[nxt] = True
                used_edges.add(idx)
                predicate = edge_predicate(i, j, edge)
                if predicate is None:
                    acc = acc.join(aliased[nxt], how=how)
                else:
                    acc = acc.join(aliased[nxt], predicate, how=how)
                queue.append(nxt)
        for idx, (i, j, edge) in enumerate(edges):
            if idx in used_edges:
                continue
            predicate = edge_predicate(i, j, edge)
            if predicate is not None:
                acc = acc.where(predicate)

    projection = _projection([("t" + str(i), aliased[i], cfg(i)) for i in range(n)])
    if projection:
        acc = acc.select(*projection)

    return {
        "joined_data": acc,
        "left_unmatched": spark.createDataFrame([], aliased[0].schema),
        "right_unmatched": spark.createDataFrame([], aliased[n - 1].schema),
    }

# generated from the system
if "ld_display_outputs" not in globals():
    try:
        _ld_param = dbutils.widgets.getAll().get("ld_display_outputs")
        if _ld_param is not None and str(_ld_param).strip() != "":
            globals()["ld_display_outputs"] = str(_ld_param).strip().lower() not in ("false", "0", "no", "off")
        else:
            try:
                from dbruntime.databricks_repl_context import get_context
                globals()["ld_display_outputs"] = not get_context().isInJob
            except Exception:
                globals()["ld_display_outputs"] = True
    except Exception:
        globals()["ld_display_outputs"] = False
if "ld_display_outputs_for" not in globals():
    try:
        _ld_for = dbutils.widgets.getAll().get("ld_display_outputs_for")
        globals()["ld_display_outputs_for"] = frozenset(_p.strip() for _p in str(_ld_for).split(",") if _p.strip()) if _ld_for is not None else frozenset()
    except Exception:
        globals()["ld_display_outputs_for"] = frozenset()
ctx = globals().setdefault("ctx", {})
config = {
    "join_type": "left",
    "match_case": False,
    "multi_table": False,
    "conditions": [
        {
            "left_input": 0,
            "right_input": 1,
            "join_keys": [
                {
                    "left": "account_id",
                    "right": "account_id"
                }
            ],
            "join_conditions": ""
        }
    ],
    "columns": [
        {
            "edits": [],
            "ordered": []
        },
        {
            "edits": [],
            "ordered": []
        }
    ]
}
config["meta_state"] = {"is_preview": False, "is_focused_preview": False, "is_disabled": False}
inputs = {
    "left": ctx["join_pipeline_health_09aadfe6.joined_data"],
    "right": ctx["prep_usage_trend_metrics_e96b3d85.prepared_data"]
}
if config["meta_state"]["is_disabled"]:
    def _lb_limit_zero(value):
        if hasattr(value, "limit"):
            return value.limit(0)
        if isinstance(value, list):
            return [_lb_limit_zero(item) for item in value]
        return value
    inputs = {k: _lb_limit_zero(v) for k, v in inputs.items()}
out = run(config, inputs, spark)
if config["meta_state"]["is_disabled"]:
    out = {k: _lb_limit_zero(v) for k, v in out.items()}
ctx["join_usage_trend_b9dc6cd8.joined_data"] = out["joined_data"]
ctx["join_usage_trend_b9dc6cd8.left_unmatched"] = out["left_unmatched"]
ctx["join_usage_trend_b9dc6cd8.right_unmatched"] = out["right_unmatched"]
if globals().get("ld_display_outputs", False) or "join_usage_trend_b9dc6cd8" in globals().get("ld_display_outputs_for", frozenset()):
    display(ctx["join_usage_trend_b9dc6cd8.joined_data"])
    display(ctx["join_usage_trend_b9dc6cd8.left_unmatched"])
    display(ctx["join_usage_trend_b9dc6cd8.right_unmatched"])

In [0]:
"""
id: join_decision_makers_7abc7d3a
template: join
templateVersion: 3.0.0
name: join_decision_makers
position:
  x: 2400
  y: 475
description:
  text: Perform a left join on account_id, keeping all records from the left side and matching records from the right.
  hash: 8d4c4c09
previewCodeHash: 7eda1bc18fa9d40c
config:
  join_type: left
  match_case: false
  multi_table: false
  conditions:
    - left_input: 0
      right_input: 1
      join_keys:
        - left: account_id
          right: account_id
      join_conditions: ""
  columns:
    - edits: []
      ordered: []
    - edits: []
      ordered: []
input:
  - node: join_usage_trend_b9dc6cd8
    input_port: left
    output_port: joined_data
  - node: aggregate_97c95dc7
    input_port: right
    output_port: aggregated_data
"""

# generated from the system
from typing import Any, Dict, List, Optional
import pyspark.sql.functions as F
from pyspark.sql.types import StringType

def _is_checked(item: Dict[str, Any]) -> bool:
    return item.get("checked", True) is not False

def _qualified_col(alias: str, name: str):
    escaped = name.replace("`", "``")
    return F.col("`" + alias + "`.`" + escaped + "`")

def _ordered_names(df_cols: List[str], cfg: Dict[str, Any]) -> List[str]:
    ordered: List[str] = cfg.get("ordered") or []
    col_set = set(df_cols)
    placed = set()
    result: List[str] = []
    for name in ordered:
        if name in placed or name not in col_set:
            continue
        placed.add(name)
        result.append(name)
    for name in df_cols:
        if name in placed:
            continue
        placed.add(name)
        result.append(name)
    return result

def _edits_by_column(cfg: Dict[str, Any]) -> Dict[str, Dict[str, Any]]:
    by_col: Dict[str, Dict[str, Any]] = {}
    for item in cfg.get("edits", []) or []:
        col = item.get("column")
        if col is not None and col not in by_col:
            by_col[col] = item
    return by_col

def _projection(frames: List[Any]):
    out = []
    seen: Dict[str, int] = {}
    n = len(frames)
    for idx, (alias, df, cfg) in enumerate(frames):
        edits = _edits_by_column(cfg)
        for name in _ordered_names(list(df.columns), cfg):
            edit = edits.get(name)
            if edit is not None and not _is_checked(edit):
                continue
            col = _qualified_col(alias, name)
            user_alias = edit.get("alias") if edit else None
            if user_alias:
                out.append(col.alias(user_alias))
                seen[user_alias.lower()] = seen.get(user_alias.lower(), 0) + 1
                continue
            key = name.lower()
            if key not in seen:
                seen[key] = 1
                out.append(col.alias(name))
                continue
            new_name = ("right_" + name) if (n == 2 and idx == 1) else (name + "_" + str(idx + 1))
            while new_name.lower() in seen:
                new_name = new_name + "_dup"
            seen[new_name.lower()] = 1
            out.append(col.alias(new_name))
    return out

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    dfs: List[Any] = []
    for port in ("left", "right"):
        df = inputs.get(port)
        if df is not None:
            dfs.append(df)
    dfs.extend(inputs.get("data") or [])
    if len(dfs) < 2:
        raise ValueError(
            f"Join requires at least 2 inputs, got {len(dfs)}. "
            "Wire 2 or more upstream operators into this Join."
        )

    n = len(dfs)
    join_type = config.get("join_type") or "split_join"
    match_case = bool(config.get("match_case", False))
    conditions: List[Dict[str, Any]] = config.get("conditions") or []
    columns: List[Dict[str, Any]] = config.get("columns") or []

    aliased = [df.alias("t" + str(i)) for i, df in enumerate(dfs)]
    types = [{f.name.lower(): f.dataType for f in df.schema} for df in aliased]

    def key_col(idx: int, name: str):
        col = _qualified_col("t" + str(idx), name)
        if not match_case and isinstance(types[idx].get(name.lower()), StringType):
            return F.upper(F.trim(col))
        return col

    def edge_predicate(i: int, j: int, edge: Dict[str, Any]) -> Optional[Any]:
        predicates = []
        for key in edge.get("join_keys") or []:
            predicates.append(key_col(i, key["left"]) == key_col(j, key["right"]))
        condition = edge.get("join_conditions") or ""
        if condition:
            predicates.append(F.expr(condition))
        join_expr = None
        for predicate in predicates:
            join_expr = predicate if join_expr is None else join_expr & predicate
        return join_expr

    edges: List[Any] = [
        (int(edge.get("left_input", 0)), int(edge.get("right_input", 0)), edge)
        for edge in conditions
    ]

    predicates_by_step: Dict[int, List[Any]] = {}
    for i, j, edge in edges:
        predicate = edge_predicate(i, j, edge)
        if predicate is not None:
            predicates_by_step.setdefault(max(i, j), []).append(predicate)

    def predicate_at(step: int) -> Optional[Any]:
        join_expr = None
        for predicate in predicates_by_step.get(step, []):
            join_expr = predicate if join_expr is None else join_expr & predicate
        return join_expr

    def cfg(i: int) -> Dict[str, Any]:
        return columns[i] if i < len(columns) else {}

    is_two = n == 2
    is_split = is_two and join_type == "split_join"

    if is_split:
        join_expr = predicate_at(1)
        left_df, right_df = aliased[0], aliased[1]
        if join_expr is None:
            matched = left_df.join(right_df, how="inner")
            left_unmatched = left_df.join(right_df, how="left_anti")
            right_unmatched = right_df.join(left_df, how="left_anti")
        else:
            matched = left_df.join(right_df, join_expr, how="inner")
            left_unmatched = left_df.join(right_df, join_expr, how="left_anti")
            right_unmatched = right_df.join(left_df, join_expr, how="left_anti")
        projection = _projection([("t0", left_df, cfg(0)), ("t1", right_df, cfg(1))])
        if projection:
            matched = matched.select(*projection)
        return {
            "joined_data": matched,
            "left_unmatched": left_unmatched,
            "right_unmatched": right_unmatched,
        }

    how = join_type if is_two else "inner"
    if is_two:
        acc = aliased[0]
        predicate = predicate_at(1)
        if predicate is None:
            acc = acc.join(aliased[1], how=how)
        else:
            acc = acc.join(aliased[1], predicate, how=how)
    else:
        adjacency: Dict[int, List[int]] = {}
        for idx, (i, j, _edge) in enumerate(edges):
            adjacency.setdefault(i, []).append(idx)
            adjacency.setdefault(j, []).append(idx)
        visited = [False] * n
        visited[0] = True
        acc = aliased[0]
        used_edges = set()
        queue: List[int] = [0]
        while queue:
            cur = queue.pop(0)
            for idx in adjacency.get(cur, []):
                i, j, edge = edges[idx]
                nxt = j if i == cur else i
                if visited[nxt]:
                    continue
                visited[nxt] = True
                used_edges.add(idx)
                predicate = edge_predicate(i, j, edge)
                if predicate is None:
                    acc = acc.join(aliased[nxt], how=how)
                else:
                    acc = acc.join(aliased[nxt], predicate, how=how)
                queue.append(nxt)
        for idx, (i, j, edge) in enumerate(edges):
            if idx in used_edges:
                continue
            predicate = edge_predicate(i, j, edge)
            if predicate is not None:
                acc = acc.where(predicate)

    projection = _projection([("t" + str(i), aliased[i], cfg(i)) for i in range(n)])
    if projection:
        acc = acc.select(*projection)

    return {
        "joined_data": acc,
        "left_unmatched": spark.createDataFrame([], aliased[0].schema),
        "right_unmatched": spark.createDataFrame([], aliased[n - 1].schema),
    }

# generated from the system
if "ld_display_outputs" not in globals():
    try:
        _ld_param = dbutils.widgets.getAll().get("ld_display_outputs")
        if _ld_param is not None and str(_ld_param).strip() != "":
            globals()["ld_display_outputs"] = str(_ld_param).strip().lower() not in ("false", "0", "no", "off")
        else:
            try:
                from dbruntime.databricks_repl_context import get_context
                globals()["ld_display_outputs"] = not get_context().isInJob
            except Exception:
                globals()["ld_display_outputs"] = True
    except Exception:
        globals()["ld_display_outputs"] = False
if "ld_display_outputs_for" not in globals():
    try:
        _ld_for = dbutils.widgets.getAll().get("ld_display_outputs_for")
        globals()["ld_display_outputs_for"] = frozenset(_p.strip() for _p in str(_ld_for).split(",") if _p.strip()) if _ld_for is not None else frozenset()
    except Exception:
        globals()["ld_display_outputs_for"] = frozenset()
ctx = globals().setdefault("ctx", {})
config = {
    "join_type": "left",
    "match_case": False,
    "multi_table": False,
    "conditions": [
        {
            "left_input": 0,
            "right_input": 1,
            "join_keys": [
                {
                    "left": "account_id",
                    "right": "account_id"
                }
            ],
            "join_conditions": ""
        }
    ],
    "columns": [
        {
            "edits": [],
            "ordered": []
        },
        {
            "edits": [],
            "ordered": []
        }
    ]
}
config["meta_state"] = {"is_preview": False, "is_focused_preview": False, "is_disabled": False}
inputs = {
    "left": ctx["join_usage_trend_b9dc6cd8.joined_data"],
    "right": ctx["aggregate_97c95dc7.aggregated_data"]
}
if config["meta_state"]["is_disabled"]:
    def _lb_limit_zero(value):
        if hasattr(value, "limit"):
            return value.limit(0)
        if isinstance(value, list):
            return [_lb_limit_zero(item) for item in value]
        return value
    inputs = {k: _lb_limit_zero(v) for k, v in inputs.items()}
out = run(config, inputs, spark)
if config["meta_state"]["is_disabled"]:
    out = {k: _lb_limit_zero(v) for k, v in out.items()}
ctx["join_decision_makers_7abc7d3a.joined_data"] = out["joined_data"]
ctx["join_decision_makers_7abc7d3a.left_unmatched"] = out["left_unmatched"]
ctx["join_decision_makers_7abc7d3a.right_unmatched"] = out["right_unmatched"]
if globals().get("ld_display_outputs", False) or "join_decision_makers_7abc7d3a" in globals().get("ld_display_outputs_for", frozenset()):
    display(ctx["join_decision_makers_7abc7d3a.joined_data"])
    display(ctx["join_decision_makers_7abc7d3a.left_unmatched"])
    display(ctx["join_decision_makers_7abc7d3a.right_unmatched"])

In [0]:
"""
id: prepare_account_risk_features_a8b748f0
template: prepare
templateVersion: 1.0.0
name: prep_account_risk_features
position:
  x: 2700
  y: 475
description:
  text: Fill missing values in key columns with 0 or false.
  hash: d364bbd0
previewCodeHash: 33fdf0064b2acaf8
config:
  actions:
    - type: fill_null
      column: open_case_count
      with: "0"
    - type: fill_null
      column: open_high_priority_case_count
      with: "0"
    - type: fill_null
      column: open_sla_breach_count
      with: "0"
    - type: fill_null
      column: max_open_case_age_days
      with: "0"
    - type: fill_null
      column: renewal_pipeline_amount
      with: "0"
    - type: fill_null
      column: stalled_renewal_opportunity_count
      with: "0"
    - type: fill_null
      column: active_expansion_opportunity_count
      with: "0"
    - type: fill_null
      column: avg_active_users_latest_14_days
      with: "0"
    - type: fill_null
      column: avg_active_users_previous_14_days
      with: "0"
    - type: fill_null
      column: api_calls_latest_14_days
      with: "0"
    - type: fill_null
      column: failed_api_calls_latest_14_days
      with: "0"
    - type: fill_null
      column: key_feature_events_latest_14_days
      with: "0"
    - type: fill_null
      column: contact_count
      with: "0"
    - type: fill_null
      column: decision_maker_contact_count
      with: "0"
    - type: fill_null
      column: has_decision_maker
      with: "false"
input:
  - node: join_decision_makers_7abc7d3a
    input_port: data
    output_port: joined_data
"""

# generated from the system
from typing import Any, Callable, Dict, List
import pyspark.sql.functions as F

TEXT_CASE_FUNCTIONS: Dict[str, Callable] = {
    "lower": F.lower,
    "upper": F.upper,
    "title": F.initcap,
}

TRIM_FUNCTIONS: Dict[str, Callable] = {
    "both": F.trim,
    "left": F.ltrim,
    "right": F.rtrim,
}

VALID_MATCH_MODES = {"exact", "contains", "prefix", "suffix", "regex"}

def _require_column(df, column: str, action_type: str) -> None:
    if not column:
        raise ValueError(f"{action_type}: 'column' is required")
    if column not in df.columns:
        raise ValueError(
            f"{action_type}: column '{column}' not found in input data. "
            f"Available columns: {df.columns}"
        )

def _column_dtype(df, column: str) -> str:
    return df.schema[column].dataType.simpleString()

def _quoted_ident(name: str) -> str:
    return "`" + name.replace("`", "``") + "`"

def _col(name: str):
    """Reference a column by exact name.

    F.col parses its argument as a column expression, so a name containing
    a '.' (e.g. "a.b") is otherwise read as struct-field access and fails
    even though the column exists. Backtick-quoting forces an exact-name
    lookup; quoting plain names is harmless.
    """
    return F.col(_quoted_ident(name))

def _coerced_lit(value: Any, target_dtype: str):
    """Wrap a user-provided value as a literal cast to the target column's type.

    UI always feeds text per the operator spec; this is where the
    type coercion happens so the per-action UI stays simple.
    """
    return F.lit(value).cast(target_dtype)

def _apply_formula(df, action: Dict[str, Any]):
    target = action.get("target", "")
    expression = action.get("expression", "")
    if not target:
        raise ValueError("formula: 'target' is required")
    if not expression:
        raise ValueError("formula: 'expression' is required")
    return df.withColumn(target, F.expr(expression))

def _apply_cast(df, action: Dict[str, Any]):
    column = action.get("column", "")
    to_type = action.get("to", "")
    on_error = action.get("on_error", "null")
    _require_column(df, column, "cast")
    if not to_type:
        raise ValueError("cast: 'to' (target type) is required")
    if on_error == "null":
        return df.withColumn(
            column,
            F.expr(f"try_cast({_quoted_ident(column)} as {to_type})"),
        )
    return df.withColumn(column, _col(column).cast(to_type))

def _apply_replace_value(df, action: Dict[str, Any]):
    column = action.get("column", "")
    match_mode = action.get("match_mode", "exact")
    match = action.get("match", "")
    with_val = action.get("with", "")
    case_sensitive = bool(action.get("case_sensitive", False))
    _require_column(df, column, "replace_value")
    if match_mode not in VALID_MATCH_MODES:
        raise ValueError(
            f"replace_value: unsupported match_mode '{match_mode}'. "
            f"Choose one of: {sorted(VALID_MATCH_MODES)}"
        )
    dtype = _column_dtype(df, column)
    col_as_string = _col(column).cast("string")
    match_lit = F.lit(match)
    if case_sensitive:
        cmp_col = col_as_string
        cmp_lit = match_lit
    else:
        cmp_col = F.lower(col_as_string)
        cmp_lit = F.lower(match_lit)

    if match_mode == "exact":
        predicate = cmp_col == cmp_lit
    elif match_mode == "contains":
        predicate = cmp_col.contains(cmp_lit)
    elif match_mode == "prefix":
        predicate = cmp_col.startswith(cmp_lit)
    elif match_mode == "suffix":
        predicate = cmp_col.endswith(cmp_lit)
    else:  # regex
        pattern = match if case_sensitive else f"(?i){match}"
        predicate = col_as_string.rlike(pattern)

    replacement = _coerced_lit(with_val, dtype)
    return df.withColumn(
        column,
        F.when(predicate, replacement).otherwise(_col(column)),
    )

def _apply_fill_null(df, action: Dict[str, Any]):
    column = action.get("column", "")
    with_val = action.get("with", "")
    _require_column(df, column, "fill_null")
    dtype = _column_dtype(df, column)
    replacement = _coerced_lit(with_val, dtype)
    return df.withColumn(
        column,
        F.when(_col(column).isNull(), replacement).otherwise(_col(column)),
    )

def _apply_text_case(df, action: Dict[str, Any]):
    column = action.get("column", "")
    case = action.get("case", "")
    _require_column(df, column, "text_case")
    fn = TEXT_CASE_FUNCTIONS.get(case)
    if fn is None:
        raise ValueError(
            f"text_case: unsupported case '{case}'. "
            f"Choose one of: {sorted(TEXT_CASE_FUNCTIONS.keys())}"
        )
    return df.withColumn(column, fn(_col(column)))

def _apply_trim(df, action: Dict[str, Any]):
    column = action.get("column", "")
    side = action.get("side", "both")
    _require_column(df, column, "trim")
    fn = TRIM_FUNCTIONS.get(side)
    if fn is None:
        raise ValueError(
            f"trim: unsupported side '{side}'. "
            f"Choose one of: {sorted(TRIM_FUNCTIONS.keys())}"
        )
    return df.withColumn(column, fn(_col(column)))

def _apply_regex_replace(df, action: Dict[str, Any]):
    column = action.get("column", "")
    pattern = action.get("pattern", "")
    replacement = action.get("replacement", "")
    _require_column(df, column, "regex_replace")
    return df.withColumn(
        column,
        F.regexp_replace(_col(column), pattern, replacement),
    )

def _apply_extract(df, action: Dict[str, Any]):
    column = action.get("column", "")
    pattern = action.get("pattern", "")
    target = action.get("target") or column
    group_raw = action.get("group", 0)
    _require_column(df, column, "extract")
    try:
        group_idx = int(group_raw)
    except (TypeError, ValueError) as exc:
        raise ValueError(
            f"extract: 'group' must be an integer, got {group_raw!r}"
        ) from exc
    return df.withColumn(
        target,
        F.regexp_extract(_col(column), pattern, group_idx),
    )

def _apply_parse_date(df, action: Dict[str, Any]):
    column = action.get("column", "")
    kind = action.get("kind", "date")
    fmt = action.get("format") or None
    on_error = action.get("on_error", "null")
    _require_column(df, column, "parse_date")
    if kind not in ("date", "timestamp"):
        raise ValueError(
            f"parse_date: unsupported kind '{kind}'. Choose date or timestamp."
        )
    # PySpark exposes F.try_to_timestamp from Spark 3.5 but only adds
    # F.try_to_date in Spark 4.0. Current Databricks Runtimes still ship
    # Spark 3.5, where referencing F.try_to_date raises AttributeError
    # before any data is touched. Route the null-on-error path through
    # primitives that exist in both 3.5 and 4.0:
    #   - F.try_to_timestamp  (PySpark 3.5+, returns NULL under ANSI too)
    #   - SQL try_cast        (Spark 3.5+, returns NULL under ANSI too)
    #   - F.to_date           (PySpark 3.5+; returns NULL on parse failure
    #                          under the default non-ANSI mode, which is
    #                          the Databricks default. Under ANSI mode it
    #                          raises — accepted limitation until
    #                          try_to_date lands on every supported DBR.)
    if on_error == "null":
        if kind == "timestamp":
            if fmt:
                return df.withColumn(
                    column, F.try_to_timestamp(_col(column), F.lit(fmt))
                )
            return df.withColumn(column, F.try_to_timestamp(_col(column)))
        # kind == "date"
        if fmt:
            return df.withColumn(column, F.to_date(_col(column), fmt))
        return df.withColumn(
            column, F.expr(f"try_cast({_quoted_ident(column)} as date)")
        )
    # on_error == "error": surface parse failures as Spark exceptions.
    fn = F.to_date if kind == "date" else F.to_timestamp
    if fmt:
        return df.withColumn(column, fn(_col(column), fmt))
    return df.withColumn(column, fn(_col(column)))

ACTION_DISPATCH: Dict[str, Callable] = {
    "formula": _apply_formula,
    "cast": _apply_cast,
    "replace_value": _apply_replace_value,
    "fill_null": _apply_fill_null,
    "text_case": _apply_text_case,
    "trim": _apply_trim,
    "regex_replace": _apply_regex_replace,
    "extract": _apply_extract,
    "parse_date": _apply_parse_date,
}

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs.get("data")
    actions: List[Dict[str, Any]] = config.get("actions", []) or []

    if not actions:
        return {"prepared_data": df}

    for index, action in enumerate(actions):
        if not isinstance(action, dict):
            raise ValueError(
                f"actions[{index}]: expected an object, got {type(action).__name__}"
            )
        if action.get("enabled", True) is False:
            continue
        action_type = action.get("type", "")
        fn = ACTION_DISPATCH.get(action_type)
        if fn is None:
            raise ValueError(
                f"actions[{index}]: unsupported action type {action_type!r}. "
                f"Choose one of: {sorted(ACTION_DISPATCH.keys())}"
            )
        df = fn(df, action)
    return {"prepared_data": df}

# generated from the system
if "ld_display_outputs" not in globals():
    try:
        _ld_param = dbutils.widgets.getAll().get("ld_display_outputs")
        if _ld_param is not None and str(_ld_param).strip() != "":
            globals()["ld_display_outputs"] = str(_ld_param).strip().lower() not in ("false", "0", "no", "off")
        else:
            try:
                from dbruntime.databricks_repl_context import get_context
                globals()["ld_display_outputs"] = not get_context().isInJob
            except Exception:
                globals()["ld_display_outputs"] = True
    except Exception:
        globals()["ld_display_outputs"] = False
if "ld_display_outputs_for" not in globals():
    try:
        _ld_for = dbutils.widgets.getAll().get("ld_display_outputs_for")
        globals()["ld_display_outputs_for"] = frozenset(_p.strip() for _p in str(_ld_for).split(",") if _p.strip()) if _ld_for is not None else frozenset()
    except Exception:
        globals()["ld_display_outputs_for"] = frozenset()
ctx = globals().setdefault("ctx", {})
config = {
    "actions": [
        {
            "type": "fill_null",
            "column": "open_case_count",
            "with": "0"
        },
        {
            "type": "fill_null",
            "column": "open_high_priority_case_count",
            "with": "0"
        },
        {
            "type": "fill_null",
            "column": "open_sla_breach_count",
            "with": "0"
        },
        {
            "type": "fill_null",
            "column": "max_open_case_age_days",
            "with": "0"
        },
        {
            "type": "fill_null",
            "column": "renewal_pipeline_amount",
            "with": "0"
        },
        {
            "type": "fill_null",
            "column": "stalled_renewal_opportunity_count",
            "with": "0"
        },
        {
            "type": "fill_null",
            "column": "active_expansion_opportunity_count",
            "with": "0"
        },
        {
            "type": "fill_null",
            "column": "avg_active_users_latest_14_days",
            "with": "0"
        },
        {
            "type": "fill_null",
            "column": "avg_active_users_previous_14_days",
            "with": "0"
        },
        {
            "type": "fill_null",
            "column": "api_calls_latest_14_days",
            "with": "0"
        },
        {
            "type": "fill_null",
            "column": "failed_api_calls_latest_14_days",
            "with": "0"
        },
        {
            "type": "fill_null",
            "column": "key_feature_events_latest_14_days",
            "with": "0"
        },
        {
            "type": "fill_null",
            "column": "contact_count",
            "with": "0"
        },
        {
            "type": "fill_null",
            "column": "decision_maker_contact_count",
            "with": "0"
        },
        {
            "type": "fill_null",
            "column": "has_decision_maker",
            "with": "false"
        }
    ]
}
config["meta_state"] = {"is_preview": False, "is_focused_preview": False, "is_disabled": False}
inputs = {
    "data": ctx["join_decision_makers_7abc7d3a.joined_data"]
}
if config["meta_state"]["is_disabled"]:
    def _lb_limit_zero(value):
        if hasattr(value, "limit"):
            return value.limit(0)
        if isinstance(value, list):
            return [_lb_limit_zero(item) for item in value]
        return value
    inputs = {k: _lb_limit_zero(v) for k, v in inputs.items()}
out = run(config, inputs, spark)
if config["meta_state"]["is_disabled"]:
    out = {k: _lb_limit_zero(v) for k, v in out.items()}
ctx["prepare_account_risk_features_a8b748f0.prepared_data"] = out["prepared_data"]
if globals().get("ld_display_outputs", False) or "prepare_account_risk_features_a8b748f0" in globals().get("ld_display_outputs_for", frozenset()):
    display(ctx["prepare_account_risk_features_a8b748f0.prepared_data"])

In [0]:
"""
id: prepare_renewal_risk_bbc08ae2
template: prepare
templateVersion: 1.0.0
name: prep_renewal_risk
position:
  x: 3000
  y: 475
description:
  text: Calculate renewal risk scores and assign risk bands and recommended actions based on account features.
  hash: 1ddd0729
previewCodeHash: 85733f17a2ca6437
config:
  actions:
    - type: formula
      target: days_to_renewal
      expression: DATEDIFF(renewal_date, current_date())
    - type: formula
      target: decision_maker_int
      expression: CAST(has_decision_maker AS INT)
    - type: formula
      target: base_risk_score
      expression: (CASE WHEN days_to_renewal BETWEEN 0 AND 90 THEN 30 ELSE 0 END) + (CASE WHEN open_high_priority_case_count > 0 THEN 25 ELSE 0 END) + (CASE WHEN open_sla_breach_count > 0 THEN 15 ELSE 0 END) + (CASE WHEN stalled_renewal_opportunity_count > 0 THEN 15 ELSE 0 END) + (CASE WHEN active_user_growth_rate <= -0.10 THEN 15 ELSE 0 END) + (CASE WHEN decision_maker_int = 0 THEN 10 ELSE 0 END) + (CASE WHEN arr >= 250000 THEN 10 ELSE 0 END) - (CASE WHEN active_expansion_opportunity_count > 0 THEN 10 ELSE 0 END)
    - type: formula
      target: risk_score
      expression: GREATEST(0, LEAST(base_risk_score, 100))
    - type: formula
      target: risk_band
      expression: CASE WHEN risk_score >= 80 THEN 'Critical' WHEN risk_score >= 60 THEN 'High' WHEN risk_score >= 40 THEN 'Medium' ELSE 'Low' END
    - type: formula
      target: recommended_action
      expression: CASE WHEN risk_band = 'Critical' THEN 'Executive escalation' WHEN risk_band = 'High' THEN 'CSM renewal save plan' WHEN risk_band = 'Medium' THEN 'Adoption and stakeholder follow-up' ELSE 'Monitor account health' END
input:
  - node: prepare_account_risk_features_a8b748f0
    input_port: data
    output_port: prepared_data
"""

# generated from the system
from typing import Any, Callable, Dict, List
import pyspark.sql.functions as F

TEXT_CASE_FUNCTIONS: Dict[str, Callable] = {
    "lower": F.lower,
    "upper": F.upper,
    "title": F.initcap,
}

TRIM_FUNCTIONS: Dict[str, Callable] = {
    "both": F.trim,
    "left": F.ltrim,
    "right": F.rtrim,
}

VALID_MATCH_MODES = {"exact", "contains", "prefix", "suffix", "regex"}

def _require_column(df, column: str, action_type: str) -> None:
    if not column:
        raise ValueError(f"{action_type}: 'column' is required")
    if column not in df.columns:
        raise ValueError(
            f"{action_type}: column '{column}' not found in input data. "
            f"Available columns: {df.columns}"
        )

def _column_dtype(df, column: str) -> str:
    return df.schema[column].dataType.simpleString()

def _quoted_ident(name: str) -> str:
    return "`" + name.replace("`", "``") + "`"

def _col(name: str):
    """Reference a column by exact name.

    F.col parses its argument as a column expression, so a name containing
    a '.' (e.g. "a.b") is otherwise read as struct-field access and fails
    even though the column exists. Backtick-quoting forces an exact-name
    lookup; quoting plain names is harmless.
    """
    return F.col(_quoted_ident(name))

def _coerced_lit(value: Any, target_dtype: str):
    """Wrap a user-provided value as a literal cast to the target column's type.

    UI always feeds text per the operator spec; this is where the
    type coercion happens so the per-action UI stays simple.
    """
    return F.lit(value).cast(target_dtype)

def _apply_formula(df, action: Dict[str, Any]):
    target = action.get("target", "")
    expression = action.get("expression", "")
    if not target:
        raise ValueError("formula: 'target' is required")
    if not expression:
        raise ValueError("formula: 'expression' is required")
    return df.withColumn(target, F.expr(expression))

def _apply_cast(df, action: Dict[str, Any]):
    column = action.get("column", "")
    to_type = action.get("to", "")
    on_error = action.get("on_error", "null")
    _require_column(df, column, "cast")
    if not to_type:
        raise ValueError("cast: 'to' (target type) is required")
    if on_error == "null":
        return df.withColumn(
            column,
            F.expr(f"try_cast({_quoted_ident(column)} as {to_type})"),
        )
    return df.withColumn(column, _col(column).cast(to_type))

def _apply_replace_value(df, action: Dict[str, Any]):
    column = action.get("column", "")
    match_mode = action.get("match_mode", "exact")
    match = action.get("match", "")
    with_val = action.get("with", "")
    case_sensitive = bool(action.get("case_sensitive", False))
    _require_column(df, column, "replace_value")
    if match_mode not in VALID_MATCH_MODES:
        raise ValueError(
            f"replace_value: unsupported match_mode '{match_mode}'. "
            f"Choose one of: {sorted(VALID_MATCH_MODES)}"
        )
    dtype = _column_dtype(df, column)
    col_as_string = _col(column).cast("string")
    match_lit = F.lit(match)
    if case_sensitive:
        cmp_col = col_as_string
        cmp_lit = match_lit
    else:
        cmp_col = F.lower(col_as_string)
        cmp_lit = F.lower(match_lit)

    if match_mode == "exact":
        predicate = cmp_col == cmp_lit
    elif match_mode == "contains":
        predicate = cmp_col.contains(cmp_lit)
    elif match_mode == "prefix":
        predicate = cmp_col.startswith(cmp_lit)
    elif match_mode == "suffix":
        predicate = cmp_col.endswith(cmp_lit)
    else:  # regex
        pattern = match if case_sensitive else f"(?i){match}"
        predicate = col_as_string.rlike(pattern)

    replacement = _coerced_lit(with_val, dtype)
    return df.withColumn(
        column,
        F.when(predicate, replacement).otherwise(_col(column)),
    )

def _apply_fill_null(df, action: Dict[str, Any]):
    column = action.get("column", "")
    with_val = action.get("with", "")
    _require_column(df, column, "fill_null")
    dtype = _column_dtype(df, column)
    replacement = _coerced_lit(with_val, dtype)
    return df.withColumn(
        column,
        F.when(_col(column).isNull(), replacement).otherwise(_col(column)),
    )

def _apply_text_case(df, action: Dict[str, Any]):
    column = action.get("column", "")
    case = action.get("case", "")
    _require_column(df, column, "text_case")
    fn = TEXT_CASE_FUNCTIONS.get(case)
    if fn is None:
        raise ValueError(
            f"text_case: unsupported case '{case}'. "
            f"Choose one of: {sorted(TEXT_CASE_FUNCTIONS.keys())}"
        )
    return df.withColumn(column, fn(_col(column)))

def _apply_trim(df, action: Dict[str, Any]):
    column = action.get("column", "")
    side = action.get("side", "both")
    _require_column(df, column, "trim")
    fn = TRIM_FUNCTIONS.get(side)
    if fn is None:
        raise ValueError(
            f"trim: unsupported side '{side}'. "
            f"Choose one of: {sorted(TRIM_FUNCTIONS.keys())}"
        )
    return df.withColumn(column, fn(_col(column)))

def _apply_regex_replace(df, action: Dict[str, Any]):
    column = action.get("column", "")
    pattern = action.get("pattern", "")
    replacement = action.get("replacement", "")
    _require_column(df, column, "regex_replace")
    return df.withColumn(
        column,
        F.regexp_replace(_col(column), pattern, replacement),
    )

def _apply_extract(df, action: Dict[str, Any]):
    column = action.get("column", "")
    pattern = action.get("pattern", "")
    target = action.get("target") or column
    group_raw = action.get("group", 0)
    _require_column(df, column, "extract")
    try:
        group_idx = int(group_raw)
    except (TypeError, ValueError) as exc:
        raise ValueError(
            f"extract: 'group' must be an integer, got {group_raw!r}"
        ) from exc
    return df.withColumn(
        target,
        F.regexp_extract(_col(column), pattern, group_idx),
    )

def _apply_parse_date(df, action: Dict[str, Any]):
    column = action.get("column", "")
    kind = action.get("kind", "date")
    fmt = action.get("format") or None
    on_error = action.get("on_error", "null")
    _require_column(df, column, "parse_date")
    if kind not in ("date", "timestamp"):
        raise ValueError(
            f"parse_date: unsupported kind '{kind}'. Choose date or timestamp."
        )
    # PySpark exposes F.try_to_timestamp from Spark 3.5 but only adds
    # F.try_to_date in Spark 4.0. Current Databricks Runtimes still ship
    # Spark 3.5, where referencing F.try_to_date raises AttributeError
    # before any data is touched. Route the null-on-error path through
    # primitives that exist in both 3.5 and 4.0:
    #   - F.try_to_timestamp  (PySpark 3.5+, returns NULL under ANSI too)
    #   - SQL try_cast        (Spark 3.5+, returns NULL under ANSI too)
    #   - F.to_date           (PySpark 3.5+; returns NULL on parse failure
    #                          under the default non-ANSI mode, which is
    #                          the Databricks default. Under ANSI mode it
    #                          raises — accepted limitation until
    #                          try_to_date lands on every supported DBR.)
    if on_error == "null":
        if kind == "timestamp":
            if fmt:
                return df.withColumn(
                    column, F.try_to_timestamp(_col(column), F.lit(fmt))
                )
            return df.withColumn(column, F.try_to_timestamp(_col(column)))
        # kind == "date"
        if fmt:
            return df.withColumn(column, F.to_date(_col(column), fmt))
        return df.withColumn(
            column, F.expr(f"try_cast({_quoted_ident(column)} as date)")
        )
    # on_error == "error": surface parse failures as Spark exceptions.
    fn = F.to_date if kind == "date" else F.to_timestamp
    if fmt:
        return df.withColumn(column, fn(_col(column), fmt))
    return df.withColumn(column, fn(_col(column)))

ACTION_DISPATCH: Dict[str, Callable] = {
    "formula": _apply_formula,
    "cast": _apply_cast,
    "replace_value": _apply_replace_value,
    "fill_null": _apply_fill_null,
    "text_case": _apply_text_case,
    "trim": _apply_trim,
    "regex_replace": _apply_regex_replace,
    "extract": _apply_extract,
    "parse_date": _apply_parse_date,
}

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs.get("data")
    actions: List[Dict[str, Any]] = config.get("actions", []) or []

    if not actions:
        return {"prepared_data": df}

    for index, action in enumerate(actions):
        if not isinstance(action, dict):
            raise ValueError(
                f"actions[{index}]: expected an object, got {type(action).__name__}"
            )
        if action.get("enabled", True) is False:
            continue
        action_type = action.get("type", "")
        fn = ACTION_DISPATCH.get(action_type)
        if fn is None:
            raise ValueError(
                f"actions[{index}]: unsupported action type {action_type!r}. "
                f"Choose one of: {sorted(ACTION_DISPATCH.keys())}"
            )
        df = fn(df, action)
    return {"prepared_data": df}

# generated from the system
if "ld_display_outputs" not in globals():
    try:
        _ld_param = dbutils.widgets.getAll().get("ld_display_outputs")
        if _ld_param is not None and str(_ld_param).strip() != "":
            globals()["ld_display_outputs"] = str(_ld_param).strip().lower() not in ("false", "0", "no", "off")
        else:
            try:
                from dbruntime.databricks_repl_context import get_context
                globals()["ld_display_outputs"] = not get_context().isInJob
            except Exception:
                globals()["ld_display_outputs"] = True
    except Exception:
        globals()["ld_display_outputs"] = False
if "ld_display_outputs_for" not in globals():
    try:
        _ld_for = dbutils.widgets.getAll().get("ld_display_outputs_for")
        globals()["ld_display_outputs_for"] = frozenset(_p.strip() for _p in str(_ld_for).split(",") if _p.strip()) if _ld_for is not None else frozenset()
    except Exception:
        globals()["ld_display_outputs_for"] = frozenset()
ctx = globals().setdefault("ctx", {})
config = {
    "actions": [
        {
            "type": "formula",
            "target": "days_to_renewal",
            "expression": "DATEDIFF(renewal_date, current_date())"
        },
        {
            "type": "formula",
            "target": "decision_maker_int",
            "expression": "CAST(has_decision_maker AS INT)"
        },
        {
            "type": "formula",
            "target": "base_risk_score",
            "expression": "(CASE WHEN days_to_renewal BETWEEN 0 AND 90 THEN 30 ELSE 0 END) + (CASE WHEN open_high_priority_case_count > 0 THEN 25 ELSE 0 END) + (CASE WHEN open_sla_breach_count > 0 THEN 15 ELSE 0 END) + (CASE WHEN stalled_renewal_opportunity_count > 0 THEN 15 ELSE 0 END) + (CASE WHEN active_user_growth_rate <= -0.10 THEN 15 ELSE 0 END) + (CASE WHEN decision_maker_int = 0 THEN 10 ELSE 0 END) + (CASE WHEN arr >= 250000 THEN 10 ELSE 0 END) - (CASE WHEN active_expansion_opportunity_count > 0 THEN 10 ELSE 0 END)"
        },
        {
            "type": "formula",
            "target": "risk_score",
            "expression": "GREATEST(0, LEAST(base_risk_score, 100))"
        },
        {
            "type": "formula",
            "target": "risk_band",
            "expression": "CASE WHEN risk_score >= 80 THEN 'Critical' WHEN risk_score >= 60 THEN 'High' WHEN risk_score >= 40 THEN 'Medium' ELSE 'Low' END"
        },
        {
            "type": "formula",
            "target": "recommended_action",
            "expression": "CASE WHEN risk_band = 'Critical' THEN 'Executive escalation' WHEN risk_band = 'High' THEN 'CSM renewal save plan' WHEN risk_band = 'Medium' THEN 'Adoption and stakeholder follow-up' ELSE 'Monitor account health' END"
        }
    ]
}
config["meta_state"] = {"is_preview": False, "is_focused_preview": False, "is_disabled": False}
inputs = {
    "data": ctx["prepare_account_risk_features_a8b748f0.prepared_data"]
}
if config["meta_state"]["is_disabled"]:
    def _lb_limit_zero(value):
        if hasattr(value, "limit"):
            return value.limit(0)
        if isinstance(value, list):
            return [_lb_limit_zero(item) for item in value]
        return value
    inputs = {k: _lb_limit_zero(v) for k, v in inputs.items()}
out = run(config, inputs, spark)
if config["meta_state"]["is_disabled"]:
    out = {k: _lb_limit_zero(v) for k, v in out.items()}
ctx["prepare_renewal_risk_bbc08ae2.prepared_data"] = out["prepared_data"]
if globals().get("ld_display_outputs", False) or "prepare_renewal_risk_bbc08ae2" in globals().get("ld_display_outputs_for", frozenset()):
    display(ctx["prepare_renewal_risk_bbc08ae2.prepared_data"])

In [0]:
"""
id: filter_actionable_accounts_3958e59b
template: filter
templateVersion: 2.0.0
name: filter_actionable_accounts
position:
  x: 3300
  y: 397.5
description:
  text: Keep rows where risk band is Critical, High, or Medium; separate others.
  hash: f37ad77a
config:
  condition: risk_band IN ('Critical', 'High', 'Medium')
input:
  - node: prepare_renewal_risk_bbc08ae2
    input_port: data
    output_port: prepared_data
"""

# generated from the system
from typing import Dict, Any

from pyspark.sql import functions as F

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs["data"]
    condition = config.get("condition", "")

    if not condition:
        return {"filtered_data": df, "excluded_data": spark.createDataFrame([], df.schema)}

    keep = F.coalesce(F.expr(condition), F.lit(False))
    return {"filtered_data": df.filter(keep), "excluded_data": df.filter(~keep)}

# generated from the system
if "ld_display_outputs" not in globals():
    try:
        _ld_param = dbutils.widgets.getAll().get("ld_display_outputs")
        if _ld_param is not None and str(_ld_param).strip() != "":
            globals()["ld_display_outputs"] = str(_ld_param).strip().lower() not in ("false", "0", "no", "off")
        else:
            try:
                from dbruntime.databricks_repl_context import get_context
                globals()["ld_display_outputs"] = not get_context().isInJob
            except Exception:
                globals()["ld_display_outputs"] = True
    except Exception:
        globals()["ld_display_outputs"] = False
if "ld_display_outputs_for" not in globals():
    try:
        _ld_for = dbutils.widgets.getAll().get("ld_display_outputs_for")
        globals()["ld_display_outputs_for"] = frozenset(_p.strip() for _p in str(_ld_for).split(",") if _p.strip()) if _ld_for is not None else frozenset()
    except Exception:
        globals()["ld_display_outputs_for"] = frozenset()
ctx = globals().setdefault("ctx", {})
config = {
    "condition": "risk_band IN ('Critical', 'High', 'Medium')"
}
config["meta_state"] = {"is_preview": False, "is_focused_preview": False, "is_disabled": False}
inputs = {
    "data": ctx["prepare_renewal_risk_bbc08ae2.prepared_data"]
}
if config["meta_state"]["is_disabled"]:
    def _lb_limit_zero(value):
        if hasattr(value, "limit"):
            return value.limit(0)
        if isinstance(value, list):
            return [_lb_limit_zero(item) for item in value]
        return value
    inputs = {k: _lb_limit_zero(v) for k, v in inputs.items()}
out = run(config, inputs, spark)
if config["meta_state"]["is_disabled"]:
    out = {k: _lb_limit_zero(v) for k, v in out.items()}
ctx["filter_actionable_accounts_3958e59b.filtered_data"] = out["filtered_data"]
ctx["filter_actionable_accounts_3958e59b.excluded_data"] = out["excluded_data"]
if globals().get("ld_display_outputs", False) or "filter_actionable_accounts_3958e59b" in globals().get("ld_display_outputs_for", frozenset()):
    display(ctx["filter_actionable_accounts_3958e59b.filtered_data"])
    display(ctx["filter_actionable_accounts_3958e59b.excluded_data"])

In [0]:
"""
id: output_renewal_risk_gold_7e3cc4b6
template: output
templateVersion: 3.0.0
name: renewal_risk_lakehouse.renewal_risk_dev.gold_account_renewal_risk
position:
  x: 3300
  y: 552.5
description:
  text: Overwrite the specified table with new data in the given catalog and schema.
  hash: 8c4d65e4
config:
  output_type: table
  catalog: renewal_risk_lakehouse
  schema: renewal_risk_dev
  table_name: gold_account_renewal_risk
  write_mode: overwrite
input:
  - node: prepare_renewal_risk_bbc08ae2
    input_port: data
    output_port: prepared_data
"""

# generated from the system
import uuid
from typing import Dict, Any, List

DELTA_COLUMN_MAPPING_PROP = "'delta.columnMapping.mode' = 'id'"

UNIFORM_PROPS = (
    "'delta.enableIcebergCompatV2' = 'true', "
    "'delta.universalFormat.enabledFormats' = 'iceberg', "
    + DELTA_COLUMN_MAPPING_PROP
)

def _quote(name: str) -> str:
    if len(name) >= 2 and name.startswith("`") and name.endswith("`"):
        name = name[1:-1].replace("``", "`")
    return "`" + name.replace("`", "``") + "`"

def _qualified(catalog: str, schema: str, table: str) -> str:
    if not table:
        raise ValueError("Output: 'table_name' is required")
    if catalog and not schema:
        raise ValueError("Output: 'schema' is required when 'catalog' is set")
    parts = [_quote(p) for p in (catalog, schema, table) if p]
    return ".".join(parts)

def _table_exists(spark, qualified_name: str) -> bool:
    try:
        return spark.catalog.tableExists(qualified_name)
    except Exception:
        return False

def _existing_column_mapping_mode(spark, qualified_name: str) -> str:
    if spark is None or not qualified_name:
        return ""
    try:
        rows = spark.sql(
            f"SHOW TBLPROPERTIES {qualified_name} ('delta.columnMapping.mode')"
        ).collect()
        if rows:
            value = str(rows[0]["value"])
            if value in ("name", "id"):
                return value
    except Exception:
        pass
    return ""

def _column_mapping_prop(spark, full_name: str) -> str:
    if spark is None or not full_name:
        return DELTA_COLUMN_MAPPING_PROP
    if not _table_exists(spark, full_name):
        return DELTA_COLUMN_MAPPING_PROP
    existing = _existing_column_mapping_mode(spark, full_name)
    if existing:
        return f"'delta.columnMapping.mode' = '{existing}'"
    return ""

def _tbl_properties_clause(
    user_props: str, format_value: str, spark=None, full_name: str = ""
) -> str:
    parts: List[str] = []
    table_exists = bool(
        spark is not None and full_name and _table_exists(spark, full_name)
    )
    column_mapping = _column_mapping_prop(spark, full_name)
    if format_value == "uniform":
        uniform_base = (
            "'delta.enableIcebergCompatV2' = 'true', "
            "'delta.universalFormat.enabledFormats' = 'iceberg'"
        )
        if column_mapping:
            parts.append(uniform_base + ", " + column_mapping)
        elif not table_exists:
            parts.append(UNIFORM_PROPS)
        else:
            parts.append(uniform_base)
    elif format_value in ("delta", "default") and column_mapping:
        parts.append(column_mapping)
    cleaned = (user_props or "").strip().rstrip(",").strip()
    if cleaned:
        parts.append(cleaned)
    if not parts:
        return ""
    return " TBLPROPERTIES (" + ", ".join(parts) + ")"

def _using_clause(format_value: str) -> str:
    if format_value == "iceberg":
        return " USING ICEBERG"
    if format_value in ("delta", "uniform"):
        return " USING DELTA"
    return ""

def _resolve_args(table_properties: str) -> Dict[str, str]:
    import re

    param_names = set(re.findall(r"(?<!:):(\w+)", table_properties or ""))
    if not param_names:
        return {}
    try:
        widgets = dbutils.widgets.getAll()
    except NameError:
        return {}
    return {name: widgets[name] for name in param_names if name in widgets}

def _comment_clause(comment: str) -> str:
    cleaned = (comment or "").strip()
    if not cleaned:
        return ""
    escaped = cleaned.replace("'", "''")
    return f" COMMENT '{escaped}'"

def _cluster_by_clause(mode: str, column: str) -> str:
    if mode == "auto":
        return " CLUSTER BY AUTO"
    if mode == "column" and column:
        return f" CLUSTER BY ({_quote(column)})"
    return ""

SUPPORTED_FILE_TYPES = {"csv", "json", "excel"}

FILE_EXTENSIONS = {"csv": "csv", "json": "json", "excel": "xlsx"}

MAX_SINGLE_FILE_ROWS = 1_000_000

MAX_EXCEL_CELLS = 5_000_000

def _excel_row_cap(num_cols: int) -> int:
    return min(MAX_SINGLE_FILE_ROWS, max(1, MAX_EXCEL_CELLS // max(1, num_cols)))

_FORMULA_TRIGGERS = "=+-@\t\r"

def _neutralize_formula(value):
    if isinstance(value, str) and value and value[0] in _FORMULA_TRIGGERS:
        return "'" + value
    return value

def _file_path(catalog: str, schema: str, volume: str, file_name: str) -> str:
    if not file_name:
        raise ValueError("Output: 'file_name' is required for a file output")
    if catalog and schema and volume:
        if "/" in file_name or ".." in file_name:
            raise ValueError(
                f"Output: invalid 'file_name' {file_name!r}: it is the final "
                "path segment under the volume and cannot contain '/' or '..'."
            )
        schema_segment = schema.replace(".", "/")
        return f"/Volumes/{catalog}/{schema_segment}/{volume}/{file_name}"
    return file_name

def _with_extension(file_name: str, file_type: str) -> str:
    suffix = f".{FILE_EXTENSIONS.get(file_type, file_type)}"
    if not file_name or file_name.lower().endswith(suffix):
        return file_name
    return file_name + suffix

def _write_single_file(rows, columns, file_type: str, path: str, append: bool) -> None:
    import csv
    import json
    import os
    import shutil
    import tempfile

    def _remove(target: str) -> None:
        if os.path.isdir(target):
            shutil.rmtree(target)
        elif os.path.exists(target):
            os.remove(target)

    def _is_spark_output_dir(target: str) -> bool:
        return os.path.isfile(os.path.join(target, "_SUCCESS"))

    def _stage(write_body) -> None:
        from databricks.sdk import WorkspaceClient

        fd, local_tmp = tempfile.mkstemp(suffix=".lb-output-stage")
        os.close(fd)
        try:
            write_body(local_tmp)
            _remove(path)
            with open(local_tmp, "rb") as f:
                WorkspaceClient().files.upload(path, f, overwrite=True)
        finally:
            if os.path.isfile(local_tmp):
                os.remove(local_tmp)

    if os.path.isdir(path) and not _is_spark_output_dir(path):
        raise ValueError(
            f"Output: cannot write file to {path}: a directory already "
            f"exists at that path."
        )
    appending = append and os.path.isfile(path)
    if file_type == "csv":
        existing_data_rows = 0
        has_existing_header = False
        header = list(columns)
        if appending:
            with open(path, newline="", encoding="utf-8") as handle:
                reader = csv.reader(handle)
                first = next(reader, None)
                if first is not None:
                    header = first
                    has_existing_header = True
                    existing_data_rows = sum(1 for _ in reader)
        if has_existing_header and sorted(header) != sorted(columns):
            raise ValueError(
                f"Output: cannot append to {path}: the data's columns "
                f"{sorted(columns)} do not match the existing file's "
                f"columns {sorted(header)}."
            )
        if existing_data_rows + len(rows) > MAX_SINGLE_FILE_ROWS:
            raise ValueError(
                f"Output: appending {len(rows):,} rows would grow {path} past "
                f"the single-file export limit ({MAX_SINGLE_FILE_ROWS:,} rows) "
                f"— write to a table instead."
            )

        def _write_csv(tmp_path: str) -> None:
            with open(tmp_path, "w", newline="", encoding="utf-8") as out:
                if has_existing_header:
                    with open(path, newline="", encoding="utf-8") as src:
                        shutil.copyfileobj(src, out)
                else:
                    csv.writer(out).writerow(header)
                writer = csv.writer(out)
                for row in rows:
                    writer.writerow([_neutralize_formula(row[c]) for c in header])

        _stage(_write_csv)
        return

    if file_type == "excel":
        import io

        try:
            import openpyxl
        except ModuleNotFoundError as exc:
            raise ModuleNotFoundError(
                "Writing Excel files requires the 'openpyxl' package, which "
                "is not installed in this environment. Add "
                "openpyxl==3.1.5 to the notebook environment "
                "and apply it, then run again."
            ) from exc

        import pandas as pd
        import re as _re

        try:
            from openpyxl.cell.cell import ILLEGAL_CHARACTERS_RE as _lb_illegal
        except ImportError:
            _lb_illegal = _re.compile(r"[\x00-\x08\x0b\x0c\x0e-\x1f]")

        def _excel_safe(value):
            if isinstance(value, str):
                return _lb_illegal.sub("", value)
            if isinstance(value, (int, float, bool, type(None))):
                return value
            return _lb_illegal.sub("", str(value))

        header = list(columns)
        source_keys = list(columns)
        existing_values: List = []
        if appending:
            workbook = openpyxl.load_workbook(path, read_only=True)
            try:
                sheet = workbook.active
                row_iter = sheet.iter_rows(values_only=True)
                first = next(row_iter, None)
                if first is not None:
                    header = list(first)
                    if sorted(str(c) for c in header) != sorted(
                        str(_excel_safe(c)) for c in columns
                    ):
                        raise ValueError(
                            f"Output: cannot append to {path}: the data's columns "
                            f"{[str(_excel_safe(c)) for c in columns]} do not match the existing "
                            f"file's columns {[str(c) for c in header]}."
                        )
                    raw_by_safe: Dict[str, Any] = {}
                    for c in columns:
                        raw_by_safe.setdefault(str(_excel_safe(c)), c)
                    source_keys = [raw_by_safe[str(c)] for c in header]
                    row_cap = _excel_row_cap(len(header))
                    for record in row_iter:
                        if len(existing_values) + len(rows) >= row_cap:
                            raise ValueError(
                                f"Output: appending {len(rows):,} rows would grow {path} past "
                                f"the single-file excel export limit ({MAX_EXCEL_CELLS:,} cells "
                                f"at {len(header):,} columns) — write to a table instead."
                            )
                        existing_values.append(list(record))
            finally:
                workbook.close()

        combined = existing_values + [
            [_neutralize_formula(_excel_safe(row[k])) for k in source_keys] for row in rows
        ]
        new_df = pd.DataFrame(combined, columns=[_excel_safe(c) for c in header])

        def _write_excel(tmp_path: str) -> None:
            buffer = io.BytesIO()
            new_df.to_excel(buffer, index=False, engine="openpyxl")
            with open(tmp_path, "wb") as out:
                out.write(buffer.getvalue())

        _stage(_write_excel)
        return

    records = []
    if appending:
        with open(path, encoding="utf-8") as handle:
            existing = json.load(handle)
        records = existing if isinstance(existing, list) else [existing]
        if records:
            existing_columns = (
                list(records[0].keys()) if isinstance(records[0], dict) else []
            )
            if existing_columns and sorted(existing_columns) != sorted(columns):
                raise ValueError(
                    f"Output: cannot append to {path}: the data's columns "
                    f"{sorted(columns)} do not match the existing file's "
                    f"columns {sorted(existing_columns)}."
                )
    if len(records) + len(rows) > MAX_SINGLE_FILE_ROWS:
        raise ValueError(
            f"Output: appending {len(rows):,} rows would grow {path} past "
            f"the single-file export limit ({MAX_SINGLE_FILE_ROWS:,} rows) "
            f"— write to a table instead."
        )
    records.extend({c: row[c] for c in columns} for row in rows)

    def _write_json(tmp_path: str) -> None:
        with open(tmp_path, "w", encoding="utf-8") as handle:
            json.dump(records, handle, default=str, indent=2)

    _stage(_write_json)

def _run_file(config: Dict[str, Any], df, spark) -> None:
    file_type = config.get("file_type", "csv")
    if file_type not in SUPPORTED_FILE_TYPES:
        raise ValueError(
            f"Output: file_type '{file_type}' is not yet supported. Use csv, json, or excel."
        )
    path = _file_path(
        config.get("catalog", ""),
        config.get("schema", ""),
        config.get("volume", ""),
        _with_extension(config.get("file_name", ""), file_type),
    )
    append = config.get("write_mode", "overwrite") == "append"
    if file_type == "excel":
        row_cap = _excel_row_cap(len(df.columns))
        rows = df.limit(row_cap + 1).collect()
        if len(rows) > row_cap:
            raise ValueError(
                f"Output: result too large for single-file excel export "
                f"(over the {MAX_EXCEL_CELLS:,}-cell limit at {len(df.columns):,} "
                f"columns) — write to a table instead."
            )
    else:
        rows = df.limit(MAX_SINGLE_FILE_ROWS + 1).collect()
        if len(rows) > MAX_SINGLE_FILE_ROWS:
            raise ValueError(
                f"Output: result too large for single-file export "
                f"(over {MAX_SINGLE_FILE_ROWS:,} rows) — write to a table instead."
            )
    _write_single_file(rows, df.columns, file_type, path, append)

def _run_materialized_view(
    config: Dict[str, Any], source_view: str, spark
) -> None:
    catalog = config.get("catalog", "")
    schema = config.get("schema", "")
    view_name = config.get("view_name", "")
    if not view_name:
        raise ValueError("Output: 'view_name' is required for a materialized view")
    full_name = _qualified(catalog, schema, view_name)
    cluster_by = _cluster_by_clause(
        config.get("cluster_by_mode", "auto"), config.get("cluster_by_column", "") or ""
    )
    comment = _comment_clause(config.get("comment", "") or "")
    stmt = (
        f"CREATE OR REFRESH MATERIALIZED VIEW {full_name}{cluster_by}{comment} "
        f"AS SELECT * FROM {_quote(source_view)}"
    )
    spark.sql(stmt)

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs["data"]
    output_type = config.get("output_type", "table")
    catalog = config.get("catalog", "")
    schema = config.get("schema", "")
    table_name = config.get("table_name", "")
    write_mode = config.get("write_mode", "overwrite")
    merge_keys: List[str] = [k for k in (config.get("merge_keys") or []) if k]
    format_value = config.get("format", "default")
    table_properties = config.get("table_properties", "") or ""

    if output_type == "file":
        _run_file(config, df, spark)
        return {}

    source_view = f"_lb_output_v2_src_{uuid.uuid4().hex}"
    df.createOrReplaceTempView(source_view)

    if output_type == "materialized_view":
        try:
            _run_materialized_view(config, source_view, spark)
        finally:
            spark.catalog.dropTempView(source_view)
        return {}

    if not table_name:
        raise ValueError("Output: 'table_name' is required")

    full_name = _qualified(catalog, schema, table_name)

    if write_mode == "merge" and not merge_keys:
        raise ValueError("Output: 'merge_keys' is required when write_mode is merge")

    try:
        args = _resolve_args(table_properties)

        using = _using_clause(format_value)
        tblprops = _tbl_properties_clause(
            table_properties, format_value, spark=spark, full_name=full_name
        )

        if write_mode == "append":
            create_stmt = (
                f"CREATE TABLE IF NOT EXISTS {full_name}{using}{tblprops} "
                f"AS SELECT * FROM {_quote(source_view)} WHERE 1=0"
            )
            spark.sql(create_stmt, args=args) if args else spark.sql(create_stmt)
            insert_stmt = (
                f"INSERT INTO {full_name} BY NAME "
                f"SELECT * FROM {_quote(source_view)}"
            )
            spark.sql(insert_stmt, args=args) if args else spark.sql(insert_stmt)
        elif write_mode == "merge":
            create_stmt = (
                f"CREATE TABLE IF NOT EXISTS {full_name}{using}{tblprops} "
                f"AS SELECT * FROM {_quote(source_view)} WHERE 1=0"
            )
            spark.sql(create_stmt, args=args) if args else spark.sql(create_stmt)
            on_clause = " AND ".join(
                f"t.{_quote(k)} = s.{_quote(k)}" for k in merge_keys
            )
            merge_stmt = (
                f"MERGE INTO {full_name} t "
                f"USING {_quote(source_view)} s "
                f"ON {on_clause} "
                f"WHEN MATCHED THEN UPDATE SET * "
                f"WHEN NOT MATCHED THEN INSERT *"
            )
            spark.sql(merge_stmt, args=args) if args else spark.sql(merge_stmt)
        else:
            stmt = (
                f"CREATE OR REPLACE TABLE {full_name}{using}{tblprops} "
                f"AS SELECT * FROM {_quote(source_view)}"
            )
            spark.sql(stmt, args=args) if args else spark.sql(stmt)
    except Exception as exc:
        if table_properties.strip():
            raise ValueError(
                "Output: failed to write the table. Check the 'table_properties' "
                "and 'schema' fields for invalid SQL. Underlying error: "
                f"{exc}"
            ) from exc
        raise
    finally:
        spark.catalog.dropTempView(source_view)
    return {}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {
    "output_type": "table",
    "catalog": "renewal_risk_lakehouse",
    "schema": "renewal_risk_dev",
    "table_name": "gold_account_renewal_risk",
    "write_mode": "overwrite"
}
config["meta_state"] = {"is_preview": False, "is_focused_preview": False, "is_disabled": False}
inputs = {
    "data": ctx["prepare_renewal_risk_bbc08ae2.prepared_data"]
}
if config["meta_state"]["is_disabled"]:
    def _lb_limit_zero(value):
        if hasattr(value, "limit"):
            return value.limit(0)
        if isinstance(value, list):
            return [_lb_limit_zero(item) for item in value]
        return value
    inputs = {k: _lb_limit_zero(v) for k, v in inputs.items()}
if not config["meta_state"]["is_disabled"]:
    out = run(config, inputs, spark)
else:
    out = {}

In [0]:
"""
id: prep_account_360_summary_34ac3296
template: prepare
templateVersion: 1.0.0
name: prep_account_360_summary
position:
  x: 3300
  y: 165
description:
  text: Create a summary text column concatenating account details and metrics.
  hash: bef6bcf9
previewCodeHash: fe08b66ede167964
config:
  actions:
    - type: formula
      target: account_summary_text
      expression: "CONCAT('Account: ', account_name, '; Segment: ', customer_segment, '; ARR: $', CAST(arr AS STRING), '; Renewal: ', CAST(renewal_date AS STRING), '; Risk Score: ', CAST(risk_score AS STRING), '; Risk Band: ', risk_band, '; Support Exposure: ', CAST(open_case_count AS STRING), ' open cases, ', CAST(open_high_priority_case_count AS STRING), ' high priority, ', CAST(open_sla_breach_count AS STRING), ' SLA breaches; Pipeline Health: $', CAST(renewal_pipeline_amount AS STRING), ' renewal, ', CAST(stalled_renewal_opportunity_count AS STRING), ' stalled renewals, ', CAST(active_expansion_opportunity_count AS STRING), ' expansions; Usage Trend: ', CAST(avg_active_users_latest_14_days AS STRING), ' recent users, ', CAST(avg_active_users_previous_14_days AS STRING), ' prior users, Growth: ', CAST(active_user_growth_rate AS STRING), '; Decision Maker Coverage: ', CAST(decision_maker_contact_count AS STRING), ' decision-makers, Flag: ', CAST(has_decision_maker AS STRING), '; Action: ', recommended_action)"
input:
  - node: prepare_renewal_risk_bbc08ae2
    input_port: data
    output_port: prepared_data
"""

# generated from the system
from typing import Any, Callable, Dict, List
import pyspark.sql.functions as F

TEXT_CASE_FUNCTIONS: Dict[str, Callable] = {
    "lower": F.lower,
    "upper": F.upper,
    "title": F.initcap,
}

TRIM_FUNCTIONS: Dict[str, Callable] = {
    "both": F.trim,
    "left": F.ltrim,
    "right": F.rtrim,
}

VALID_MATCH_MODES = {"exact", "contains", "prefix", "suffix", "regex"}

def _require_column(df, column: str, action_type: str) -> None:
    if not column:
        raise ValueError(f"{action_type}: 'column' is required")
    if column not in df.columns:
        raise ValueError(
            f"{action_type}: column '{column}' not found in input data. "
            f"Available columns: {df.columns}"
        )

def _column_dtype(df, column: str) -> str:
    return df.schema[column].dataType.simpleString()

def _quoted_ident(name: str) -> str:
    return "`" + name.replace("`", "``") + "`"

def _col(name: str):
    """Reference a column by exact name.

    F.col parses its argument as a column expression, so a name containing
    a '.' (e.g. "a.b") is otherwise read as struct-field access and fails
    even though the column exists. Backtick-quoting forces an exact-name
    lookup; quoting plain names is harmless.
    """
    return F.col(_quoted_ident(name))

def _coerced_lit(value: Any, target_dtype: str):
    """Wrap a user-provided value as a literal cast to the target column's type.

    UI always feeds text per the operator spec; this is where the
    type coercion happens so the per-action UI stays simple.
    """
    return F.lit(value).cast(target_dtype)

def _apply_formula(df, action: Dict[str, Any]):
    target = action.get("target", "")
    expression = action.get("expression", "")
    if not target:
        raise ValueError("formula: 'target' is required")
    if not expression:
        raise ValueError("formula: 'expression' is required")
    return df.withColumn(target, F.expr(expression))

def _apply_cast(df, action: Dict[str, Any]):
    column = action.get("column", "")
    to_type = action.get("to", "")
    on_error = action.get("on_error", "null")
    _require_column(df, column, "cast")
    if not to_type:
        raise ValueError("cast: 'to' (target type) is required")
    if on_error == "null":
        return df.withColumn(
            column,
            F.expr(f"try_cast({_quoted_ident(column)} as {to_type})"),
        )
    return df.withColumn(column, _col(column).cast(to_type))

def _apply_replace_value(df, action: Dict[str, Any]):
    column = action.get("column", "")
    match_mode = action.get("match_mode", "exact")
    match = action.get("match", "")
    with_val = action.get("with", "")
    case_sensitive = bool(action.get("case_sensitive", False))
    _require_column(df, column, "replace_value")
    if match_mode not in VALID_MATCH_MODES:
        raise ValueError(
            f"replace_value: unsupported match_mode '{match_mode}'. "
            f"Choose one of: {sorted(VALID_MATCH_MODES)}"
        )
    dtype = _column_dtype(df, column)
    col_as_string = _col(column).cast("string")
    match_lit = F.lit(match)
    if case_sensitive:
        cmp_col = col_as_string
        cmp_lit = match_lit
    else:
        cmp_col = F.lower(col_as_string)
        cmp_lit = F.lower(match_lit)

    if match_mode == "exact":
        predicate = cmp_col == cmp_lit
    elif match_mode == "contains":
        predicate = cmp_col.contains(cmp_lit)
    elif match_mode == "prefix":
        predicate = cmp_col.startswith(cmp_lit)
    elif match_mode == "suffix":
        predicate = cmp_col.endswith(cmp_lit)
    else:  # regex
        pattern = match if case_sensitive else f"(?i){match}"
        predicate = col_as_string.rlike(pattern)

    replacement = _coerced_lit(with_val, dtype)
    return df.withColumn(
        column,
        F.when(predicate, replacement).otherwise(_col(column)),
    )

def _apply_fill_null(df, action: Dict[str, Any]):
    column = action.get("column", "")
    with_val = action.get("with", "")
    _require_column(df, column, "fill_null")
    dtype = _column_dtype(df, column)
    replacement = _coerced_lit(with_val, dtype)
    return df.withColumn(
        column,
        F.when(_col(column).isNull(), replacement).otherwise(_col(column)),
    )

def _apply_text_case(df, action: Dict[str, Any]):
    column = action.get("column", "")
    case = action.get("case", "")
    _require_column(df, column, "text_case")
    fn = TEXT_CASE_FUNCTIONS.get(case)
    if fn is None:
        raise ValueError(
            f"text_case: unsupported case '{case}'. "
            f"Choose one of: {sorted(TEXT_CASE_FUNCTIONS.keys())}"
        )
    return df.withColumn(column, fn(_col(column)))

def _apply_trim(df, action: Dict[str, Any]):
    column = action.get("column", "")
    side = action.get("side", "both")
    _require_column(df, column, "trim")
    fn = TRIM_FUNCTIONS.get(side)
    if fn is None:
        raise ValueError(
            f"trim: unsupported side '{side}'. "
            f"Choose one of: {sorted(TRIM_FUNCTIONS.keys())}"
        )
    return df.withColumn(column, fn(_col(column)))

def _apply_regex_replace(df, action: Dict[str, Any]):
    column = action.get("column", "")
    pattern = action.get("pattern", "")
    replacement = action.get("replacement", "")
    _require_column(df, column, "regex_replace")
    return df.withColumn(
        column,
        F.regexp_replace(_col(column), pattern, replacement),
    )

def _apply_extract(df, action: Dict[str, Any]):
    column = action.get("column", "")
    pattern = action.get("pattern", "")
    target = action.get("target") or column
    group_raw = action.get("group", 0)
    _require_column(df, column, "extract")
    try:
        group_idx = int(group_raw)
    except (TypeError, ValueError) as exc:
        raise ValueError(
            f"extract: 'group' must be an integer, got {group_raw!r}"
        ) from exc
    return df.withColumn(
        target,
        F.regexp_extract(_col(column), pattern, group_idx),
    )

def _apply_parse_date(df, action: Dict[str, Any]):
    column = action.get("column", "")
    kind = action.get("kind", "date")
    fmt = action.get("format") or None
    on_error = action.get("on_error", "null")
    _require_column(df, column, "parse_date")
    if kind not in ("date", "timestamp"):
        raise ValueError(
            f"parse_date: unsupported kind '{kind}'. Choose date or timestamp."
        )
    # PySpark exposes F.try_to_timestamp from Spark 3.5 but only adds
    # F.try_to_date in Spark 4.0. Current Databricks Runtimes still ship
    # Spark 3.5, where referencing F.try_to_date raises AttributeError
    # before any data is touched. Route the null-on-error path through
    # primitives that exist in both 3.5 and 4.0:
    #   - F.try_to_timestamp  (PySpark 3.5+, returns NULL under ANSI too)
    #   - SQL try_cast        (Spark 3.5+, returns NULL under ANSI too)
    #   - F.to_date           (PySpark 3.5+; returns NULL on parse failure
    #                          under the default non-ANSI mode, which is
    #                          the Databricks default. Under ANSI mode it
    #                          raises — accepted limitation until
    #                          try_to_date lands on every supported DBR.)
    if on_error == "null":
        if kind == "timestamp":
            if fmt:
                return df.withColumn(
                    column, F.try_to_timestamp(_col(column), F.lit(fmt))
                )
            return df.withColumn(column, F.try_to_timestamp(_col(column)))
        # kind == "date"
        if fmt:
            return df.withColumn(column, F.to_date(_col(column), fmt))
        return df.withColumn(
            column, F.expr(f"try_cast({_quoted_ident(column)} as date)")
        )
    # on_error == "error": surface parse failures as Spark exceptions.
    fn = F.to_date if kind == "date" else F.to_timestamp
    if fmt:
        return df.withColumn(column, fn(_col(column), fmt))
    return df.withColumn(column, fn(_col(column)))

ACTION_DISPATCH: Dict[str, Callable] = {
    "formula": _apply_formula,
    "cast": _apply_cast,
    "replace_value": _apply_replace_value,
    "fill_null": _apply_fill_null,
    "text_case": _apply_text_case,
    "trim": _apply_trim,
    "regex_replace": _apply_regex_replace,
    "extract": _apply_extract,
    "parse_date": _apply_parse_date,
}

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs.get("data")
    actions: List[Dict[str, Any]] = config.get("actions", []) or []

    if not actions:
        return {"prepared_data": df}

    for index, action in enumerate(actions):
        if not isinstance(action, dict):
            raise ValueError(
                f"actions[{index}]: expected an object, got {type(action).__name__}"
            )
        if action.get("enabled", True) is False:
            continue
        action_type = action.get("type", "")
        fn = ACTION_DISPATCH.get(action_type)
        if fn is None:
            raise ValueError(
                f"actions[{index}]: unsupported action type {action_type!r}. "
                f"Choose one of: {sorted(ACTION_DISPATCH.keys())}"
            )
        df = fn(df, action)
    return {"prepared_data": df}

# generated from the system
if "ld_display_outputs" not in globals():
    try:
        _ld_param = dbutils.widgets.getAll().get("ld_display_outputs")
        if _ld_param is not None and str(_ld_param).strip() != "":
            globals()["ld_display_outputs"] = str(_ld_param).strip().lower() not in ("false", "0", "no", "off")
        else:
            try:
                from dbruntime.databricks_repl_context import get_context
                globals()["ld_display_outputs"] = not get_context().isInJob
            except Exception:
                globals()["ld_display_outputs"] = True
    except Exception:
        globals()["ld_display_outputs"] = False
if "ld_display_outputs_for" not in globals():
    try:
        _ld_for = dbutils.widgets.getAll().get("ld_display_outputs_for")
        globals()["ld_display_outputs_for"] = frozenset(_p.strip() for _p in str(_ld_for).split(",") if _p.strip()) if _ld_for is not None else frozenset()
    except Exception:
        globals()["ld_display_outputs_for"] = frozenset()
ctx = globals().setdefault("ctx", {})
config = {
    "actions": [
        {
            "type": "formula",
            "target": "account_summary_text",
            "expression": "CONCAT('Account: ', account_name, '; Segment: ', customer_segment, '; ARR: $', CAST(arr AS STRING), '; Renewal: ', CAST(renewal_date AS STRING), '; Risk Score: ', CAST(risk_score AS STRING), '; Risk Band: ', risk_band, '; Support Exposure: ', CAST(open_case_count AS STRING), ' open cases, ', CAST(open_high_priority_case_count AS STRING), ' high priority, ', CAST(open_sla_breach_count AS STRING), ' SLA breaches; Pipeline Health: $', CAST(renewal_pipeline_amount AS STRING), ' renewal, ', CAST(stalled_renewal_opportunity_count AS STRING), ' stalled renewals, ', CAST(active_expansion_opportunity_count AS STRING), ' expansions; Usage Trend: ', CAST(avg_active_users_latest_14_days AS STRING), ' recent users, ', CAST(avg_active_users_previous_14_days AS STRING), ' prior users, Growth: ', CAST(active_user_growth_rate AS STRING), '; Decision Maker Coverage: ', CAST(decision_maker_contact_count AS STRING), ' decision-makers, Flag: ', CAST(has_decision_maker AS STRING), '; Action: ', recommended_action)"
        }
    ]
}
config["meta_state"] = {"is_preview": False, "is_focused_preview": False, "is_disabled": False}
inputs = {
    "data": ctx["prepare_renewal_risk_bbc08ae2.prepared_data"]
}
if config["meta_state"]["is_disabled"]:
    def _lb_limit_zero(value):
        if hasattr(value, "limit"):
            return value.limit(0)
        if isinstance(value, list):
            return [_lb_limit_zero(item) for item in value]
        return value
    inputs = {k: _lb_limit_zero(v) for k, v in inputs.items()}
out = run(config, inputs, spark)
if config["meta_state"]["is_disabled"]:
    out = {k: _lb_limit_zero(v) for k, v in out.items()}
ctx["prep_account_360_summary_34ac3296.prepared_data"] = out["prepared_data"]
if globals().get("ld_display_outputs", False) or "prep_account_360_summary_34ac3296" in globals().get("ld_display_outputs_for", frozenset()):
    display(ctx["prep_account_360_summary_34ac3296.prepared_data"])

In [0]:
"""
id: sort_csm_priority_e0479c5b
template: sort
templateVersion: 1.0.0
name: sort_csm_priority
position:
  x: 3600
  y: 397.5
description:
  text: Sort data by risk score and arr descending, then days to renewal ascending.
  hash: 6dfd5b9a
previewCodeHash: 6edd3b698a070baa
config:
  sort_expressions:
    - columnExpr:
        expr: risk_score
        type: expr
      sortBy: DESC
    - columnExpr:
        expr: arr
        type: expr
      sortBy: DESC
    - columnExpr:
        expr: days_to_renewal
        type: expr
      sortBy: ASC
input:
  - node: filter_actionable_accounts_3958e59b
    input_port: data
    output_port: filtered_data
"""

# generated from the system
from typing import Dict, Any
import pyspark.sql.functions as F

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs.get("data")
    sort_expressions = config.get("sort_expressions", [])

    if not sort_expressions:
        return {"sorted_data": df}

    order_cols = []
    for sort_def in sort_expressions:
        col_expr = sort_def.get("columnExpr", {})
        raw_expr = col_expr.get("expr", "")
        direction = sort_def.get("sortBy", "UNSET")

        col = F.col(raw_expr)
        if direction == "DESC":
            col = col.desc()
        elif direction == "ASC":
            col = col.asc()

        order_cols.append(col)

    return {"sorted_data": df.orderBy(*order_cols)}

# generated from the system
if "ld_display_outputs" not in globals():
    try:
        _ld_param = dbutils.widgets.getAll().get("ld_display_outputs")
        if _ld_param is not None and str(_ld_param).strip() != "":
            globals()["ld_display_outputs"] = str(_ld_param).strip().lower() not in ("false", "0", "no", "off")
        else:
            try:
                from dbruntime.databricks_repl_context import get_context
                globals()["ld_display_outputs"] = not get_context().isInJob
            except Exception:
                globals()["ld_display_outputs"] = True
    except Exception:
        globals()["ld_display_outputs"] = False
if "ld_display_outputs_for" not in globals():
    try:
        _ld_for = dbutils.widgets.getAll().get("ld_display_outputs_for")
        globals()["ld_display_outputs_for"] = frozenset(_p.strip() for _p in str(_ld_for).split(",") if _p.strip()) if _ld_for is not None else frozenset()
    except Exception:
        globals()["ld_display_outputs_for"] = frozenset()
ctx = globals().setdefault("ctx", {})
config = {
    "sort_expressions": [
        {
            "columnExpr": {
                "expr": "risk_score",
                "type": "expr"
            },
            "sortBy": "DESC"
        },
        {
            "columnExpr": {
                "expr": "arr",
                "type": "expr"
            },
            "sortBy": "DESC"
        },
        {
            "columnExpr": {
                "expr": "days_to_renewal",
                "type": "expr"
            },
            "sortBy": "ASC"
        }
    ]
}
config["meta_state"] = {"is_preview": False, "is_focused_preview": False, "is_disabled": False}
inputs = {
    "data": ctx["filter_actionable_accounts_3958e59b.filtered_data"]
}
if config["meta_state"]["is_disabled"]:
    def _lb_limit_zero(value):
        if hasattr(value, "limit"):
            return value.limit(0)
        if isinstance(value, list):
            return [_lb_limit_zero(item) for item in value]
        return value
    inputs = {k: _lb_limit_zero(v) for k, v in inputs.items()}
out = run(config, inputs, spark)
if config["meta_state"]["is_disabled"]:
    out = {k: _lb_limit_zero(v) for k, v in out.items()}
ctx["sort_csm_priority_e0479c5b.sorted_data"] = out["sorted_data"]
if globals().get("ld_display_outputs", False) or "sort_csm_priority_e0479c5b" in globals().get("ld_display_outputs_for", frozenset()):
    display(ctx["sort_csm_priority_e0479c5b.sorted_data"])

In [0]:
"""
id: output_account_360_summary_94211597
template: output
templateVersion: 3.0.0
name: gold_account_360_summary
position:
  x: 3600
  y: 242.5
description:
  text: Overwrite the specified table with the input data.
  hash: 37eefe88
config:
  output_type: table
  catalog: renewal_risk_lakehouse
  schema: renewal_risk_dev
  table_name: gold_account_360_summary
  write_mode: overwrite
input:
  - node: prep_account_360_summary_34ac3296
    input_port: data
    output_port: prepared_data
"""

# generated from the system
import uuid
from typing import Dict, Any, List

DELTA_COLUMN_MAPPING_PROP = "'delta.columnMapping.mode' = 'id'"

UNIFORM_PROPS = (
    "'delta.enableIcebergCompatV2' = 'true', "
    "'delta.universalFormat.enabledFormats' = 'iceberg', "
    + DELTA_COLUMN_MAPPING_PROP
)

def _quote(name: str) -> str:
    if len(name) >= 2 and name.startswith("`") and name.endswith("`"):
        name = name[1:-1].replace("``", "`")
    return "`" + name.replace("`", "``") + "`"

def _qualified(catalog: str, schema: str, table: str) -> str:
    if not table:
        raise ValueError("Output: 'table_name' is required")
    if catalog and not schema:
        raise ValueError("Output: 'schema' is required when 'catalog' is set")
    parts = [_quote(p) for p in (catalog, schema, table) if p]
    return ".".join(parts)

def _table_exists(spark, qualified_name: str) -> bool:
    try:
        return spark.catalog.tableExists(qualified_name)
    except Exception:
        return False

def _existing_column_mapping_mode(spark, qualified_name: str) -> str:
    if spark is None or not qualified_name:
        return ""
    try:
        rows = spark.sql(
            f"SHOW TBLPROPERTIES {qualified_name} ('delta.columnMapping.mode')"
        ).collect()
        if rows:
            value = str(rows[0]["value"])
            if value in ("name", "id"):
                return value
    except Exception:
        pass
    return ""

def _column_mapping_prop(spark, full_name: str) -> str:
    if spark is None or not full_name:
        return DELTA_COLUMN_MAPPING_PROP
    if not _table_exists(spark, full_name):
        return DELTA_COLUMN_MAPPING_PROP
    existing = _existing_column_mapping_mode(spark, full_name)
    if existing:
        return f"'delta.columnMapping.mode' = '{existing}'"
    return ""

def _tbl_properties_clause(
    user_props: str, format_value: str, spark=None, full_name: str = ""
) -> str:
    parts: List[str] = []
    table_exists = bool(
        spark is not None and full_name and _table_exists(spark, full_name)
    )
    column_mapping = _column_mapping_prop(spark, full_name)
    if format_value == "uniform":
        uniform_base = (
            "'delta.enableIcebergCompatV2' = 'true', "
            "'delta.universalFormat.enabledFormats' = 'iceberg'"
        )
        if column_mapping:
            parts.append(uniform_base + ", " + column_mapping)
        elif not table_exists:
            parts.append(UNIFORM_PROPS)
        else:
            parts.append(uniform_base)
    elif format_value in ("delta", "default") and column_mapping:
        parts.append(column_mapping)
    cleaned = (user_props or "").strip().rstrip(",").strip()
    if cleaned:
        parts.append(cleaned)
    if not parts:
        return ""
    return " TBLPROPERTIES (" + ", ".join(parts) + ")"

def _using_clause(format_value: str) -> str:
    if format_value == "iceberg":
        return " USING ICEBERG"
    if format_value in ("delta", "uniform"):
        return " USING DELTA"
    return ""

def _resolve_args(table_properties: str) -> Dict[str, str]:
    import re

    param_names = set(re.findall(r"(?<!:):(\w+)", table_properties or ""))
    if not param_names:
        return {}
    try:
        widgets = dbutils.widgets.getAll()
    except NameError:
        return {}
    return {name: widgets[name] for name in param_names if name in widgets}

def _comment_clause(comment: str) -> str:
    cleaned = (comment or "").strip()
    if not cleaned:
        return ""
    escaped = cleaned.replace("'", "''")
    return f" COMMENT '{escaped}'"

def _cluster_by_clause(mode: str, column: str) -> str:
    if mode == "auto":
        return " CLUSTER BY AUTO"
    if mode == "column" and column:
        return f" CLUSTER BY ({_quote(column)})"
    return ""

SUPPORTED_FILE_TYPES = {"csv", "json", "excel"}

FILE_EXTENSIONS = {"csv": "csv", "json": "json", "excel": "xlsx"}

MAX_SINGLE_FILE_ROWS = 1_000_000

MAX_EXCEL_CELLS = 5_000_000

def _excel_row_cap(num_cols: int) -> int:
    return min(MAX_SINGLE_FILE_ROWS, max(1, MAX_EXCEL_CELLS // max(1, num_cols)))

_FORMULA_TRIGGERS = "=+-@\t\r"

def _neutralize_formula(value):
    if isinstance(value, str) and value and value[0] in _FORMULA_TRIGGERS:
        return "'" + value
    return value

def _file_path(catalog: str, schema: str, volume: str, file_name: str) -> str:
    if not file_name:
        raise ValueError("Output: 'file_name' is required for a file output")
    if catalog and schema and volume:
        if "/" in file_name or ".." in file_name:
            raise ValueError(
                f"Output: invalid 'file_name' {file_name!r}: it is the final "
                "path segment under the volume and cannot contain '/' or '..'."
            )
        schema_segment = schema.replace(".", "/")
        return f"/Volumes/{catalog}/{schema_segment}/{volume}/{file_name}"
    return file_name

def _with_extension(file_name: str, file_type: str) -> str:
    suffix = f".{FILE_EXTENSIONS.get(file_type, file_type)}"
    if not file_name or file_name.lower().endswith(suffix):
        return file_name
    return file_name + suffix

def _write_single_file(rows, columns, file_type: str, path: str, append: bool) -> None:
    import csv
    import json
    import os
    import shutil
    import tempfile

    def _remove(target: str) -> None:
        if os.path.isdir(target):
            shutil.rmtree(target)
        elif os.path.exists(target):
            os.remove(target)

    def _is_spark_output_dir(target: str) -> bool:
        return os.path.isfile(os.path.join(target, "_SUCCESS"))

    def _stage(write_body) -> None:
        from databricks.sdk import WorkspaceClient

        fd, local_tmp = tempfile.mkstemp(suffix=".lb-output-stage")
        os.close(fd)
        try:
            write_body(local_tmp)
            _remove(path)
            with open(local_tmp, "rb") as f:
                WorkspaceClient().files.upload(path, f, overwrite=True)
        finally:
            if os.path.isfile(local_tmp):
                os.remove(local_tmp)

    if os.path.isdir(path) and not _is_spark_output_dir(path):
        raise ValueError(
            f"Output: cannot write file to {path}: a directory already "
            f"exists at that path."
        )
    appending = append and os.path.isfile(path)
    if file_type == "csv":
        existing_data_rows = 0
        has_existing_header = False
        header = list(columns)
        if appending:
            with open(path, newline="", encoding="utf-8") as handle:
                reader = csv.reader(handle)
                first = next(reader, None)
                if first is not None:
                    header = first
                    has_existing_header = True
                    existing_data_rows = sum(1 for _ in reader)
        if has_existing_header and sorted(header) != sorted(columns):
            raise ValueError(
                f"Output: cannot append to {path}: the data's columns "
                f"{sorted(columns)} do not match the existing file's "
                f"columns {sorted(header)}."
            )
        if existing_data_rows + len(rows) > MAX_SINGLE_FILE_ROWS:
            raise ValueError(
                f"Output: appending {len(rows):,} rows would grow {path} past "
                f"the single-file export limit ({MAX_SINGLE_FILE_ROWS:,} rows) "
                f"— write to a table instead."
            )

        def _write_csv(tmp_path: str) -> None:
            with open(tmp_path, "w", newline="", encoding="utf-8") as out:
                if has_existing_header:
                    with open(path, newline="", encoding="utf-8") as src:
                        shutil.copyfileobj(src, out)
                else:
                    csv.writer(out).writerow(header)
                writer = csv.writer(out)
                for row in rows:
                    writer.writerow([_neutralize_formula(row[c]) for c in header])

        _stage(_write_csv)
        return

    if file_type == "excel":
        import io

        try:
            import openpyxl
        except ModuleNotFoundError as exc:
            raise ModuleNotFoundError(
                "Writing Excel files requires the 'openpyxl' package, which "
                "is not installed in this environment. Add "
                "openpyxl==3.1.5 to the notebook environment "
                "and apply it, then run again."
            ) from exc

        import pandas as pd
        import re as _re

        try:
            from openpyxl.cell.cell import ILLEGAL_CHARACTERS_RE as _lb_illegal
        except ImportError:
            _lb_illegal = _re.compile(r"[\x00-\x08\x0b\x0c\x0e-\x1f]")

        def _excel_safe(value):
            if isinstance(value, str):
                return _lb_illegal.sub("", value)
            if isinstance(value, (int, float, bool, type(None))):
                return value
            return _lb_illegal.sub("", str(value))

        header = list(columns)
        source_keys = list(columns)
        existing_values: List = []
        if appending:
            workbook = openpyxl.load_workbook(path, read_only=True)
            try:
                sheet = workbook.active
                row_iter = sheet.iter_rows(values_only=True)
                first = next(row_iter, None)
                if first is not None:
                    header = list(first)
                    if sorted(str(c) for c in header) != sorted(
                        str(_excel_safe(c)) for c in columns
                    ):
                        raise ValueError(
                            f"Output: cannot append to {path}: the data's columns "
                            f"{[str(_excel_safe(c)) for c in columns]} do not match the existing "
                            f"file's columns {[str(c) for c in header]}."
                        )
                    raw_by_safe: Dict[str, Any] = {}
                    for c in columns:
                        raw_by_safe.setdefault(str(_excel_safe(c)), c)
                    source_keys = [raw_by_safe[str(c)] for c in header]
                    row_cap = _excel_row_cap(len(header))
                    for record in row_iter:
                        if len(existing_values) + len(rows) >= row_cap:
                            raise ValueError(
                                f"Output: appending {len(rows):,} rows would grow {path} past "
                                f"the single-file excel export limit ({MAX_EXCEL_CELLS:,} cells "
                                f"at {len(header):,} columns) — write to a table instead."
                            )
                        existing_values.append(list(record))
            finally:
                workbook.close()

        combined = existing_values + [
            [_neutralize_formula(_excel_safe(row[k])) for k in source_keys] for row in rows
        ]
        new_df = pd.DataFrame(combined, columns=[_excel_safe(c) for c in header])

        def _write_excel(tmp_path: str) -> None:
            buffer = io.BytesIO()
            new_df.to_excel(buffer, index=False, engine="openpyxl")
            with open(tmp_path, "wb") as out:
                out.write(buffer.getvalue())

        _stage(_write_excel)
        return

    records = []
    if appending:
        with open(path, encoding="utf-8") as handle:
            existing = json.load(handle)
        records = existing if isinstance(existing, list) else [existing]
        if records:
            existing_columns = (
                list(records[0].keys()) if isinstance(records[0], dict) else []
            )
            if existing_columns and sorted(existing_columns) != sorted(columns):
                raise ValueError(
                    f"Output: cannot append to {path}: the data's columns "
                    f"{sorted(columns)} do not match the existing file's "
                    f"columns {sorted(existing_columns)}."
                )
    if len(records) + len(rows) > MAX_SINGLE_FILE_ROWS:
        raise ValueError(
            f"Output: appending {len(rows):,} rows would grow {path} past "
            f"the single-file export limit ({MAX_SINGLE_FILE_ROWS:,} rows) "
            f"— write to a table instead."
        )
    records.extend({c: row[c] for c in columns} for row in rows)

    def _write_json(tmp_path: str) -> None:
        with open(tmp_path, "w", encoding="utf-8") as handle:
            json.dump(records, handle, default=str, indent=2)

    _stage(_write_json)

def _run_file(config: Dict[str, Any], df, spark) -> None:
    file_type = config.get("file_type", "csv")
    if file_type not in SUPPORTED_FILE_TYPES:
        raise ValueError(
            f"Output: file_type '{file_type}' is not yet supported. Use csv, json, or excel."
        )
    path = _file_path(
        config.get("catalog", ""),
        config.get("schema", ""),
        config.get("volume", ""),
        _with_extension(config.get("file_name", ""), file_type),
    )
    append = config.get("write_mode", "overwrite") == "append"
    if file_type == "excel":
        row_cap = _excel_row_cap(len(df.columns))
        rows = df.limit(row_cap + 1).collect()
        if len(rows) > row_cap:
            raise ValueError(
                f"Output: result too large for single-file excel export "
                f"(over the {MAX_EXCEL_CELLS:,}-cell limit at {len(df.columns):,} "
                f"columns) — write to a table instead."
            )
    else:
        rows = df.limit(MAX_SINGLE_FILE_ROWS + 1).collect()
        if len(rows) > MAX_SINGLE_FILE_ROWS:
            raise ValueError(
                f"Output: result too large for single-file export "
                f"(over {MAX_SINGLE_FILE_ROWS:,} rows) — write to a table instead."
            )
    _write_single_file(rows, df.columns, file_type, path, append)

def _run_materialized_view(
    config: Dict[str, Any], source_view: str, spark
) -> None:
    catalog = config.get("catalog", "")
    schema = config.get("schema", "")
    view_name = config.get("view_name", "")
    if not view_name:
        raise ValueError("Output: 'view_name' is required for a materialized view")
    full_name = _qualified(catalog, schema, view_name)
    cluster_by = _cluster_by_clause(
        config.get("cluster_by_mode", "auto"), config.get("cluster_by_column", "") or ""
    )
    comment = _comment_clause(config.get("comment", "") or "")
    stmt = (
        f"CREATE OR REFRESH MATERIALIZED VIEW {full_name}{cluster_by}{comment} "
        f"AS SELECT * FROM {_quote(source_view)}"
    )
    spark.sql(stmt)

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs["data"]
    output_type = config.get("output_type", "table")
    catalog = config.get("catalog", "")
    schema = config.get("schema", "")
    table_name = config.get("table_name", "")
    write_mode = config.get("write_mode", "overwrite")
    merge_keys: List[str] = [k for k in (config.get("merge_keys") or []) if k]
    format_value = config.get("format", "default")
    table_properties = config.get("table_properties", "") or ""

    if output_type == "file":
        _run_file(config, df, spark)
        return {}

    source_view = f"_lb_output_v2_src_{uuid.uuid4().hex}"
    df.createOrReplaceTempView(source_view)

    if output_type == "materialized_view":
        try:
            _run_materialized_view(config, source_view, spark)
        finally:
            spark.catalog.dropTempView(source_view)
        return {}

    if not table_name:
        raise ValueError("Output: 'table_name' is required")

    full_name = _qualified(catalog, schema, table_name)

    if write_mode == "merge" and not merge_keys:
        raise ValueError("Output: 'merge_keys' is required when write_mode is merge")

    try:
        args = _resolve_args(table_properties)

        using = _using_clause(format_value)
        tblprops = _tbl_properties_clause(
            table_properties, format_value, spark=spark, full_name=full_name
        )

        if write_mode == "append":
            create_stmt = (
                f"CREATE TABLE IF NOT EXISTS {full_name}{using}{tblprops} "
                f"AS SELECT * FROM {_quote(source_view)} WHERE 1=0"
            )
            spark.sql(create_stmt, args=args) if args else spark.sql(create_stmt)
            insert_stmt = (
                f"INSERT INTO {full_name} BY NAME "
                f"SELECT * FROM {_quote(source_view)}"
            )
            spark.sql(insert_stmt, args=args) if args else spark.sql(insert_stmt)
        elif write_mode == "merge":
            create_stmt = (
                f"CREATE TABLE IF NOT EXISTS {full_name}{using}{tblprops} "
                f"AS SELECT * FROM {_quote(source_view)} WHERE 1=0"
            )
            spark.sql(create_stmt, args=args) if args else spark.sql(create_stmt)
            on_clause = " AND ".join(
                f"t.{_quote(k)} = s.{_quote(k)}" for k in merge_keys
            )
            merge_stmt = (
                f"MERGE INTO {full_name} t "
                f"USING {_quote(source_view)} s "
                f"ON {on_clause} "
                f"WHEN MATCHED THEN UPDATE SET * "
                f"WHEN NOT MATCHED THEN INSERT *"
            )
            spark.sql(merge_stmt, args=args) if args else spark.sql(merge_stmt)
        else:
            stmt = (
                f"CREATE OR REPLACE TABLE {full_name}{using}{tblprops} "
                f"AS SELECT * FROM {_quote(source_view)}"
            )
            spark.sql(stmt, args=args) if args else spark.sql(stmt)
    except Exception as exc:
        if table_properties.strip():
            raise ValueError(
                "Output: failed to write the table. Check the 'table_properties' "
                "and 'schema' fields for invalid SQL. Underlying error: "
                f"{exc}"
            ) from exc
        raise
    finally:
        spark.catalog.dropTempView(source_view)
    return {}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {
    "output_type": "table",
    "catalog": "renewal_risk_lakehouse",
    "schema": "renewal_risk_dev",
    "table_name": "gold_account_360_summary",
    "write_mode": "overwrite"
}
config["meta_state"] = {"is_preview": False, "is_focused_preview": False, "is_disabled": False}
inputs = {
    "data": ctx["prep_account_360_summary_34ac3296.prepared_data"]
}
if config["meta_state"]["is_disabled"]:
    def _lb_limit_zero(value):
        if hasattr(value, "limit"):
            return value.limit(0)
        if isinstance(value, list):
            return [_lb_limit_zero(item) for item in value]
        return value
    inputs = {k: _lb_limit_zero(v) for k, v in inputs.items()}
if not config["meta_state"]["is_disabled"]:
    out = run(config, inputs, spark)
else:
    out = {}

In [0]:
"""
id: prep_ai_account_briefing_documents_6e21c82b
template: prepare
templateVersion: 1.0.0
name: prep_ai_account_briefing_documents
position:
  x: 3600
  y: 87.5
description:
  text: Create or rename columns to prepare account briefing document data.
  hash: f6d94993
previewCodeHash: bc9ca58152e64612
config:
  actions:
    - type: formula
      target: document_id
      expression: account_id
    - type: formula
      target: account_id
      expression: account_id
    - type: formula
      target: account_name
      expression: account_name
    - type: formula
      target: content
      expression: account_summary_text
    - type: formula
      target: source_table
      expression: "'renewal_risk_lakehouse.renewal_risk_dev.gold_account_360_summary'"
    - type: formula
      target: created_at
      expression: current_timestamp()
input:
  - node: prep_account_360_summary_34ac3296
    input_port: data
    output_port: prepared_data
"""

# generated from the system
from typing import Any, Callable, Dict, List
import pyspark.sql.functions as F

TEXT_CASE_FUNCTIONS: Dict[str, Callable] = {
    "lower": F.lower,
    "upper": F.upper,
    "title": F.initcap,
}

TRIM_FUNCTIONS: Dict[str, Callable] = {
    "both": F.trim,
    "left": F.ltrim,
    "right": F.rtrim,
}

VALID_MATCH_MODES = {"exact", "contains", "prefix", "suffix", "regex"}

def _require_column(df, column: str, action_type: str) -> None:
    if not column:
        raise ValueError(f"{action_type}: 'column' is required")
    if column not in df.columns:
        raise ValueError(
            f"{action_type}: column '{column}' not found in input data. "
            f"Available columns: {df.columns}"
        )

def _column_dtype(df, column: str) -> str:
    return df.schema[column].dataType.simpleString()

def _quoted_ident(name: str) -> str:
    return "`" + name.replace("`", "``") + "`"

def _col(name: str):
    """Reference a column by exact name.

    F.col parses its argument as a column expression, so a name containing
    a '.' (e.g. "a.b") is otherwise read as struct-field access and fails
    even though the column exists. Backtick-quoting forces an exact-name
    lookup; quoting plain names is harmless.
    """
    return F.col(_quoted_ident(name))

def _coerced_lit(value: Any, target_dtype: str):
    """Wrap a user-provided value as a literal cast to the target column's type.

    UI always feeds text per the operator spec; this is where the
    type coercion happens so the per-action UI stays simple.
    """
    return F.lit(value).cast(target_dtype)

def _apply_formula(df, action: Dict[str, Any]):
    target = action.get("target", "")
    expression = action.get("expression", "")
    if not target:
        raise ValueError("formula: 'target' is required")
    if not expression:
        raise ValueError("formula: 'expression' is required")
    return df.withColumn(target, F.expr(expression))

def _apply_cast(df, action: Dict[str, Any]):
    column = action.get("column", "")
    to_type = action.get("to", "")
    on_error = action.get("on_error", "null")
    _require_column(df, column, "cast")
    if not to_type:
        raise ValueError("cast: 'to' (target type) is required")
    if on_error == "null":
        return df.withColumn(
            column,
            F.expr(f"try_cast({_quoted_ident(column)} as {to_type})"),
        )
    return df.withColumn(column, _col(column).cast(to_type))

def _apply_replace_value(df, action: Dict[str, Any]):
    column = action.get("column", "")
    match_mode = action.get("match_mode", "exact")
    match = action.get("match", "")
    with_val = action.get("with", "")
    case_sensitive = bool(action.get("case_sensitive", False))
    _require_column(df, column, "replace_value")
    if match_mode not in VALID_MATCH_MODES:
        raise ValueError(
            f"replace_value: unsupported match_mode '{match_mode}'. "
            f"Choose one of: {sorted(VALID_MATCH_MODES)}"
        )
    dtype = _column_dtype(df, column)
    col_as_string = _col(column).cast("string")
    match_lit = F.lit(match)
    if case_sensitive:
        cmp_col = col_as_string
        cmp_lit = match_lit
    else:
        cmp_col = F.lower(col_as_string)
        cmp_lit = F.lower(match_lit)

    if match_mode == "exact":
        predicate = cmp_col == cmp_lit
    elif match_mode == "contains":
        predicate = cmp_col.contains(cmp_lit)
    elif match_mode == "prefix":
        predicate = cmp_col.startswith(cmp_lit)
    elif match_mode == "suffix":
        predicate = cmp_col.endswith(cmp_lit)
    else:  # regex
        pattern = match if case_sensitive else f"(?i){match}"
        predicate = col_as_string.rlike(pattern)

    replacement = _coerced_lit(with_val, dtype)
    return df.withColumn(
        column,
        F.when(predicate, replacement).otherwise(_col(column)),
    )

def _apply_fill_null(df, action: Dict[str, Any]):
    column = action.get("column", "")
    with_val = action.get("with", "")
    _require_column(df, column, "fill_null")
    dtype = _column_dtype(df, column)
    replacement = _coerced_lit(with_val, dtype)
    return df.withColumn(
        column,
        F.when(_col(column).isNull(), replacement).otherwise(_col(column)),
    )

def _apply_text_case(df, action: Dict[str, Any]):
    column = action.get("column", "")
    case = action.get("case", "")
    _require_column(df, column, "text_case")
    fn = TEXT_CASE_FUNCTIONS.get(case)
    if fn is None:
        raise ValueError(
            f"text_case: unsupported case '{case}'. "
            f"Choose one of: {sorted(TEXT_CASE_FUNCTIONS.keys())}"
        )
    return df.withColumn(column, fn(_col(column)))

def _apply_trim(df, action: Dict[str, Any]):
    column = action.get("column", "")
    side = action.get("side", "both")
    _require_column(df, column, "trim")
    fn = TRIM_FUNCTIONS.get(side)
    if fn is None:
        raise ValueError(
            f"trim: unsupported side '{side}'. "
            f"Choose one of: {sorted(TRIM_FUNCTIONS.keys())}"
        )
    return df.withColumn(column, fn(_col(column)))

def _apply_regex_replace(df, action: Dict[str, Any]):
    column = action.get("column", "")
    pattern = action.get("pattern", "")
    replacement = action.get("replacement", "")
    _require_column(df, column, "regex_replace")
    return df.withColumn(
        column,
        F.regexp_replace(_col(column), pattern, replacement),
    )

def _apply_extract(df, action: Dict[str, Any]):
    column = action.get("column", "")
    pattern = action.get("pattern", "")
    target = action.get("target") or column
    group_raw = action.get("group", 0)
    _require_column(df, column, "extract")
    try:
        group_idx = int(group_raw)
    except (TypeError, ValueError) as exc:
        raise ValueError(
            f"extract: 'group' must be an integer, got {group_raw!r}"
        ) from exc
    return df.withColumn(
        target,
        F.regexp_extract(_col(column), pattern, group_idx),
    )

def _apply_parse_date(df, action: Dict[str, Any]):
    column = action.get("column", "")
    kind = action.get("kind", "date")
    fmt = action.get("format") or None
    on_error = action.get("on_error", "null")
    _require_column(df, column, "parse_date")
    if kind not in ("date", "timestamp"):
        raise ValueError(
            f"parse_date: unsupported kind '{kind}'. Choose date or timestamp."
        )
    # PySpark exposes F.try_to_timestamp from Spark 3.5 but only adds
    # F.try_to_date in Spark 4.0. Current Databricks Runtimes still ship
    # Spark 3.5, where referencing F.try_to_date raises AttributeError
    # before any data is touched. Route the null-on-error path through
    # primitives that exist in both 3.5 and 4.0:
    #   - F.try_to_timestamp  (PySpark 3.5+, returns NULL under ANSI too)
    #   - SQL try_cast        (Spark 3.5+, returns NULL under ANSI too)
    #   - F.to_date           (PySpark 3.5+; returns NULL on parse failure
    #                          under the default non-ANSI mode, which is
    #                          the Databricks default. Under ANSI mode it
    #                          raises — accepted limitation until
    #                          try_to_date lands on every supported DBR.)
    if on_error == "null":
        if kind == "timestamp":
            if fmt:
                return df.withColumn(
                    column, F.try_to_timestamp(_col(column), F.lit(fmt))
                )
            return df.withColumn(column, F.try_to_timestamp(_col(column)))
        # kind == "date"
        if fmt:
            return df.withColumn(column, F.to_date(_col(column), fmt))
        return df.withColumn(
            column, F.expr(f"try_cast({_quoted_ident(column)} as date)")
        )
    # on_error == "error": surface parse failures as Spark exceptions.
    fn = F.to_date if kind == "date" else F.to_timestamp
    if fmt:
        return df.withColumn(column, fn(_col(column), fmt))
    return df.withColumn(column, fn(_col(column)))

ACTION_DISPATCH: Dict[str, Callable] = {
    "formula": _apply_formula,
    "cast": _apply_cast,
    "replace_value": _apply_replace_value,
    "fill_null": _apply_fill_null,
    "text_case": _apply_text_case,
    "trim": _apply_trim,
    "regex_replace": _apply_regex_replace,
    "extract": _apply_extract,
    "parse_date": _apply_parse_date,
}

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs.get("data")
    actions: List[Dict[str, Any]] = config.get("actions", []) or []

    if not actions:
        return {"prepared_data": df}

    for index, action in enumerate(actions):
        if not isinstance(action, dict):
            raise ValueError(
                f"actions[{index}]: expected an object, got {type(action).__name__}"
            )
        if action.get("enabled", True) is False:
            continue
        action_type = action.get("type", "")
        fn = ACTION_DISPATCH.get(action_type)
        if fn is None:
            raise ValueError(
                f"actions[{index}]: unsupported action type {action_type!r}. "
                f"Choose one of: {sorted(ACTION_DISPATCH.keys())}"
            )
        df = fn(df, action)
    return {"prepared_data": df}

# generated from the system
if "ld_display_outputs" not in globals():
    try:
        _ld_param = dbutils.widgets.getAll().get("ld_display_outputs")
        if _ld_param is not None and str(_ld_param).strip() != "":
            globals()["ld_display_outputs"] = str(_ld_param).strip().lower() not in ("false", "0", "no", "off")
        else:
            try:
                from dbruntime.databricks_repl_context import get_context
                globals()["ld_display_outputs"] = not get_context().isInJob
            except Exception:
                globals()["ld_display_outputs"] = True
    except Exception:
        globals()["ld_display_outputs"] = False
if "ld_display_outputs_for" not in globals():
    try:
        _ld_for = dbutils.widgets.getAll().get("ld_display_outputs_for")
        globals()["ld_display_outputs_for"] = frozenset(_p.strip() for _p in str(_ld_for).split(",") if _p.strip()) if _ld_for is not None else frozenset()
    except Exception:
        globals()["ld_display_outputs_for"] = frozenset()
ctx = globals().setdefault("ctx", {})
config = {
    "actions": [
        {
            "type": "formula",
            "target": "document_id",
            "expression": "account_id"
        },
        {
            "type": "formula",
            "target": "account_id",
            "expression": "account_id"
        },
        {
            "type": "formula",
            "target": "account_name",
            "expression": "account_name"
        },
        {
            "type": "formula",
            "target": "content",
            "expression": "account_summary_text"
        },
        {
            "type": "formula",
            "target": "source_table",
            "expression": "'renewal_risk_lakehouse.renewal_risk_dev.gold_account_360_summary'"
        },
        {
            "type": "formula",
            "target": "created_at",
            "expression": "current_timestamp()"
        }
    ]
}
config["meta_state"] = {"is_preview": False, "is_focused_preview": False, "is_disabled": False}
inputs = {
    "data": ctx["prep_account_360_summary_34ac3296.prepared_data"]
}
if config["meta_state"]["is_disabled"]:
    def _lb_limit_zero(value):
        if hasattr(value, "limit"):
            return value.limit(0)
        if isinstance(value, list):
            return [_lb_limit_zero(item) for item in value]
        return value
    inputs = {k: _lb_limit_zero(v) for k, v in inputs.items()}
out = run(config, inputs, spark)
if config["meta_state"]["is_disabled"]:
    out = {k: _lb_limit_zero(v) for k, v in out.items()}
ctx["prep_ai_account_briefing_documents_6e21c82b.prepared_data"] = out["prepared_data"]
if globals().get("ld_display_outputs", False) or "prep_ai_account_briefing_documents_6e21c82b" in globals().get("ld_display_outputs_for", frozenset()):
    display(ctx["prep_ai_account_briefing_documents_6e21c82b.prepared_data"])

In [0]:
"""
id: output_csm_priority_queue_ef29cd90
template: output
templateVersion: 3.0.0
name: gold_csm_priority_queue
position:
  x: 3900
  y: 397.5
description:
  text: Overwrite the specified table with new data.
  hash: 6fa2f6a0
config:
  output_type: table
  catalog: renewal_risk_lakehouse
  schema: renewal_risk_dev
  table_name: gold_csm_priority_queue
  write_mode: overwrite
input:
  - node: sort_csm_priority_e0479c5b
    input_port: data
    output_port: sorted_data
"""

# generated from the system
import uuid
from typing import Dict, Any, List

DELTA_COLUMN_MAPPING_PROP = "'delta.columnMapping.mode' = 'id'"

UNIFORM_PROPS = (
    "'delta.enableIcebergCompatV2' = 'true', "
    "'delta.universalFormat.enabledFormats' = 'iceberg', "
    + DELTA_COLUMN_MAPPING_PROP
)

def _quote(name: str) -> str:
    if len(name) >= 2 and name.startswith("`") and name.endswith("`"):
        name = name[1:-1].replace("``", "`")
    return "`" + name.replace("`", "``") + "`"

def _qualified(catalog: str, schema: str, table: str) -> str:
    if not table:
        raise ValueError("Output: 'table_name' is required")
    if catalog and not schema:
        raise ValueError("Output: 'schema' is required when 'catalog' is set")
    parts = [_quote(p) for p in (catalog, schema, table) if p]
    return ".".join(parts)

def _table_exists(spark, qualified_name: str) -> bool:
    try:
        return spark.catalog.tableExists(qualified_name)
    except Exception:
        return False

def _existing_column_mapping_mode(spark, qualified_name: str) -> str:
    if spark is None or not qualified_name:
        return ""
    try:
        rows = spark.sql(
            f"SHOW TBLPROPERTIES {qualified_name} ('delta.columnMapping.mode')"
        ).collect()
        if rows:
            value = str(rows[0]["value"])
            if value in ("name", "id"):
                return value
    except Exception:
        pass
    return ""

def _column_mapping_prop(spark, full_name: str) -> str:
    if spark is None or not full_name:
        return DELTA_COLUMN_MAPPING_PROP
    if not _table_exists(spark, full_name):
        return DELTA_COLUMN_MAPPING_PROP
    existing = _existing_column_mapping_mode(spark, full_name)
    if existing:
        return f"'delta.columnMapping.mode' = '{existing}'"
    return ""

def _tbl_properties_clause(
    user_props: str, format_value: str, spark=None, full_name: str = ""
) -> str:
    parts: List[str] = []
    table_exists = bool(
        spark is not None and full_name and _table_exists(spark, full_name)
    )
    column_mapping = _column_mapping_prop(spark, full_name)
    if format_value == "uniform":
        uniform_base = (
            "'delta.enableIcebergCompatV2' = 'true', "
            "'delta.universalFormat.enabledFormats' = 'iceberg'"
        )
        if column_mapping:
            parts.append(uniform_base + ", " + column_mapping)
        elif not table_exists:
            parts.append(UNIFORM_PROPS)
        else:
            parts.append(uniform_base)
    elif format_value in ("delta", "default") and column_mapping:
        parts.append(column_mapping)
    cleaned = (user_props or "").strip().rstrip(",").strip()
    if cleaned:
        parts.append(cleaned)
    if not parts:
        return ""
    return " TBLPROPERTIES (" + ", ".join(parts) + ")"

def _using_clause(format_value: str) -> str:
    if format_value == "iceberg":
        return " USING ICEBERG"
    if format_value in ("delta", "uniform"):
        return " USING DELTA"
    return ""

def _resolve_args(table_properties: str) -> Dict[str, str]:
    import re

    param_names = set(re.findall(r"(?<!:):(\w+)", table_properties or ""))
    if not param_names:
        return {}
    try:
        widgets = dbutils.widgets.getAll()
    except NameError:
        return {}
    return {name: widgets[name] for name in param_names if name in widgets}

def _comment_clause(comment: str) -> str:
    cleaned = (comment or "").strip()
    if not cleaned:
        return ""
    escaped = cleaned.replace("'", "''")
    return f" COMMENT '{escaped}'"

def _cluster_by_clause(mode: str, column: str) -> str:
    if mode == "auto":
        return " CLUSTER BY AUTO"
    if mode == "column" and column:
        return f" CLUSTER BY ({_quote(column)})"
    return ""

SUPPORTED_FILE_TYPES = {"csv", "json", "excel"}

FILE_EXTENSIONS = {"csv": "csv", "json": "json", "excel": "xlsx"}

MAX_SINGLE_FILE_ROWS = 1_000_000

MAX_EXCEL_CELLS = 5_000_000

def _excel_row_cap(num_cols: int) -> int:
    return min(MAX_SINGLE_FILE_ROWS, max(1, MAX_EXCEL_CELLS // max(1, num_cols)))

_FORMULA_TRIGGERS = "=+-@\t\r"

def _neutralize_formula(value):
    if isinstance(value, str) and value and value[0] in _FORMULA_TRIGGERS:
        return "'" + value
    return value

def _file_path(catalog: str, schema: str, volume: str, file_name: str) -> str:
    if not file_name:
        raise ValueError("Output: 'file_name' is required for a file output")
    if catalog and schema and volume:
        if "/" in file_name or ".." in file_name:
            raise ValueError(
                f"Output: invalid 'file_name' {file_name!r}: it is the final "
                "path segment under the volume and cannot contain '/' or '..'."
            )
        schema_segment = schema.replace(".", "/")
        return f"/Volumes/{catalog}/{schema_segment}/{volume}/{file_name}"
    return file_name

def _with_extension(file_name: str, file_type: str) -> str:
    suffix = f".{FILE_EXTENSIONS.get(file_type, file_type)}"
    if not file_name or file_name.lower().endswith(suffix):
        return file_name
    return file_name + suffix

def _write_single_file(rows, columns, file_type: str, path: str, append: bool) -> None:
    import csv
    import json
    import os
    import shutil
    import tempfile

    def _remove(target: str) -> None:
        if os.path.isdir(target):
            shutil.rmtree(target)
        elif os.path.exists(target):
            os.remove(target)

    def _is_spark_output_dir(target: str) -> bool:
        return os.path.isfile(os.path.join(target, "_SUCCESS"))

    def _stage(write_body) -> None:
        from databricks.sdk import WorkspaceClient

        fd, local_tmp = tempfile.mkstemp(suffix=".lb-output-stage")
        os.close(fd)
        try:
            write_body(local_tmp)
            _remove(path)
            with open(local_tmp, "rb") as f:
                WorkspaceClient().files.upload(path, f, overwrite=True)
        finally:
            if os.path.isfile(local_tmp):
                os.remove(local_tmp)

    if os.path.isdir(path) and not _is_spark_output_dir(path):
        raise ValueError(
            f"Output: cannot write file to {path}: a directory already "
            f"exists at that path."
        )
    appending = append and os.path.isfile(path)
    if file_type == "csv":
        existing_data_rows = 0
        has_existing_header = False
        header = list(columns)
        if appending:
            with open(path, newline="", encoding="utf-8") as handle:
                reader = csv.reader(handle)
                first = next(reader, None)
                if first is not None:
                    header = first
                    has_existing_header = True
                    existing_data_rows = sum(1 for _ in reader)
        if has_existing_header and sorted(header) != sorted(columns):
            raise ValueError(
                f"Output: cannot append to {path}: the data's columns "
                f"{sorted(columns)} do not match the existing file's "
                f"columns {sorted(header)}."
            )
        if existing_data_rows + len(rows) > MAX_SINGLE_FILE_ROWS:
            raise ValueError(
                f"Output: appending {len(rows):,} rows would grow {path} past "
                f"the single-file export limit ({MAX_SINGLE_FILE_ROWS:,} rows) "
                f"— write to a table instead."
            )

        def _write_csv(tmp_path: str) -> None:
            with open(tmp_path, "w", newline="", encoding="utf-8") as out:
                if has_existing_header:
                    with open(path, newline="", encoding="utf-8") as src:
                        shutil.copyfileobj(src, out)
                else:
                    csv.writer(out).writerow(header)
                writer = csv.writer(out)
                for row in rows:
                    writer.writerow([_neutralize_formula(row[c]) for c in header])

        _stage(_write_csv)
        return

    if file_type == "excel":
        import io

        try:
            import openpyxl
        except ModuleNotFoundError as exc:
            raise ModuleNotFoundError(
                "Writing Excel files requires the 'openpyxl' package, which "
                "is not installed in this environment. Add "
                "openpyxl==3.1.5 to the notebook environment "
                "and apply it, then run again."
            ) from exc

        import pandas as pd
        import re as _re

        try:
            from openpyxl.cell.cell import ILLEGAL_CHARACTERS_RE as _lb_illegal
        except ImportError:
            _lb_illegal = _re.compile(r"[\x00-\x08\x0b\x0c\x0e-\x1f]")

        def _excel_safe(value):
            if isinstance(value, str):
                return _lb_illegal.sub("", value)
            if isinstance(value, (int, float, bool, type(None))):
                return value
            return _lb_illegal.sub("", str(value))

        header = list(columns)
        source_keys = list(columns)
        existing_values: List = []
        if appending:
            workbook = openpyxl.load_workbook(path, read_only=True)
            try:
                sheet = workbook.active
                row_iter = sheet.iter_rows(values_only=True)
                first = next(row_iter, None)
                if first is not None:
                    header = list(first)
                    if sorted(str(c) for c in header) != sorted(
                        str(_excel_safe(c)) for c in columns
                    ):
                        raise ValueError(
                            f"Output: cannot append to {path}: the data's columns "
                            f"{[str(_excel_safe(c)) for c in columns]} do not match the existing "
                            f"file's columns {[str(c) for c in header]}."
                        )
                    raw_by_safe: Dict[str, Any] = {}
                    for c in columns:
                        raw_by_safe.setdefault(str(_excel_safe(c)), c)
                    source_keys = [raw_by_safe[str(c)] for c in header]
                    row_cap = _excel_row_cap(len(header))
                    for record in row_iter:
                        if len(existing_values) + len(rows) >= row_cap:
                            raise ValueError(
                                f"Output: appending {len(rows):,} rows would grow {path} past "
                                f"the single-file excel export limit ({MAX_EXCEL_CELLS:,} cells "
                                f"at {len(header):,} columns) — write to a table instead."
                            )
                        existing_values.append(list(record))
            finally:
                workbook.close()

        combined = existing_values + [
            [_neutralize_formula(_excel_safe(row[k])) for k in source_keys] for row in rows
        ]
        new_df = pd.DataFrame(combined, columns=[_excel_safe(c) for c in header])

        def _write_excel(tmp_path: str) -> None:
            buffer = io.BytesIO()
            new_df.to_excel(buffer, index=False, engine="openpyxl")
            with open(tmp_path, "wb") as out:
                out.write(buffer.getvalue())

        _stage(_write_excel)
        return

    records = []
    if appending:
        with open(path, encoding="utf-8") as handle:
            existing = json.load(handle)
        records = existing if isinstance(existing, list) else [existing]
        if records:
            existing_columns = (
                list(records[0].keys()) if isinstance(records[0], dict) else []
            )
            if existing_columns and sorted(existing_columns) != sorted(columns):
                raise ValueError(
                    f"Output: cannot append to {path}: the data's columns "
                    f"{sorted(columns)} do not match the existing file's "
                    f"columns {sorted(existing_columns)}."
                )
    if len(records) + len(rows) > MAX_SINGLE_FILE_ROWS:
        raise ValueError(
            f"Output: appending {len(rows):,} rows would grow {path} past "
            f"the single-file export limit ({MAX_SINGLE_FILE_ROWS:,} rows) "
            f"— write to a table instead."
        )
    records.extend({c: row[c] for c in columns} for row in rows)

    def _write_json(tmp_path: str) -> None:
        with open(tmp_path, "w", encoding="utf-8") as handle:
            json.dump(records, handle, default=str, indent=2)

    _stage(_write_json)

def _run_file(config: Dict[str, Any], df, spark) -> None:
    file_type = config.get("file_type", "csv")
    if file_type not in SUPPORTED_FILE_TYPES:
        raise ValueError(
            f"Output: file_type '{file_type}' is not yet supported. Use csv, json, or excel."
        )
    path = _file_path(
        config.get("catalog", ""),
        config.get("schema", ""),
        config.get("volume", ""),
        _with_extension(config.get("file_name", ""), file_type),
    )
    append = config.get("write_mode", "overwrite") == "append"
    if file_type == "excel":
        row_cap = _excel_row_cap(len(df.columns))
        rows = df.limit(row_cap + 1).collect()
        if len(rows) > row_cap:
            raise ValueError(
                f"Output: result too large for single-file excel export "
                f"(over the {MAX_EXCEL_CELLS:,}-cell limit at {len(df.columns):,} "
                f"columns) — write to a table instead."
            )
    else:
        rows = df.limit(MAX_SINGLE_FILE_ROWS + 1).collect()
        if len(rows) > MAX_SINGLE_FILE_ROWS:
            raise ValueError(
                f"Output: result too large for single-file export "
                f"(over {MAX_SINGLE_FILE_ROWS:,} rows) — write to a table instead."
            )
    _write_single_file(rows, df.columns, file_type, path, append)

def _run_materialized_view(
    config: Dict[str, Any], source_view: str, spark
) -> None:
    catalog = config.get("catalog", "")
    schema = config.get("schema", "")
    view_name = config.get("view_name", "")
    if not view_name:
        raise ValueError("Output: 'view_name' is required for a materialized view")
    full_name = _qualified(catalog, schema, view_name)
    cluster_by = _cluster_by_clause(
        config.get("cluster_by_mode", "auto"), config.get("cluster_by_column", "") or ""
    )
    comment = _comment_clause(config.get("comment", "") or "")
    stmt = (
        f"CREATE OR REFRESH MATERIALIZED VIEW {full_name}{cluster_by}{comment} "
        f"AS SELECT * FROM {_quote(source_view)}"
    )
    spark.sql(stmt)

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs["data"]
    output_type = config.get("output_type", "table")
    catalog = config.get("catalog", "")
    schema = config.get("schema", "")
    table_name = config.get("table_name", "")
    write_mode = config.get("write_mode", "overwrite")
    merge_keys: List[str] = [k for k in (config.get("merge_keys") or []) if k]
    format_value = config.get("format", "default")
    table_properties = config.get("table_properties", "") or ""

    if output_type == "file":
        _run_file(config, df, spark)
        return {}

    source_view = f"_lb_output_v2_src_{uuid.uuid4().hex}"
    df.createOrReplaceTempView(source_view)

    if output_type == "materialized_view":
        try:
            _run_materialized_view(config, source_view, spark)
        finally:
            spark.catalog.dropTempView(source_view)
        return {}

    if not table_name:
        raise ValueError("Output: 'table_name' is required")

    full_name = _qualified(catalog, schema, table_name)

    if write_mode == "merge" and not merge_keys:
        raise ValueError("Output: 'merge_keys' is required when write_mode is merge")

    try:
        args = _resolve_args(table_properties)

        using = _using_clause(format_value)
        tblprops = _tbl_properties_clause(
            table_properties, format_value, spark=spark, full_name=full_name
        )

        if write_mode == "append":
            create_stmt = (
                f"CREATE TABLE IF NOT EXISTS {full_name}{using}{tblprops} "
                f"AS SELECT * FROM {_quote(source_view)} WHERE 1=0"
            )
            spark.sql(create_stmt, args=args) if args else spark.sql(create_stmt)
            insert_stmt = (
                f"INSERT INTO {full_name} BY NAME "
                f"SELECT * FROM {_quote(source_view)}"
            )
            spark.sql(insert_stmt, args=args) if args else spark.sql(insert_stmt)
        elif write_mode == "merge":
            create_stmt = (
                f"CREATE TABLE IF NOT EXISTS {full_name}{using}{tblprops} "
                f"AS SELECT * FROM {_quote(source_view)} WHERE 1=0"
            )
            spark.sql(create_stmt, args=args) if args else spark.sql(create_stmt)
            on_clause = " AND ".join(
                f"t.{_quote(k)} = s.{_quote(k)}" for k in merge_keys
            )
            merge_stmt = (
                f"MERGE INTO {full_name} t "
                f"USING {_quote(source_view)} s "
                f"ON {on_clause} "
                f"WHEN MATCHED THEN UPDATE SET * "
                f"WHEN NOT MATCHED THEN INSERT *"
            )
            spark.sql(merge_stmt, args=args) if args else spark.sql(merge_stmt)
        else:
            stmt = (
                f"CREATE OR REPLACE TABLE {full_name}{using}{tblprops} "
                f"AS SELECT * FROM {_quote(source_view)}"
            )
            spark.sql(stmt, args=args) if args else spark.sql(stmt)
    except Exception as exc:
        if table_properties.strip():
            raise ValueError(
                "Output: failed to write the table. Check the 'table_properties' "
                "and 'schema' fields for invalid SQL. Underlying error: "
                f"{exc}"
            ) from exc
        raise
    finally:
        spark.catalog.dropTempView(source_view)
    return {}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {
    "output_type": "table",
    "catalog": "renewal_risk_lakehouse",
    "schema": "renewal_risk_dev",
    "table_name": "gold_csm_priority_queue",
    "write_mode": "overwrite"
}
config["meta_state"] = {"is_preview": False, "is_focused_preview": False, "is_disabled": False}
inputs = {
    "data": ctx["sort_csm_priority_e0479c5b.sorted_data"]
}
if config["meta_state"]["is_disabled"]:
    def _lb_limit_zero(value):
        if hasattr(value, "limit"):
            return value.limit(0)
        if isinstance(value, list):
            return [_lb_limit_zero(item) for item in value]
        return value
    inputs = {k: _lb_limit_zero(v) for k, v in inputs.items()}
if not config["meta_state"]["is_disabled"]:
    out = run(config, inputs, spark)
else:
    out = {}

In [0]:
"""
id: output_ai_account_briefing_documents_a2b95d5e
template: output
templateVersion: 3.0.0
name: ai_account_briefing_documents
position:
  x: 3900
  y: 87.5
description:
  text: Overwrite the table with new data in the specified catalog and schema.
  hash: 403e97bf
config:
  output_type: table
  catalog: renewal_risk_lakehouse
  schema: renewal_risk_dev
  table_name: ai_account_briefing_documents
  write_mode: overwrite
input:
  - node: prep_ai_account_briefing_documents_6e21c82b
    input_port: data
    output_port: prepared_data
"""

# generated from the system
import uuid
from typing import Dict, Any, List

DELTA_COLUMN_MAPPING_PROP = "'delta.columnMapping.mode' = 'id'"

UNIFORM_PROPS = (
    "'delta.enableIcebergCompatV2' = 'true', "
    "'delta.universalFormat.enabledFormats' = 'iceberg', "
    + DELTA_COLUMN_MAPPING_PROP
)

def _quote(name: str) -> str:
    if len(name) >= 2 and name.startswith("`") and name.endswith("`"):
        name = name[1:-1].replace("``", "`")
    return "`" + name.replace("`", "``") + "`"

def _qualified(catalog: str, schema: str, table: str) -> str:
    if not table:
        raise ValueError("Output: 'table_name' is required")
    if catalog and not schema:
        raise ValueError("Output: 'schema' is required when 'catalog' is set")
    parts = [_quote(p) for p in (catalog, schema, table) if p]
    return ".".join(parts)

def _table_exists(spark, qualified_name: str) -> bool:
    try:
        return spark.catalog.tableExists(qualified_name)
    except Exception:
        return False

def _existing_column_mapping_mode(spark, qualified_name: str) -> str:
    if spark is None or not qualified_name:
        return ""
    try:
        rows = spark.sql(
            f"SHOW TBLPROPERTIES {qualified_name} ('delta.columnMapping.mode')"
        ).collect()
        if rows:
            value = str(rows[0]["value"])
            if value in ("name", "id"):
                return value
    except Exception:
        pass
    return ""

def _column_mapping_prop(spark, full_name: str) -> str:
    if spark is None or not full_name:
        return DELTA_COLUMN_MAPPING_PROP
    if not _table_exists(spark, full_name):
        return DELTA_COLUMN_MAPPING_PROP
    existing = _existing_column_mapping_mode(spark, full_name)
    if existing:
        return f"'delta.columnMapping.mode' = '{existing}'"
    return ""

def _tbl_properties_clause(
    user_props: str, format_value: str, spark=None, full_name: str = ""
) -> str:
    parts: List[str] = []
    table_exists = bool(
        spark is not None and full_name and _table_exists(spark, full_name)
    )
    column_mapping = _column_mapping_prop(spark, full_name)
    if format_value == "uniform":
        uniform_base = (
            "'delta.enableIcebergCompatV2' = 'true', "
            "'delta.universalFormat.enabledFormats' = 'iceberg'"
        )
        if column_mapping:
            parts.append(uniform_base + ", " + column_mapping)
        elif not table_exists:
            parts.append(UNIFORM_PROPS)
        else:
            parts.append(uniform_base)
    elif format_value in ("delta", "default") and column_mapping:
        parts.append(column_mapping)
    cleaned = (user_props or "").strip().rstrip(",").strip()
    if cleaned:
        parts.append(cleaned)
    if not parts:
        return ""
    return " TBLPROPERTIES (" + ", ".join(parts) + ")"

def _using_clause(format_value: str) -> str:
    if format_value == "iceberg":
        return " USING ICEBERG"
    if format_value in ("delta", "uniform"):
        return " USING DELTA"
    return ""

def _resolve_args(table_properties: str) -> Dict[str, str]:
    import re

    param_names = set(re.findall(r"(?<!:):(\w+)", table_properties or ""))
    if not param_names:
        return {}
    try:
        widgets = dbutils.widgets.getAll()
    except NameError:
        return {}
    return {name: widgets[name] for name in param_names if name in widgets}

def _comment_clause(comment: str) -> str:
    cleaned = (comment or "").strip()
    if not cleaned:
        return ""
    escaped = cleaned.replace("'", "''")
    return f" COMMENT '{escaped}'"

def _cluster_by_clause(mode: str, column: str) -> str:
    if mode == "auto":
        return " CLUSTER BY AUTO"
    if mode == "column" and column:
        return f" CLUSTER BY ({_quote(column)})"
    return ""

SUPPORTED_FILE_TYPES = {"csv", "json", "excel"}

FILE_EXTENSIONS = {"csv": "csv", "json": "json", "excel": "xlsx"}

MAX_SINGLE_FILE_ROWS = 1_000_000

MAX_EXCEL_CELLS = 5_000_000

def _excel_row_cap(num_cols: int) -> int:
    return min(MAX_SINGLE_FILE_ROWS, max(1, MAX_EXCEL_CELLS // max(1, num_cols)))

_FORMULA_TRIGGERS = "=+-@\t\r"

def _neutralize_formula(value):
    if isinstance(value, str) and value and value[0] in _FORMULA_TRIGGERS:
        return "'" + value
    return value

def _file_path(catalog: str, schema: str, volume: str, file_name: str) -> str:
    if not file_name:
        raise ValueError("Output: 'file_name' is required for a file output")
    if catalog and schema and volume:
        if "/" in file_name or ".." in file_name:
            raise ValueError(
                f"Output: invalid 'file_name' {file_name!r}: it is the final "
                "path segment under the volume and cannot contain '/' or '..'."
            )
        schema_segment = schema.replace(".", "/")
        return f"/Volumes/{catalog}/{schema_segment}/{volume}/{file_name}"
    return file_name

def _with_extension(file_name: str, file_type: str) -> str:
    suffix = f".{FILE_EXTENSIONS.get(file_type, file_type)}"
    if not file_name or file_name.lower().endswith(suffix):
        return file_name
    return file_name + suffix

def _write_single_file(rows, columns, file_type: str, path: str, append: bool) -> None:
    import csv
    import json
    import os
    import shutil
    import tempfile

    def _remove(target: str) -> None:
        if os.path.isdir(target):
            shutil.rmtree(target)
        elif os.path.exists(target):
            os.remove(target)

    def _is_spark_output_dir(target: str) -> bool:
        return os.path.isfile(os.path.join(target, "_SUCCESS"))

    def _stage(write_body) -> None:
        from databricks.sdk import WorkspaceClient

        fd, local_tmp = tempfile.mkstemp(suffix=".lb-output-stage")
        os.close(fd)
        try:
            write_body(local_tmp)
            _remove(path)
            with open(local_tmp, "rb") as f:
                WorkspaceClient().files.upload(path, f, overwrite=True)
        finally:
            if os.path.isfile(local_tmp):
                os.remove(local_tmp)

    if os.path.isdir(path) and not _is_spark_output_dir(path):
        raise ValueError(
            f"Output: cannot write file to {path}: a directory already "
            f"exists at that path."
        )
    appending = append and os.path.isfile(path)
    if file_type == "csv":
        existing_data_rows = 0
        has_existing_header = False
        header = list(columns)
        if appending:
            with open(path, newline="", encoding="utf-8") as handle:
                reader = csv.reader(handle)
                first = next(reader, None)
                if first is not None:
                    header = first
                    has_existing_header = True
                    existing_data_rows = sum(1 for _ in reader)
        if has_existing_header and sorted(header) != sorted(columns):
            raise ValueError(
                f"Output: cannot append to {path}: the data's columns "
                f"{sorted(columns)} do not match the existing file's "
                f"columns {sorted(header)}."
            )
        if existing_data_rows + len(rows) > MAX_SINGLE_FILE_ROWS:
            raise ValueError(
                f"Output: appending {len(rows):,} rows would grow {path} past "
                f"the single-file export limit ({MAX_SINGLE_FILE_ROWS:,} rows) "
                f"— write to a table instead."
            )

        def _write_csv(tmp_path: str) -> None:
            with open(tmp_path, "w", newline="", encoding="utf-8") as out:
                if has_existing_header:
                    with open(path, newline="", encoding="utf-8") as src:
                        shutil.copyfileobj(src, out)
                else:
                    csv.writer(out).writerow(header)
                writer = csv.writer(out)
                for row in rows:
                    writer.writerow([_neutralize_formula(row[c]) for c in header])

        _stage(_write_csv)
        return

    if file_type == "excel":
        import io

        try:
            import openpyxl
        except ModuleNotFoundError as exc:
            raise ModuleNotFoundError(
                "Writing Excel files requires the 'openpyxl' package, which "
                "is not installed in this environment. Add "
                "openpyxl==3.1.5 to the notebook environment "
                "and apply it, then run again."
            ) from exc

        import pandas as pd
        import re as _re

        try:
            from openpyxl.cell.cell import ILLEGAL_CHARACTERS_RE as _lb_illegal
        except ImportError:
            _lb_illegal = _re.compile(r"[\x00-\x08\x0b\x0c\x0e-\x1f]")

        def _excel_safe(value):
            if isinstance(value, str):
                return _lb_illegal.sub("", value)
            if isinstance(value, (int, float, bool, type(None))):
                return value
            return _lb_illegal.sub("", str(value))

        header = list(columns)
        source_keys = list(columns)
        existing_values: List = []
        if appending:
            workbook = openpyxl.load_workbook(path, read_only=True)
            try:
                sheet = workbook.active
                row_iter = sheet.iter_rows(values_only=True)
                first = next(row_iter, None)
                if first is not None:
                    header = list(first)
                    if sorted(str(c) for c in header) != sorted(
                        str(_excel_safe(c)) for c in columns
                    ):
                        raise ValueError(
                            f"Output: cannot append to {path}: the data's columns "
                            f"{[str(_excel_safe(c)) for c in columns]} do not match the existing "
                            f"file's columns {[str(c) for c in header]}."
                        )
                    raw_by_safe: Dict[str, Any] = {}
                    for c in columns:
                        raw_by_safe.setdefault(str(_excel_safe(c)), c)
                    source_keys = [raw_by_safe[str(c)] for c in header]
                    row_cap = _excel_row_cap(len(header))
                    for record in row_iter:
                        if len(existing_values) + len(rows) >= row_cap:
                            raise ValueError(
                                f"Output: appending {len(rows):,} rows would grow {path} past "
                                f"the single-file excel export limit ({MAX_EXCEL_CELLS:,} cells "
                                f"at {len(header):,} columns) — write to a table instead."
                            )
                        existing_values.append(list(record))
            finally:
                workbook.close()

        combined = existing_values + [
            [_neutralize_formula(_excel_safe(row[k])) for k in source_keys] for row in rows
        ]
        new_df = pd.DataFrame(combined, columns=[_excel_safe(c) for c in header])

        def _write_excel(tmp_path: str) -> None:
            buffer = io.BytesIO()
            new_df.to_excel(buffer, index=False, engine="openpyxl")
            with open(tmp_path, "wb") as out:
                out.write(buffer.getvalue())

        _stage(_write_excel)
        return

    records = []
    if appending:
        with open(path, encoding="utf-8") as handle:
            existing = json.load(handle)
        records = existing if isinstance(existing, list) else [existing]
        if records:
            existing_columns = (
                list(records[0].keys()) if isinstance(records[0], dict) else []
            )
            if existing_columns and sorted(existing_columns) != sorted(columns):
                raise ValueError(
                    f"Output: cannot append to {path}: the data's columns "
                    f"{sorted(columns)} do not match the existing file's "
                    f"columns {sorted(existing_columns)}."
                )
    if len(records) + len(rows) > MAX_SINGLE_FILE_ROWS:
        raise ValueError(
            f"Output: appending {len(rows):,} rows would grow {path} past "
            f"the single-file export limit ({MAX_SINGLE_FILE_ROWS:,} rows) "
            f"— write to a table instead."
        )
    records.extend({c: row[c] for c in columns} for row in rows)

    def _write_json(tmp_path: str) -> None:
        with open(tmp_path, "w", encoding="utf-8") as handle:
            json.dump(records, handle, default=str, indent=2)

    _stage(_write_json)

def _run_file(config: Dict[str, Any], df, spark) -> None:
    file_type = config.get("file_type", "csv")
    if file_type not in SUPPORTED_FILE_TYPES:
        raise ValueError(
            f"Output: file_type '{file_type}' is not yet supported. Use csv, json, or excel."
        )
    path = _file_path(
        config.get("catalog", ""),
        config.get("schema", ""),
        config.get("volume", ""),
        _with_extension(config.get("file_name", ""), file_type),
    )
    append = config.get("write_mode", "overwrite") == "append"
    if file_type == "excel":
        row_cap = _excel_row_cap(len(df.columns))
        rows = df.limit(row_cap + 1).collect()
        if len(rows) > row_cap:
            raise ValueError(
                f"Output: result too large for single-file excel export "
                f"(over the {MAX_EXCEL_CELLS:,}-cell limit at {len(df.columns):,} "
                f"columns) — write to a table instead."
            )
    else:
        rows = df.limit(MAX_SINGLE_FILE_ROWS + 1).collect()
        if len(rows) > MAX_SINGLE_FILE_ROWS:
            raise ValueError(
                f"Output: result too large for single-file export "
                f"(over {MAX_SINGLE_FILE_ROWS:,} rows) — write to a table instead."
            )
    _write_single_file(rows, df.columns, file_type, path, append)

def _run_materialized_view(
    config: Dict[str, Any], source_view: str, spark
) -> None:
    catalog = config.get("catalog", "")
    schema = config.get("schema", "")
    view_name = config.get("view_name", "")
    if not view_name:
        raise ValueError("Output: 'view_name' is required for a materialized view")
    full_name = _qualified(catalog, schema, view_name)
    cluster_by = _cluster_by_clause(
        config.get("cluster_by_mode", "auto"), config.get("cluster_by_column", "") or ""
    )
    comment = _comment_clause(config.get("comment", "") or "")
    stmt = (
        f"CREATE OR REFRESH MATERIALIZED VIEW {full_name}{cluster_by}{comment} "
        f"AS SELECT * FROM {_quote(source_view)}"
    )
    spark.sql(stmt)

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs["data"]
    output_type = config.get("output_type", "table")
    catalog = config.get("catalog", "")
    schema = config.get("schema", "")
    table_name = config.get("table_name", "")
    write_mode = config.get("write_mode", "overwrite")
    merge_keys: List[str] = [k for k in (config.get("merge_keys") or []) if k]
    format_value = config.get("format", "default")
    table_properties = config.get("table_properties", "") or ""

    if output_type == "file":
        _run_file(config, df, spark)
        return {}

    source_view = f"_lb_output_v2_src_{uuid.uuid4().hex}"
    df.createOrReplaceTempView(source_view)

    if output_type == "materialized_view":
        try:
            _run_materialized_view(config, source_view, spark)
        finally:
            spark.catalog.dropTempView(source_view)
        return {}

    if not table_name:
        raise ValueError("Output: 'table_name' is required")

    full_name = _qualified(catalog, schema, table_name)

    if write_mode == "merge" and not merge_keys:
        raise ValueError("Output: 'merge_keys' is required when write_mode is merge")

    try:
        args = _resolve_args(table_properties)

        using = _using_clause(format_value)
        tblprops = _tbl_properties_clause(
            table_properties, format_value, spark=spark, full_name=full_name
        )

        if write_mode == "append":
            create_stmt = (
                f"CREATE TABLE IF NOT EXISTS {full_name}{using}{tblprops} "
                f"AS SELECT * FROM {_quote(source_view)} WHERE 1=0"
            )
            spark.sql(create_stmt, args=args) if args else spark.sql(create_stmt)
            insert_stmt = (
                f"INSERT INTO {full_name} BY NAME "
                f"SELECT * FROM {_quote(source_view)}"
            )
            spark.sql(insert_stmt, args=args) if args else spark.sql(insert_stmt)
        elif write_mode == "merge":
            create_stmt = (
                f"CREATE TABLE IF NOT EXISTS {full_name}{using}{tblprops} "
                f"AS SELECT * FROM {_quote(source_view)} WHERE 1=0"
            )
            spark.sql(create_stmt, args=args) if args else spark.sql(create_stmt)
            on_clause = " AND ".join(
                f"t.{_quote(k)} = s.{_quote(k)}" for k in merge_keys
            )
            merge_stmt = (
                f"MERGE INTO {full_name} t "
                f"USING {_quote(source_view)} s "
                f"ON {on_clause} "
                f"WHEN MATCHED THEN UPDATE SET * "
                f"WHEN NOT MATCHED THEN INSERT *"
            )
            spark.sql(merge_stmt, args=args) if args else spark.sql(merge_stmt)
        else:
            stmt = (
                f"CREATE OR REPLACE TABLE {full_name}{using}{tblprops} "
                f"AS SELECT * FROM {_quote(source_view)}"
            )
            spark.sql(stmt, args=args) if args else spark.sql(stmt)
    except Exception as exc:
        if table_properties.strip():
            raise ValueError(
                "Output: failed to write the table. Check the 'table_properties' "
                "and 'schema' fields for invalid SQL. Underlying error: "
                f"{exc}"
            ) from exc
        raise
    finally:
        spark.catalog.dropTempView(source_view)
    return {}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {
    "output_type": "table",
    "catalog": "renewal_risk_lakehouse",
    "schema": "renewal_risk_dev",
    "table_name": "ai_account_briefing_documents",
    "write_mode": "overwrite"
}
config["meta_state"] = {"is_preview": False, "is_focused_preview": False, "is_disabled": False}
inputs = {
    "data": ctx["prep_ai_account_briefing_documents_6e21c82b.prepared_data"]
}
if config["meta_state"]["is_disabled"]:
    def _lb_limit_zero(value):
        if hasattr(value, "limit"):
            return value.limit(0)
        if isinstance(value, list):
            return [_lb_limit_zero(item) for item in value]
        return value
    inputs = {k: _lb_limit_zero(v) for k, v in inputs.items()}
if not config["meta_state"]["is_disabled"]:
    out = run(config, inputs, spark)
else:
    out = {}